# DICE ITC Results Notebook

This is the single canonical notebook for the ITC DICE paper. It runs the released notebook-local pipeline end to end and produces the main results, the design-space analysis, the case studies, and the paper artifacts in one place.

The core detector remains a benign-trained behavioral digital twin. Any LLM material appears only at the end as a post-diagnostic extension, not as part of the detector itself.


## Scope of This Notebook

Use this notebook as the main paper notebook. It is organized as one reproducible flow, from dataset setup through DSE, case studies, and final paper exports.

- **DICE is a behavioral micro-twin.** The detector is a benign-trained digital twin that predicts nominal telemetry and scores residual divergence.
- **LLM material is downstream.** Any LLM content appears only as a post-diagnostic reporting layer and is not part of the detector.
- **The accelerator section is a projection.** The notebook reports a projected score-stage complexity estimate, not a synthesized hardware result.
- **The DSE section is analytical.** It combines released outputs, tuning sweeps, and notebook-local feature-budget retraining experiments.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import sys
import tempfile
import types

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import average_precision_score, roc_auc_score


## Notebook-Local Helper Functions

These helpers keep the later paper-facing cells readable.
They do not run experiments by themselves; they only support rendering, path resolution, and paper artifact organization.


In [ ]:
CFG_LABEL = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}


def _cfg_labels(values: pd.Series) -> list[str]:
    return [CFG_LABEL.get(str(v), str(v)) for v in values]


def render_tier_correlation_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    tier_case = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    tier_final = tier_case[tier_case["config"] == "tier0_tier1_tier2"].copy()
    tier_cols = ["tier0_share", "tier1_alt_share", "tier2_share"]

    tier_corr = tier_final[tier_cols].corr().round(4)
    tier_corr.to_csv(paper_full / "tier_share_correlation.csv")

    stressor_tier = tier_final.groupby("stressor", sort=False)[tier_cols].mean().reset_index()
    stressor_tier.to_csv(paper_full / "stressor_tier_share_summary.csv", index=False)

    tier_final["ternary_x"] = tier_final["tier1_alt_share"] + 0.5 * tier_final["tier2_share"]
    tier_final["ternary_y"] = (np.sqrt(3.0) / 2.0) * tier_final["tier2_share"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    im = axes[0].imshow(tier_corr.values, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_xticks(range(3), ["Tier-0", "Tier-1", "Tier-2"], rotation=30, ha="right")
    axes[0].set_yticks(range(3), ["Tier-0", "Tier-1", "Tier-2"])
    axes[0].set_title("Tier-share correlation")
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f"{tier_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(stressor_tier))
    axes[1].bar(x, stressor_tier["tier0_share"], label="Tier-0")
    axes[1].bar(x, stressor_tier["tier1_alt_share"], bottom=stressor_tier["tier0_share"], label="Tier-1")
    axes[1].bar(
        x,
        stressor_tier["tier2_share"],
        bottom=stressor_tier["tier0_share"] + stressor_tier["tier1_alt_share"],
        label="Tier-2",
    )
    axes[1].set_xticks(x, stressor_tier["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Mean tier evidence by stressor")
    axes[1].legend(loc="upper right")

    triangle = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [0.5, np.sqrt(3.0) / 2.0],
            [0.0, 0.0],
        ]
    )
    axes[2].plot(triangle[:, 0], triangle[:, 1], color="black")
    for stressor, d in tier_final.groupby("stressor", sort=False):
        axes[2].scatter(d["ternary_x"], d["ternary_y"], s=36, alpha=0.8, label=stressor)
    axes[2].text(-0.04, -0.03, "Tier-0")
    axes[2].text(1.01, -0.03, "Tier-1")
    axes[2].text(0.46, np.sqrt(3.0) / 2.0 + 0.03, "Tier-2")
    axes[2].set_title("Per-case tier composition")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
    axes[2].legend(loc="upper right", fontsize=7)

    fig.tight_layout()
    png = paper_fig / "fig_tier_correlation_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_corr, stressor_tier, png


def render_bootstrap_confidence(
    case_pred: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
    samples: int = 1000,
    seed: int = 0,
) -> tuple[pd.DataFrame, Path]:
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for cfg, d in case_pred.groupby("config", sort=False):
        stats: list[dict[str, float]] = []
        for _ in range(samples):
            sample = d.sample(n=len(d), replace=True, random_state=int(rng.integers(1 << 32)))
            if sample["label"].nunique() < 2:
                continue
            benign = sample[sample["label"] == 0]
            anomaly = sample[sample["label"] == 1]
            stats.append(
                {
                    "roc_auc_wc": roc_auc_score(sample["label"], sample["run_score_wc"]),
                    "pr_auc_wc": average_precision_score(sample["label"], sample["run_score_wc"]),
                    "benign_run_false_alarm_rate": benign["run_alert"].mean(),
                    "anomaly_run_detection_rate": anomaly["run_alert"].mean(),
                    "median_time_to_detect_s": anomaly.loc[
                        anomaly["run_alert"] == 1, "time_to_detect_s"
                    ].median(),
                }
            )

        boot = pd.DataFrame(stats)
        rows.append(
            {
                "config": cfg,
                "roc_auc_wc_lo": boot["roc_auc_wc"].quantile(0.025),
                "roc_auc_wc_hi": boot["roc_auc_wc"].quantile(0.975),
                "pr_auc_wc_lo": boot["pr_auc_wc"].quantile(0.025),
                "pr_auc_wc_hi": boot["pr_auc_wc"].quantile(0.975),
                "fpr_lo": boot["benign_run_false_alarm_rate"].quantile(0.025),
                "fpr_hi": boot["benign_run_false_alarm_rate"].quantile(0.975),
                "detect_lo": boot["anomaly_run_detection_rate"].quantile(0.025),
                "detect_hi": boot["anomaly_run_detection_rate"].quantile(0.975),
                "ttd_lo": boot["median_time_to_detect_s"].quantile(0.025),
                "ttd_hi": boot["median_time_to_detect_s"].quantile(0.975),
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(paper_full / "bootstrap_confidence_intervals.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    x = np.arange(len(out))

    pr_mid = (out["pr_auc_wc_lo"] + out["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - out["pr_auc_wc_lo"], out["pr_auc_wc_hi"] - pr_mid])
    axes[0].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4)
    axes[0].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[0].set_title("Bootstrap AUC-PR CI")

    fpr_mid = (out["fpr_lo"] + out["fpr_hi"]) / 2.0
    fpr_err = np.vstack([fpr_mid - out["fpr_lo"], out["fpr_hi"] - fpr_mid])
    axes[1].errorbar(x, fpr_mid, yerr=fpr_err, fmt="o", capsize=4)
    axes[1].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[1].set_title("Bootstrap benign-FPR CI")

    ttd_mid = (out["ttd_lo"] + out["ttd_hi"]) / 2.0
    ttd_err = np.vstack([ttd_mid - out["ttd_lo"], out["ttd_hi"] - ttd_mid])
    axes[2].errorbar(x, ttd_mid, yerr=ttd_err, fmt="o", capsize=4)
    axes[2].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[2].set_title("Bootstrap time-to-detect CI")

    fig.tight_layout()
    png = paper_fig / "fig_bootstrap_confidence_intervals.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out, png


def export_llm_case_cards(
    out_full: Path,
    appendix_full: Path,
) -> pd.DataFrame:
    case_diag = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    cards = case_diag[case_diag["config"] == "tier0_tier1_tier2"].copy()
    keep_cols = [
        "case_id",
        "workload",
        "stressor",
        "dominant_tier",
        "dominant_mechanism",
        "tier0_share",
        "tier1_alt_share",
        "tier2_share",
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
        "top_feature_1",
        "top_feature_score_1",
        "top_feature_2",
        "top_feature_score_2",
        "top_feature_3",
        "top_feature_score_3",
        "top_feature_4",
        "top_feature_score_4",
        "top_feature_5",
        "top_feature_score_5",
        "top_mechanism_1",
        "top_mechanism_score_1",
        "top_mechanism_2",
        "top_mechanism_score_2",
        "top_mechanism_3",
        "top_mechanism_score_3",
    ]
    cards = cards[[c for c in keep_cols if c in cards.columns]].copy()

    def _case_card_json(row: pd.Series) -> str:
        payload = {
            "case_id": row.get("case_id"),
            "workload": row.get("workload"),
            "stressor": row.get("stressor"),
            "dominant_tier": row.get("dominant_tier"),
            "dominant_mechanism": row.get("dominant_mechanism"),
            "tier_share": {
                "tier0": row.get("tier0_share"),
                "tier1": row.get("tier1_alt_share"),
                "tier2": row.get("tier2_share"),
            },
            "mechanism_share": {
                "compute": row.get("compute_share"),
                "memory_io": row.get("memory_io_share"),
                "thermal_power": row.get("thermal_power_share"),
                "scheduler_runtime": row.get("scheduler_runtime_share"),
                "platform_pressure": row.get("platform_pressure_share"),
            },
            "top_features": [
                {"name": row.get(f"top_feature_{i}"), "score": row.get(f"top_feature_score_{i}")}
                for i in range(1, 6)
                if pd.notna(row.get(f"top_feature_{i}"))
            ],
            "top_mechanisms": [
                {"name": row.get(f"top_mechanism_{i}"), "score": row.get(f"top_mechanism_score_{i}")}
                for i in range(1, 4)
                if pd.notna(row.get(f"top_mechanism_{i}"))
            ],
        }
        return json.dumps(payload, sort_keys=True)

    cards["diagnostic_case_card_json"] = cards.apply(_case_card_json, axis=1)
    cards["reviewer_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "You are preparing a grounded DICE diagnostic note for a silicon-reliability reviewer. "
            "Use only the supplied case card. Do not invent missing evidence. "
            "Explain the dominant tier, dominant mechanism, and the top residual cues in plain English.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["triage_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Write a compact triage report with five fields: "
            "severity, likely subsystem, evidence summary, two follow-up measurements, and confidence. "
            "If the evidence is weak, say so directly.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["followup_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Recommend up to three next diagnostic steps. "
            "Each step must cite the specific feature or mechanism that motivated it.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards.to_csv(appendix_full / "llm_case_cards.csv", index=False)
    return cards


def export_llm_diagnostic_model_catalog(appendix_full: Path) -> pd.DataFrame:
    models = pd.DataFrame(
        [
            {
                "model_id": "Qwen/Qwen2.5-7B-Instruct",
                "deployment_role": "primary DICE baseline",
                "priority_rank": 1,
                "params_billions": 7.61,
                "context_tokens": 131072,
                "strengths": "Strong instruction following, structured output behavior, and long-context support.",
                "best_for_dice": "Primary grounded reviewer summaries and structured incident reports from exported case cards.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct",
            },
            {
                "model_id": "microsoft/Phi-4-mini-instruct",
                "deployment_role": "lightweight comparison",
                "priority_rank": 2,
                "params_billions": 3.8,
                "context_tokens": 128000,
                "strengths": "Small footprint, strong reasoning density, and good fit for constrained local diagnostics.",
                "best_for_dice": "Fast first-pass case summaries and follow-up recommendations on a laptop or edge workstation.",
                "source_url": "https://huggingface.co/microsoft/Phi-4-mini-instruct",
            },
            {
                "model_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
                "deployment_role": "ecosystem baseline",
                "priority_rank": 3,
                "params_billions": 8.0,
                "context_tokens": 128000,
                "strengths": "Broad tooling support, stable chat behavior, and strong general-purpose instruction tuning.",
                "best_for_dice": "Fallback baseline when the deployment stack already supports Llama-family models.",
                "source_url": "https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct",
            },
            {
                "model_id": "Qwen/Qwen2.5-14B-Instruct",
                "deployment_role": "stronger offline review",
                "priority_rank": 4,
                "params_billions": 14.7,
                "context_tokens": 131072,
                "strengths": "Higher-capacity structured reasoning while remaining practical for offline workstation use.",
                "best_for_dice": "Second-pass failure analysis and richer postmortem summaries after the detector has already raised a case.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-14B-Instruct",
            },
        ]
    ).sort_values('priority_rank').reset_index(drop=True)
    models.to_csv(appendix_full / "llm_diagnostic_model_catalog.csv", index=False)
    return models


def export_llm_diagnostic_prompt_bundle(
    cards: pd.DataFrame,
    models: pd.DataFrame,
    appendix_full: Path,
) -> pd.DataFrame:
    system_prompt = (
        "You are a DICE diagnostic copilot. You may use only the structured DICE evidence supplied to you. "
        "Do not claim access to raw telemetry, hidden logs, or external knowledge about the run. "
        "If the evidence is incomplete, say that the conclusion is tentative."
    )
    rows = []
    for _, model in models.iterrows():
        for _, card in cards.iterrows():
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "triage",
                    "system_prompt": system_prompt,
                    "user_prompt": card["triage_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "reviewer_summary",
                    "system_prompt": system_prompt,
                    "user_prompt": card["reviewer_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "followup",
                    "system_prompt": system_prompt,
                    "user_prompt": card["followup_prompt"],
                }
            )

    bundle = pd.DataFrame(rows)
    bundle.to_csv(appendix_full / "llm_diagnostic_prompt_bundle.csv", index=False)
    with (appendix_full / "llm_diagnostic_prompt_bundle.jsonl").open("w") as f:
        for row in bundle.to_dict(orient="records"):
            f.write(json.dumps(row) + "\n")
    return bundle



TIER_ALIAS_MAP = {
    "tier0": ["tier0", "tier-0", "tier 0", "tier-0 evidence", "tier 0 evidence"],
    "tier1_alt": ["tier1", "tier-1", "tier 1", "tier1_alt", "tier-1 evidence", "tier 1 evidence"],
    "tier2": ["tier2", "tier-2", "tier 2", "tier-2 evidence", "tier 2 evidence"],
}

MECHANISM_ALIAS_MAP = {
    "compute": ["compute"],
    "memory_io": ["memory_io", "memory/io", "memory io"],
    "thermal_power": ["thermal_power", "thermal/power", "thermal power"],
    "scheduler_runtime": ["scheduler_runtime", "scheduler runtime"],
    "platform_pressure": ["platform_pressure", "platform pressure"],
}


def _slugify_model_id(model_id: str) -> str:
    return model_id.replace('/', '__').replace('-', '_').replace('.', '_')


def _normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace('_', ' ')
    text = text.replace(':', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def _feature_aliases(name: str | float | None) -> set[str]:
    if pd.isna(name):
        return set()
    name = str(name)
    aliases = {name.lower(), _normalize_text(name)}
    if ':' in name:
        tail = name.split(':', 1)[1]
        aliases.add(tail.lower())
        aliases.add(_normalize_text(tail))
    return {a for a in aliases if a}


def _contains_any(text: str, aliases: set[str] | list[str]) -> bool:
    norm = _normalize_text(text)
    return any(alias and _normalize_text(alias) in norm for alias in aliases)


def run_transformers_llm_batch(
    prompt_bundle: pd.DataFrame,
    appendix_full: Path,
    model_id: str,
    prompt_type: str = "triage",
    max_cases: int = 8,
    max_new_tokens: int = 320,
    temperature: float = 0.0,
) -> pd.DataFrame:
    if importlib.util.find_spec("transformers") is None or importlib.util.find_spec("torch") is None:
        raise RuntimeError("transformers and torch must be installed to run local model inference.")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    subset = prompt_bundle[
        (prompt_bundle["model_id"] == model_id) & (prompt_bundle["prompt_type"] == prompt_type)
    ].copy().head(max_cases)
    if subset.empty:
        raise ValueError(f"No prompts found for model_id={model_id!r} and prompt_type={prompt_type!r}.")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )

    rows = []
    for row in subset.itertuples(index=False):
        messages = [
            {"role": "system", "content": row.system_prompt},
            {"role": "user", "content": row.user_prompt},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            text = row.system_prompt + "\n\n" + row.user_prompt

        model_inputs = tokenizer([text], return_tensors="pt")
        model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=max(temperature, 1e-5),
        )
        new_ids = generated_ids[:, model_inputs["input_ids"].shape[1]:]
        response_text = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
        rows.append(
            {
                "model_id": model_id,
                "case_id": row.case_id,
                "prompt_type": prompt_type,
                "response_text": response_text,
            }
        )

    outputs = pd.DataFrame(rows)
    slug = _slugify_model_id(model_id)
    outputs.to_csv(appendix_full / f"llm_outputs_{slug}_{prompt_type}.csv", index=False)
    with (appendix_full / f"llm_outputs_{slug}_{prompt_type}.jsonl").open("w") as f:
        for rec in outputs.to_dict(orient="records"):
            f.write(json.dumps(rec) + "\n")
    return outputs


def evaluate_llm_grounding_outputs(
    llm_outputs: pd.DataFrame,
    llm_cards: pd.DataFrame,
    appendix_full: Path,
    stem: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    detail = llm_outputs.merge(llm_cards, on="case_id", how="left", suffixes=("", "_card")).copy()

    supported_tier_threshold = 0.05
    supported_mech_threshold = 0.10
    global_mechanisms = list(MECHANISM_ALIAS_MAP.keys())
    global_tiers = list(TIER_ALIAS_MAP.keys())

    rows = []
    for row in detail.itertuples(index=False):
        text = str(row.response_text)
        dominant_tier = getattr(row, "dominant_tier")
        dominant_mechanism = getattr(row, "dominant_mechanism")

        tier_aliases = set(TIER_ALIAS_MAP.get(str(dominant_tier), []))
        dominant_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(dominant_mechanism), [str(dominant_mechanism)]))
        top_feature_aliases = _feature_aliases(getattr(row, "top_feature_1", None))
        top_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(getattr(row, "top_mechanism_1", '')), [str(getattr(row, "top_mechanism_1", ''))]))

        mentions_dominant_tier = _contains_any(text, tier_aliases)
        mentions_dominant_mechanism = _contains_any(text, dominant_mech_aliases)
        mentions_top_feature_1 = _contains_any(text, top_feature_aliases)
        mentions_top_mechanism_1 = _contains_any(text, top_mech_aliases)

        supported_tiers = {
            "tier0": getattr(row, "tier0_share", 0.0),
            "tier1_alt": getattr(row, "tier1_alt_share", 0.0),
            "tier2": getattr(row, "tier2_share", 0.0),
        }
        unsupported_tier_mentions = any(
            _contains_any(text, TIER_ALIAS_MAP[t]) and supported_tiers.get(t, 0.0) < supported_tier_threshold
            for t in global_tiers
        )

        supported_mechs = {
            "compute": getattr(row, "compute_share", 0.0),
            "memory_io": getattr(row, "memory_io_share", 0.0),
            "thermal_power": getattr(row, "thermal_power_share", 0.0),
            "scheduler_runtime": getattr(row, "scheduler_runtime_share", 0.0),
            "platform_pressure": getattr(row, "platform_pressure_share", 0.0),
        }
        unsupported_mechanism_mentions = any(
            _contains_any(text, MECHANISM_ALIAS_MAP[m]) and supported_mechs.get(m, 0.0) < supported_mech_threshold
            for m in global_mechanisms
        )

        cue_coverage = np.mean([
            float(mentions_dominant_tier),
            float(mentions_dominant_mechanism),
            float(mentions_top_feature_1),
            float(mentions_top_mechanism_1),
        ])

        rows.append(
            {
                "model_id": getattr(row, "model_id"),
                "case_id": getattr(row, "case_id"),
                "prompt_type": getattr(row, "prompt_type"),
                "mentions_dominant_tier": mentions_dominant_tier,
                "mentions_dominant_mechanism": mentions_dominant_mechanism,
                "mentions_top_feature_1": mentions_top_feature_1,
                "mentions_top_mechanism_1": mentions_top_mechanism_1,
                "grounded_core": bool(mentions_dominant_tier and mentions_dominant_mechanism),
                "cue_coverage": cue_coverage,
                "unsupported_tier_mentions": unsupported_tier_mentions,
                "unsupported_mechanism_mentions": unsupported_mechanism_mentions,
                "hallucination_flag": bool(unsupported_tier_mentions or unsupported_mechanism_mentions),
            }
        )

    scored = pd.DataFrame(rows)
    summary = (
        scored.groupby(["model_id", "prompt_type"], sort=False)
        .agg(
            n_outputs=("case_id", "count"),
            grounded_core_rate=("grounded_core", "mean"),
            dominant_tier_rate=("mentions_dominant_tier", "mean"),
            dominant_mechanism_rate=("mentions_dominant_mechanism", "mean"),
            top_feature_1_rate=("mentions_top_feature_1", "mean"),
            top_mechanism_1_rate=("mentions_top_mechanism_1", "mean"),
            mean_cue_coverage=("cue_coverage", "mean"),
            hallucination_rate=("hallucination_flag", "mean"),
        )
        .reset_index()
    )

    summary.to_csv(appendix_full / f"llm_grounding_summary_{stem}.csv", index=False)
    scored.to_csv(appendix_full / f"llm_grounding_detail_{stem}.csv", index=False)
    return summary, scored


def render_paper_performance_stack(
    case_pred: pd.DataFrame,
    overall_full: pd.DataFrame,
    sequential: pd.DataFrame,
    reliability: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    summary = (
        overall_full[
            [
                "config",
                "roc_auc",
                "pr_auc",
                "roc_auc_wc",
                "pr_auc_wc",
                "median_nominal_score_wc",
                "median_anomaly_score_wc",
            ]
        ]
        .merge(
            sequential[
                [
                    "config",
                    "anomaly_detect_rate",
                    "median_time_to_detect_s",
                ]
            ],
            on="config",
        )
        .merge(
            reliability[
                [
                    "config",
                    "target_alpha",
                    "benign_block_false_alarm_rate",
                ]
            ],
            on="config",
        )
    )
    summary["config_label"] = _cfg_labels(summary["config"])
    summary["reliability_margin"] = summary["target_alpha"] - summary["benign_block_false_alarm_rate"]
    summary.to_csv(paper_full / "paper_performance_stack_summary.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    final = case_pred[case_pred["config"] == "tier0_tier1_tier2"].copy()
    benign = final[final["label"] == 0]["run_score_wc"].to_numpy(dtype=float)
    anomaly = final[final["label"] == 1]["run_score_wc"].to_numpy(dtype=float)
    bp = axes[0, 0].boxplot([benign, anomaly], labels=["Benign", "Anomaly"], patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#2563eb", "#dc2626"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[0, 0].set_title("A. Final-head score separation")
    axes[0, 0].set_ylabel("Workload-conditioned run score")

    base_color = "#94a3b8"
    wc_color = "#0f766e"
    for _, row in summary.iterrows():
        axes[0, 1].scatter(row["roc_auc"], row["pr_auc"], color=base_color, s=70)
        axes[0, 1].scatter(row["roc_auc_wc"], row["pr_auc_wc"], color=wc_color, s=90)
        axes[0, 1].annotate(
            row["config_label"],
            (row["roc_auc_wc"], row["pr_auc_wc"]),
            textcoords="offset points",
            xytext=(6, 6),
        )
        axes[0, 1].plot([row["roc_auc"], row["roc_auc_wc"]], [row["pr_auc"], row["pr_auc_wc"]], color="#475569")
    axes[0, 1].set_xlabel("Run-level ROC-AUC")
    axes[0, 1].set_ylabel("Run-level Average Precision")
    axes[0, 1].set_title("B. Digital-twin score refinement")

    x = np.arange(len(summary))
    axes[1, 0].bar(x, summary["benign_block_false_alarm_rate"], color="#f59e0b")
    axes[1, 0].axhline(float(summary["target_alpha"].iloc[0]), color="black", linestyle="--", linewidth=1.2)
    axes[1, 0].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 0].set_ylabel("Empirical benign block FAR")
    axes[1, 0].set_title("C. Calibrated reliability")

    bars = axes[1, 1].bar(x, summary["anomaly_detect_rate"], color="#16a34a", label="Detection rate")
    ax2 = axes[1, 1].twinx()
    ax2.plot(x, summary["median_time_to_detect_s"], color="#1d4ed8", marker="o", linewidth=2, label="Median TTD")
    axes[1, 1].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 1].set_ylabel("Run-level detection rate")
    ax2.set_ylabel("Median time-to-detect (s)")
    axes[1, 1].set_title("D. Operational decision performance")
    axes[1, 1].legend([bars], ["Detection rate"], loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_performance_stack.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return summary, png


def render_explainability_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Path]:
    tier_contrib = pd.read_csv(out_full / "stressor_tier_contributions.csv")
    mechanism = pd.read_csv(out_full / "mechanism_group_summary.csv")
    cm = pd.read_csv(out_full / "stressor_confusion_matrix.csv", index_col=0)

    tier_contrib.to_csv(paper_full / "paper_tier_contribution_summary.csv", index=False)
    mechanism.to_csv(paper_full / "paper_mechanism_summary.csv", index=False)
    cm.to_csv(paper_full / "paper_stressor_confusion_matrix.csv")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

    x = np.arange(len(tier_contrib))
    axes[0].bar(x, tier_contrib["tier0_share"], label="Tier-0", color="#4E79A7")
    axes[0].bar(x, tier_contrib["tier1_alt_share"], bottom=tier_contrib["tier0_share"], label="Tier-1", color="#59A14F")
    axes[0].bar(
        x,
        tier_contrib["tier2_share"],
        bottom=tier_contrib["tier0_share"] + tier_contrib["tier1_alt_share"],
        label="Tier-2",
        color="#F28E2B",
    )
    axes[0].set_xticks(x, tier_contrib["stressor"], rotation=30, ha="right")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("A. Tier contribution by stressor")
    axes[0].legend(loc="upper right")
    axes[0].grid(axis="y", alpha=0.20)

    mech_cols = [
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
    ]
    mech_labels = {
        "compute_share": "Compute",
        "memory_io_share": "Memory/IO",
        "thermal_power_share": "Thermal/Power",
        "scheduler_runtime_share": "Scheduler/Runtime",
        "platform_pressure_share": "Platform Pressure",
    }
    plot_df = mechanism[["stressor", *mech_cols]].copy().rename(columns=mech_labels)
    long_df = plot_df.melt(id_vars="stressor", var_name="mechanism", value_name="share")

    x_order = [mech_labels[c] for c in mech_cols]
    y_order = list(plot_df["stressor"])
    x_map = {name: idx for idx, name in enumerate(x_order)}
    y_map = {name: idx for idx, name in enumerate(y_order)}

    sc = axes[1].scatter(
        long_df["mechanism"].map(x_map),
        long_df["stressor"].map(y_map),
        s=1250 * long_df["share"] + 40,
        c=long_df["share"],
        cmap="YlGnBu",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in long_df.iterrows():
        axes[1].text(
            x_map[row["mechanism"]],
            y_map[row["stressor"]],
            f'{row["share"]:.2f}',
            ha="center",
            va="center",
            fontsize=9,
        )
    axes[1].set_xticks(range(len(x_order)), x_order, rotation=20, ha="right")
    axes[1].set_yticks(range(len(y_order)), y_order)
    axes[1].set_title("B. Mechanism fingerprint by stressor")
    axes[1].grid(alpha=0.15)
    fig.colorbar(sc, ax=axes[1], fraction=0.046, pad=0.04, label="Mean mechanism share")

    im = axes[2].imshow(cm.values, cmap="Blues")
    axes[2].set_xticks(range(len(cm.columns)), list(cm.columns), rotation=30, ha="right")
    axes[2].set_yticks(range(len(cm.index)), list(cm.index))
    axes[2].set_title("C. Stressor diagnosis confusion")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[2].text(j, i, str(int(cm.iloc[i, j])), ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.tight_layout()
    png = paper_fig / "fig_paper_explainability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_contrib, mechanism, cm, png
def render_portability_dashboard(
    frontier: pd.DataFrame,
    holdout: pd.DataFrame,
    bootstrap_ci: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    view = frontier.copy()
    view["config_label"] = _cfg_labels(view["config"])
    holdout_view = holdout.copy()
    holdout_view["config_label"] = _cfg_labels(holdout_view["config"])
    boot_view = bootstrap_ci.copy()
    boot_view["config_label"] = _cfg_labels(boot_view["config"])

    portability_summary = view[
        [
            "config",
            "config_label",
            "n_features",
            "portable_pr_auc",
            "holdout_worst_pr_auc",
            "reliability_margin",
            "joint_detection_diagnosis",
        ]
    ].copy()
    portability_summary.to_csv(paper_full / "paper_portability_summary.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    scatter = axes[0].scatter(
        view["n_features"],
        view["portable_pr_auc"],
        s=view["joint_detection_diagnosis"].fillna(0.0) * 1800 + 140,
        c=view["reliability_margin"],
        cmap="viridis",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in view.iterrows():
        axes[0].annotate(row["config_label"], (row["n_features"], row["portable_pr_auc"]), textcoords="offset points", xytext=(6, 6))
    axes[0].set_xlabel("Median active features")
    axes[0].set_ylabel("Portable AUC-PR")
    axes[0].set_title("A. Observability-portability frontier")
    fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(holdout_view))
    width = 0.35
    axes[1].bar(x - width / 2.0, holdout_view["mean_pr_auc"], width=width, label="Mean holdout PR")
    axes[1].bar(x + width / 2.0, holdout_view["worst_pr_auc"], width=width, label="Worst holdout PR")
    axes[1].set_xticks(x, holdout_view["config_label"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_title("B. Holdout portability")
    axes[1].legend(loc="upper right")

    x = np.arange(len(boot_view))
    pr_mid = (boot_view["pr_auc_wc_lo"] + boot_view["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - boot_view["pr_auc_wc_lo"], boot_view["pr_auc_wc_hi"] - pr_mid])
    axes[2].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4, color="#1d4ed8", label="AP CI")
    det_mid = (boot_view["detect_lo"] + boot_view["detect_hi"]) / 2.0
    det_err = np.vstack([det_mid - boot_view["detect_lo"], boot_view["detect_hi"] - det_mid])
    axes[2].errorbar(x, det_mid, yerr=det_err, fmt="o", capsize=4, color="#16a34a", label="Detect-rate CI")
    axes[2].set_xticks(x, boot_view["config_label"], rotation=30, ha="right")
    axes[2].set_ylim(0.0, 1.05)
    axes[2].set_title("C. Bootstrap uncertainty")
    axes[2].legend(loc="lower right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_portability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return portability_summary, png


In [ ]:
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## Execution Guard

The notebook should be launched from the intended DICE clone, not from a stale copy in `Trash` or another transient folder.

The next cell validates the repository location, confirms the released dataset is available, and prepares the main paper and appendix output folders.


In [ ]:
if '.Trash' in str(REPO_ROOT):
    raise RuntimeError(
        'This notebook was launched from a Trash clone. Reopen it from your intended DICE repository checkout.'
    )
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'

PAPER_FULL = OUT_PAPER / 'full'
PAPER_FIG = PAPER_FULL / 'figures'
APPENDIX_FULL = OUT_APPENDIX / 'full'
NOTEBOOK_RUNTIME = OUT_PAPER / 'runtime_summary.json'

for path in [OUT_PAPER, OUT_APPENDIX, PAPER_FULL, PAPER_FIG, APPENDIX_FULL]:
    path.mkdir(parents=True, exist_ok=True)

print('Validated repository root :', REPO_ROOT)
print('Validated dataset root    :', DATASET_ROOT)
print('Main paper output folder  :', PAPER_FULL)
print('Appendix output folder    :', APPENDIX_FULL)


## Embedded DICE Engine

This cell embeds the released DICE analysis and training/evaluation pipeline directly inside the notebook.
No external Python runner is required.

Important:
- this is where the notebook defines the notebook-local DICE engine;
- the actual experiment begins in the next section, **Run End-to-End**;
- the notebook also includes a small patch cell right after this one so the embedded engine exports block traces for the virtual-system overlay.


In [ ]:
import hashlib
import importlib.metadata
import platform
import types
from datetime import datetime, timezone
from time import perf_counter

NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE = '#!/usr/bin/env python3\n"""\nGenerate paper-ready Results/Analysis artifacts from DICE tiered dataset.\n\nOutputs:\n- CSV tables (overall metrics, stressor metrics, workload summaries, feature inventory)\n- LaTeX tables ready for Overleaf\n- PNG figures (AF-index trajectories, separability heatmaps, score distributions)\n- Markdown summary with key values to paste into paper draft\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Tuple\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nIGNORE_COLS = {\n    "idx",\n    "ts_unix_s",\n    "t_rel_s",\n    "timestamp",\n    "time",\n    "ts",\n}\n\nTIER_PRETTY = {\n    "tier0": "Tier-0",\n    "tier1_alt": "Tier-1",\n    "tier2": "Tier-2",\n}\n\nCOLOR_BY_STRESSOR = {\n    "NOMINAL": "#000000",\n    "ATOMIC": "#e57373",\n    "BRANCH": "#66bb6a",\n    "CACHE": "#f6a04d",\n    "MEMBW": "#b39ddb",\n    "TLB": "#bcaaa4",\n}\n\n\n@dataclass(frozen=True)\nclass TierData:\n    tier: str\n    features: List[str]\n    run_df: pd.DataFrame\n    timeseries: Dict[str, Dict[str, np.ndarray]]\n    case_quality: pd.DataFrame\n    union_features: List[str]\n\n\ndef case_id(workload: str, stressor: str) -> str:\n    return f"{workload}__{stressor}"\n\n\ndef robust_scale(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, y_score))\n\n\ndef safe_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    y_true = np.asarray(y_true, dtype=int)\n    y_score = np.asarray(y_score, dtype=float)\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, y_score))\n\n\ndef downsample_to_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    out = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return out.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef read_case_csv(root: Path, tier: str, workload: str, stressor: str) -> pd.DataFrame:\n    p = root / tier / case_id(workload, stressor) / TIER_FILE[tier]\n    if not p.exists():\n        raise FileNotFoundError(f"Missing case file: {p}")\n    df = pd.read_csv(p)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_feature_columns(df: pd.DataFrame) -> List[str]:\n    cols = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            cols.append(c)\n    return cols\n\n\ndef discover_features(root: Path, tier: str, source_hz: int = 5) -> Tuple[List[str], List[str], pd.DataFrame]:\n    common = None\n    union = set()\n    rows = []\n    for w in WORKLOADS:\n        for s in STRESSORS:\n            p = root / tier / case_id(w, s) / TIER_FILE[tier]\n            df = pd.read_csv(p)\n            cols = set(numeric_feature_columns(df))\n            union |= cols\n            common = cols if common is None else (common & cols)\n            rows.append(\n                {\n                    "tier": tier,\n                    "case_id": case_id(w, s),\n                    "workload": w,\n                    "stressor": s,\n                    "rows_5hz": int(len(df)),\n                    "cols_total": int(df.shape[1]),\n                    "numeric_cols": int(len(cols)),\n                    "nan_fraction": float(df.isna().mean().mean()),\n                    "file_bytes": int(p.stat().st_size),\n                }\n            )\n    common_list = sorted(common) if common else []\n    union_list = sorted(union)\n\n    # Drop globally near-constant channels from common list.\n    keep = []\n    for f in common_list:\n        vals = []\n        for w in WORKLOADS:\n            for s in STRESSORS:\n                d = downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz)\n                vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep, union_list, pd.DataFrame(rows)\n\n\ndef build_tier_data(root: Path, tier: str, source_hz: int = 5) -> TierData:\n    features, union_features, quality = discover_features(root, tier, source_hz=source_hz)\n    run_rows = []\n    timeseries = {w: {} for w in WORKLOADS}\n\n    for w in WORKLOADS:\n        ds = {s: downsample_to_1hz(read_case_csv(root, tier, w, s), source_hz=source_hz) for s in STRESSORS}\n        n = min(len(v) for v in ds.values())\n        arr = {\n            s: ds[s].iloc[:n][features].to_numpy(dtype=float, copy=True)\n            for s in STRESSORS\n        }\n        baseline = arr["NOMINAL"]\n        med = np.nanmedian(baseline, axis=0)\n        scale = np.array([robust_scale(baseline[:, j]) for j in range(baseline.shape[1])], dtype=float)\n        scale[scale <= 1e-12] = 1.0\n\n        for s in STRESSORS:\n            z = np.abs((arr[s] - med) / (scale + 1e-12))\n            score_ts = np.nanmean(z, axis=1)\n            timeseries[w][s] = score_ts\n            run_rows.append(\n                {\n                    "tier": tier,\n                    "workload": w,\n                    "stressor": s,\n                    "label": 0 if s == "NOMINAL" else 1,\n                    "run_score_median": float(np.nanmedian(score_ts)),\n                    "run_score_mean": float(np.nanmean(score_ts)),\n                    "run_score_p95": float(np.nanpercentile(score_ts, 95)),\n                    "samples_1hz": int(len(score_ts)),\n                }\n            )\n\n    return TierData(\n        tier=tier,\n        features=features,\n        run_df=pd.DataFrame(run_rows),\n        timeseries=timeseries,\n        case_quality=quality,\n        union_features=union_features,\n    )\n\n\ndef make_overall_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        y = df["label"].to_numpy(dtype=int)\n        s = df["run_score_median"].to_numpy(dtype=float)\n\n        nom = df[df["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = df[df["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        tau95 = float(np.quantile(nom, 0.95))\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": int(len(td.features)),\n                "n_features_union": int(len(td.union_features)),\n                "n_cases": int(len(df)),\n                "roc_auc": safe_auc(y, s),\n                "pr_auc": safe_ap(y, s),\n                "median_nominal": float(np.median(nom)),\n                "median_anomaly": float(np.median(anm)),\n                "anom_nom_ratio": float(np.median(anm) / (np.median(nom) + 1e-12)),\n                "threshold_q95_nominal": tau95,\n                "fpr_at_q95": float(np.mean(nom > tau95)),\n                "tpr_at_q95": float(np.mean(anm > tau95)),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef make_stressor_metrics(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        neg = df[df["stressor"] == "NOMINAL"][["workload", "run_score_median"]].set_index("workload")\n        for a in ANOMALIES:\n            pos = df[df["stressor"] == a][["workload", "run_score_median"]].set_index("workload")\n            merged = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n            y_true = np.array([0] * len(merged) + [1] * len(merged), dtype=int)\n            y_score = np.concatenate(\n                [\n                    merged["run_score_median_neg"].to_numpy(dtype=float),\n                    merged["run_score_median_pos"].to_numpy(dtype=float),\n                ]\n            )\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "stressor": a,\n                    "n_pos": int(len(merged)),\n                    "n_neg": int(len(merged)),\n                    "roc_auc": safe_auc(y_true, y_score),\n                    "pr_auc": safe_ap(y_true, y_score),\n                    "median_neg": float(np.median(merged["run_score_median_neg"])),\n                    "median_pos": float(np.median(merged["run_score_median_pos"])),\n                    "pos_neg_ratio": float(\n                        np.median(merged["run_score_median_pos"])\n                        / (np.median(merged["run_score_median_neg"]) + 1e-12)\n                    ),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef make_workload_summary(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        df = td.run_df.copy()\n        for w in WORKLOADS:\n            d = df[df["workload"] == w]\n            nom = d[d["stressor"] == "NOMINAL"]["run_score_median"].iloc[0]\n            anm = d[d["stressor"] != "NOMINAL"]["run_score_median"].to_numpy(dtype=float)\n            rows.append(\n                {\n                    "tier": td.tier,\n                    "tier_name": TIER_PRETTY[td.tier],\n                    "workload": w,\n                    "nominal_score": float(nom),\n                    "anomaly_median_score": float(np.median(anm)),\n                    "anomaly_nominal_ratio": float(np.median(anm) / (float(nom) + 1e-12)),\n                    "anomaly_p95_score": float(np.percentile(anm, 95)),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef table_to_latex(df: pd.DataFrame, caption: str, label: str) -> str:\n    rendered = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{rendered}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef save_metric_tables(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    workload: pd.DataFrame,\n    features: pd.DataFrame,\n    quality: pd.DataFrame,\n    runs: pd.DataFrame,\n) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    overall_out = overall.sort_values("tier")\n    stressor_out = stressor.sort_values(["tier", "stressor"])\n    workload_out = workload.sort_values(["tier", "workload"])\n    features_out = features.sort_values("tier")\n    quality_out = quality.sort_values(["tier", "case_id"])\n    runs_out = runs.sort_values(["tier", "workload", "stressor"])\n\n    overall_out.to_csv(out_dir / "table_overall_metrics.csv", index=False)\n    stressor_out.to_csv(out_dir / "table_stressor_metrics.csv", index=False)\n    workload_out.to_csv(out_dir / "table_workload_summary.csv", index=False)\n    features_out.to_csv(out_dir / "table_feature_inventory.csv", index=False)\n    quality_out.to_csv(out_dir / "table_case_quality.csv", index=False)\n    runs_out.to_csv(out_dir / "table_run_scores.csv", index=False)\n\n    overall_tex = overall_out[\n        [\n            "tier_name",\n            "n_features_common",\n            "roc_auc",\n            "pr_auc",\n            "median_nominal",\n            "median_anomaly",\n            "anom_nom_ratio",\n        ]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "n_features_common": "Common Features",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "median_nominal": "Median(Nominal)",\n            "median_anomaly": "Median(Anomaly)",\n            "anom_nom_ratio": "Anomaly/Nominal",\n        }\n    )\n\n    stressor_tex = stressor_out[\n        ["tier_name", "stressor", "roc_auc", "pr_auc", "pos_neg_ratio"]\n    ].rename(\n        columns={\n            "tier_name": "Tier",\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC",\n            "pr_auc": "AUC-PR",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n        }\n    )\n\n    (out_dir / "table_overall_metrics.tex").write_text(\n        table_to_latex(\n            overall_tex,\n            "Run-level anomaly separability by telemetry tier (AF-index score).",\n            "tab:dice_overall_metrics",\n        )\n    )\n    (out_dir / "table_stressor_metrics.tex").write_text(\n        table_to_latex(\n            stressor_tex,\n            "Per-stressor separability by tier (four workloads pooled per stressor).",\n            "tab:dice_stressor_metrics",\n        )\n    )\n\n\ndef plot_af_timeseries(out_dir: Path, tier_data: Iterable[TierData]) -> List[str]:\n    out_paths = []\n    for td in tier_data:\n        fig, axes = plt.subplots(len(WORKLOADS), 1, figsize=(16, 13), sharex=True)\n        if len(WORKLOADS) == 1:\n            axes = [axes]\n\n        for i, w in enumerate(WORKLOADS):\n            ax = axes[i]\n            nom = td.timeseries[w]["NOMINAL"]\n            x = np.arange(len(nom), dtype=float) / 60.0  # minutes (1Hz grid)\n\n            stack = np.vstack([td.timeseries[w][a] for a in ANOMALIES])\n            anom_mean = np.mean(stack, axis=0)\n            anom_min = np.min(stack, axis=0)\n            anom_max = np.max(stack, axis=0)\n\n            ax.plot(x, nom, color="black", linewidth=2.4, label="Benign (NOMINAL)")\n            for a in ANOMALIES:\n                ax.plot(\n                    x,\n                    td.timeseries[w][a],\n                    color=COLOR_BY_STRESSOR[a],\n                    alpha=0.6,\n                    linewidth=1.0,\n                    label=a,\n                )\n            ax.plot(x, anom_mean, color="#c62828", linewidth=2.2, label="Anomaly mean")\n            ax.fill_between(x, anom_min, anom_max, color="#ef5350", alpha=0.18, label="Anomaly range")\n            ax.set_ylabel("AF Index", fontsize=14)\n            ax.set_xlabel("Time (minutes)", fontsize=14)\n            ax.set_title(w, fontsize=16, fontweight="bold")\n            ax.grid(alpha=0.25)\n            ax.tick_params(axis="both", labelsize=12)\n\n        h, l = axes[0].get_legend_handles_labels()\n        dedup = dict(zip(l, h))\n        fig.legend(\n            dedup.values(),\n            dedup.keys(),\n            loc="upper center",\n            ncol=4,\n            frameon=True,\n            fontsize=12,\n            bbox_to_anchor=(0.5, 1.02),\n        )\n        fig.suptitle(f"All-feature AF-index trajectories | {TIER_PRETTY[td.tier]}", fontsize=20, y=1.04)\n        fig.tight_layout(rect=[0, 0, 1, 0.98])\n\n        out = out_dir / f"fig_af_timeseries_{td.tier}.png"\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_auc_heatmaps(out_dir: Path, stressor: pd.DataFrame) -> List[str]:\n    out_paths = []\n    for metric, title, fname in [\n        ("roc_auc", "ROC-AUC by tier and stressor", "fig_heatmap_roc_auc.png"),\n        ("pr_auc", "AUC-PR by tier and stressor", "fig_heatmap_pr_auc.png"),\n    ]:\n        piv = stressor.pivot(index="stressor", columns="tier_name", values=metric).loc[ANOMALIES]\n        cols = [c for c in ["Tier-0", "Tier-1", "Tier-2"] if c in piv.columns]\n        piv = piv[cols]\n\n        fig, ax = plt.subplots(figsize=(8.5, 4.5))\n        im = ax.imshow(piv.to_numpy(dtype=float), vmin=0.5, vmax=1.0, cmap="viridis")\n        ax.set_xticks(np.arange(len(piv.columns)))\n        ax.set_xticklabels(piv.columns, fontsize=12)\n        ax.set_yticks(np.arange(len(piv.index)))\n        ax.set_yticklabels(piv.index, fontsize=12)\n        ax.set_title(title, fontsize=16, fontweight="bold")\n        for i in range(len(piv.index)):\n            for j in range(len(piv.columns)):\n                v = float(piv.iloc[i, j])\n                ax.text(j, i, f"{v:.3f}", ha="center", va="center", color="white", fontsize=11)\n        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n        cbar.ax.set_ylabel(metric.upper(), rotation=90, fontsize=11)\n        fig.tight_layout()\n        out = out_dir / fname\n        fig.savefig(out, dpi=240, bbox_inches="tight")\n        plt.close(fig)\n        out_paths.append(str(out))\n    return out_paths\n\n\ndef plot_run_score_distributions(out_dir: Path, runs: pd.DataFrame) -> str:\n    tiers = ["tier0", "tier1_alt", "tier2"]\n    fig, axes = plt.subplots(1, len(tiers), figsize=(14.5, 4.6), sharey=False)\n    if len(tiers) == 1:\n        axes = [axes]\n\n    for i, t in enumerate(tiers):\n        ax = axes[i]\n        d = runs[runs["tier"] == t]\n        nom = d[d["label"] == 0]["run_score_median"].to_numpy(dtype=float)\n        anm = d[d["label"] == 1]["run_score_median"].to_numpy(dtype=float)\n        bp = ax.boxplot([nom, anm], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(nom)), nom, color="black", s=24, alpha=0.8)\n        ax.scatter(np.repeat(2, len(anm)), anm, color="#c62828", s=24, alpha=0.7)\n        ax.set_title(TIER_PRETTY[t], fontsize=14, fontweight="bold")\n        ax.set_ylabel("Run AF Index (median)", fontsize=12)\n        ax.grid(alpha=0.22)\n        ax.tick_params(axis="both", labelsize=11)\n\n    fig.suptitle("Run-level AF-index score distributions", fontsize=18, y=1.02)\n    fig.tight_layout()\n    out = out_dir / "fig_run_score_distributions.png"\n    fig.savefig(out, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n    return str(out)\n\n\ndef build_feature_inventory(tier_data: Iterable[TierData]) -> pd.DataFrame:\n    rows = []\n    for td in tier_data:\n        rows.append(\n            {\n                "tier": td.tier,\n                "tier_name": TIER_PRETTY[td.tier],\n                "n_features_common": len(td.features),\n                "n_features_union": len(td.union_features),\n                "common_features_json": json.dumps(td.features),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef write_markdown_summary(\n    out_dir: Path,\n    overall: pd.DataFrame,\n    stressor: pd.DataFrame,\n    fig_paths: List[str],\n) -> None:\n    best_tier = overall.sort_values("pr_auc", ascending=False).iloc[0]\n    weakest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=True)\n        .head(2)\n        .index.tolist()\n    )\n    strongest = (\n        stressor.groupby("stressor")[["roc_auc", "pr_auc"]]\n        .mean()\n        .sort_values("pr_auc", ascending=False)\n        .head(3)\n        .index.tolist()\n    )\n    lines = []\n    lines.append("# DICE Results/Analysis Auto-Summary")\n    lines.append("")\n    lines.append("## Key Findings")\n    lines.append(\n        f"- Best run-level AUC-PR tier: **{best_tier[\'tier_name\']}** "\n        f"(AUC-PR={best_tier[\'pr_auc\']:.4f}, ROC-AUC={best_tier[\'roc_auc\']:.4f})."\n    )\n    lines.append(f"- Strongest stressors (mean AUC-PR across tiers): **{\', \'.join(strongest)}**.")\n    lines.append(f"- Hardest stressors (mean AUC-PR across tiers): **{\', \'.join(weakest)}**.")\n    lines.append("")\n    lines.append("## Suggested Results Narrative")\n    lines.append(\n        "Across the 24-run Apple dataset, AF-index separation is consistently visible between nominal and "\n        "anomalous runs in all telemetry tiers. Tier-aware scoring indicates that anomaly/nominal score ratios "\n        "remain above 1.0 in every tier, confirming stable separability under the fixed collection protocol. "\n        "Per-stressor analysis shows stronger separation for ATOMIC, CACHE, and MEMBW, while BRANCH and TLB "\n        "remain comparatively harder due to weaker host-visible signatures. These observations match the "\n        "expected mechanism-level difficulty ordering in software-driven stressors."\n    )\n    lines.append("")\n    lines.append("## Generated Figures")\n    for p in fig_paths:\n        lines.append(f"- `{p}`")\n    lines.append("")\n    lines.append("## Generated Tables")\n    for p in [\n        out_dir / "table_overall_metrics.csv",\n        out_dir / "table_stressor_metrics.csv",\n        out_dir / "table_workload_summary.csv",\n        out_dir / "table_feature_inventory.csv",\n        out_dir / "table_overall_metrics.tex",\n        out_dir / "table_stressor_metrics.tex",\n    ]:\n        lines.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(lines) + "\\n")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n        help="Dataset root containing tier0, tier1_alt, tier2 folders.",\n    )\n    ap.add_argument(\n        "--out_dir",\n        type=Path,\n        default=None,\n        help="Output directory for results tables/figures (default: <root>/results_analysis).",\n    )\n    ap.add_argument("--source_hz", type=int, default=5, help="Source sampling Hz used for downsampling.")\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = (args.out_dir.expanduser().resolve() if args.out_dir else (root / "results_analysis"))\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir = out_dir / "figures"\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_data = [build_tier_data(root, t, source_hz=args.source_hz) for t in ["tier0", "tier1_alt", "tier2"]]\n\n    runs = pd.concat([td.run_df for td in tier_data], ignore_index=True)\n    quality = pd.concat([td.case_quality for td in tier_data], ignore_index=True)\n    features = build_feature_inventory(tier_data)\n    overall = make_overall_metrics(tier_data)\n    stressor = make_stressor_metrics(tier_data)\n    workload = make_workload_summary(tier_data)\n\n    save_metric_tables(out_dir, overall, stressor, workload, features, quality, runs)\n\n    figs = []\n    figs.extend(plot_af_timeseries(fig_dir, tier_data))\n    figs.extend(plot_auc_heatmaps(fig_dir, stressor))\n    figs.append(plot_run_score_distributions(fig_dir, runs))\n\n    write_markdown_summary(out_dir, overall, stressor, figs)\n\n    print(f"[OK] Results generated at: {out_dir}")\n    print("[OK] Figures:")\n    for p in figs:\n        print(f" - {p}")\n    print("[OK] Tables:")\n    print(f" - {out_dir / \'table_overall_metrics.csv\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.csv\'}")\n    print(f" - {out_dir / \'table_workload_summary.csv\'}")\n    print(f" - {out_dir / \'table_overall_metrics.tex\'}")\n    print(f" - {out_dir / \'table_stressor_metrics.tex\'}")\n\n\nif __name__ == "__main__":\n    main()\n'
NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = '#!/usr/bin/env python3\n"""\nFull retrain/evaluation for DICE micro-twin + split-conformal pipeline.\n\nProtocol:\n- Use Tier-0 / Tier-1-alt / Tier-2 clean dataset (5000 rows @ 5Hz per run).\n- Align to 1Hz via mean pooling.\n- Train only on benign runs (NOMINAL) with workload-holdout folds.\n- Fit linear micro-twin dynamics in normalized feature space.\n- Build residual signatures on decision blocks.\n- Calibrate conformal threshold on benign calibration blocks.\n- Evaluate run-level labels (Benign vs Anomaly) via persistent block alerts.\n\nOutputs:\n- CSV metrics tables and per-case predictions\n- LaTeX table snippets for paper\n- ROC/PR and score distribution figures\n- Markdown summary for direct paste into Results section\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Sequence, Tuple\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    roc_auc_score,\n    roc_curve,\n)\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nREPO_ROOT = PROJECT_ROOT\nDEFAULT_DATASET_ROOT = REPO_ROOT / "data generation" / "dataset" / "ITC_M2Pro_DATA"\n\n\nWORKLOADS = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]\nSTRESSORS = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]\nANOMALIES = [s for s in STRESSORS if s != "NOMINAL"]\n\nIGNORE_COLS = {"idx", "ts_unix_s", "t_rel_s", "timestamp", "time", "ts"}\n\nTIER_FILE = {\n    "tier0": "tier0_full_5hz.csv",\n    "tier1_alt": "tier1_alt_core_5hz.csv",\n    "tier2": "tier2_core_5hz.csv",\n}\n\nCONFIGS = {\n    "tier0": ["tier0"],\n    "tier0_tier1": ["tier0", "tier1_alt"],\n    "tier0_tier1_tier2": ["tier0", "tier1_alt", "tier2"],\n}\n\nDIAG_TOP_K = 5\nMECHANISM_GROUPS = [\n    "compute",\n    "memory_io",\n    "thermal_power",\n    "scheduler_runtime",\n    "platform_pressure",\n]\n\n\n@dataclass(frozen=True)\nclass CaseRef:\n    workload: str\n    stressor: str\n\n    @property\n    def case_id(self) -> str:\n        return f"{self.workload}__{self.stressor}"\n\n    @property\n    def label(self) -> int:\n        return 0 if self.stressor == "NOMINAL" else 1\n\n\n@dataclass\nclass ModelBundle:\n    feature_names: List[str]\n    median: np.ndarray\n    scale: np.ndarray\n    A: np.ndarray\n    weights: np.ndarray\n    cal_scores: np.ndarray\n    tau: float\n\n\ndef all_cases() -> List[CaseRef]:\n    return [CaseRef(w, s) for w in WORKLOADS for s in STRESSORS]\n\n\ndef robust_scale_1d(x: np.ndarray) -> float:\n    x = np.asarray(x, dtype=float)\n    x = x[np.isfinite(x)]\n    if x.size == 0:\n        return 1.0\n    q75, q25 = np.percentile(x, [75, 25])\n    iqr = q75 - q25\n    if iqr > 1e-12:\n        return float(iqr / 1.349)\n    med = np.median(x)\n    mad = np.median(np.abs(x - med))\n    if mad > 1e-12:\n        return float(1.4826 * mad)\n    std = float(np.std(x))\n    if std > 1e-12:\n        return std\n    return 1.0\n\n\ndef robust_fit_matrix(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    med = np.nanmedian(X, axis=0)\n    scale = np.zeros(X.shape[1], dtype=float)\n    for j in range(X.shape[1]):\n        scale[j] = robust_scale_1d(X[:, j])\n    scale[scale <= 1e-12] = 1.0\n    return med, scale\n\n\ndef safe_auc(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, score))\n\n\ndef safe_ap(y_true: np.ndarray, score: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(average_precision_score(y_true, score))\n\n\ndef case_path(root: Path, tier: str, case: CaseRef) -> Path:\n    return root / tier / case.case_id / TIER_FILE[tier]\n\n\ndef read_df(path: Path) -> pd.DataFrame:\n    if not path.exists():\n        raise FileNotFoundError(f"Missing file: {path}")\n    df = pd.read_csv(path)\n    for c in df.columns:\n        if pd.api.types.is_numeric_dtype(df[c]):\n            df[c] = pd.to_numeric(df[c], errors="coerce")\n    return df\n\n\ndef numeric_features(df: pd.DataFrame) -> List[str]:\n    out = []\n    for c in df.columns:\n        if c in IGNORE_COLS:\n            continue\n        if pd.api.types.is_numeric_dtype(df[c]):\n            out.append(c)\n    return out\n\n\ndef downsample_1hz(df: pd.DataFrame, source_hz: int = 5) -> pd.DataFrame:\n    if len(df) < source_hz:\n        return df.copy()\n    n = (len(df) // source_hz) * source_hz\n    tmp = df.iloc[:n].copy()\n    grp = np.arange(n) // source_hz\n    return tmp.groupby(grp, sort=False).mean(numeric_only=True)\n\n\ndef common_features_per_tier(root: Path, tier: str) -> List[str]:\n    common = None\n    for case in all_cases():\n        df = read_df(case_path(root, tier, case))\n        cols = set(numeric_features(df))\n        common = cols if common is None else (common & cols)\n    common_list = sorted(common) if common else []\n\n    # Drop globally constant features.\n    keep = []\n    for f in common_list:\n        vals = []\n        for case in all_cases():\n            d = downsample_1hz(read_df(case_path(root, tier, case)))\n            vals.append(d[f].to_numpy(dtype=float))\n        x = np.concatenate(vals)\n        if np.nanstd(x) > 1e-10:\n            keep.append(f)\n    return keep\n\n\ndef build_case_matrix(\n    root: Path,\n    case: CaseRef,\n    tiers: Sequence[str],\n    feature_map: Dict[str, List[str]],\n    source_hz: int = 5,\n) -> Tuple[np.ndarray, List[str]]:\n    mats = []\n    names = []\n    lengths = []\n    for t in tiers:\n        df = downsample_1hz(read_df(case_path(root, t, case)), source_hz=source_hz)\n        feats = feature_map[t]\n        arr = df[feats].to_numpy(dtype=float)\n        mats.append(arr)\n        lengths.append(arr.shape[0])\n        names.extend([f"{t}:{f}" for f in feats])\n\n    n = min(lengths)\n    mats = [m[:n] for m in mats]\n    X = np.concatenate(mats, axis=1)\n    return X, names\n\n\ndef fit_linear_dynamics(X_runs: List[np.ndarray], ridge_lambda: float = 1e-3) -> np.ndarray:\n    X_prev = []\n    X_next = []\n    for X in X_runs:\n        if len(X) < 2:\n            continue\n        X_prev.append(X[:-1])\n        X_next.append(X[1:])\n    if not X_prev:\n        raise RuntimeError("Not enough samples to fit dynamics.")\n    P = np.vstack(X_prev)  # [N, d]\n    N = np.vstack(X_next)  # [N, d]\n    d = P.shape[1]\n    xtx = P.T @ P + ridge_lambda * np.eye(d)\n    xty = P.T @ N\n    A = np.linalg.solve(xtx, xty)  # [d, d]\n    return A\n\n\ndef residual_timeseries(X_norm: np.ndarray, A: np.ndarray, gain: float) -> np.ndarray:\n    """\n    Kalman-style fixed-gain synchronization:\n    z_pred = A z_prev\n    r_t    = x_t - z_pred\n    z_t    = z_pred + gain * r_t\n    """\n    T, d = X_norm.shape\n    if T < 2:\n        return np.zeros((0, d), dtype=float)\n    z = X_norm[0].copy()\n    residuals = []\n    for t in range(1, T):\n        z_pred = z @ A\n        r = X_norm[t] - z_pred\n        residuals.append(r)\n        z = z_pred + gain * r\n    return np.vstack(residuals)\n\n\ndef block_signatures(residual: np.ndarray, B: int) -> np.ndarray:\n    """\n    Signature per block: mean absolute residual over a sliding window.\n    """\n    if residual.shape[0] == 0:\n        return np.zeros((0, residual.shape[1]), dtype=float)\n    a = np.abs(residual)\n    T, d = a.shape\n    if T < B:\n        return np.mean(a, axis=0, keepdims=True)\n    cs = np.vstack([np.zeros((1, d)), np.cumsum(a, axis=0)])\n    out = (cs[B:] - cs[:-B]) / float(B)\n    return out\n\n\ndef fit_weights(signatures_fit: np.ndarray) -> np.ndarray:\n    sigma = np.std(signatures_fit, axis=0)\n    w = 1.0 / (sigma + 1e-6)\n    w = np.maximum(w, 0.0)\n    s = np.sum(w)\n    if s <= 0:\n        return np.ones_like(w) / len(w)\n    return w / s\n\n\ndef signature_scores(signatures: np.ndarray, weights: np.ndarray) -> np.ndarray:\n    if signatures.shape[0] == 0:\n        return np.zeros((0,), dtype=float)\n    return signatures @ weights\n\n\ndef conformal_threshold(cal_scores: np.ndarray, alpha: float) -> float:\n    sc = np.sort(np.asarray(cal_scores, dtype=float))\n    n = len(sc)\n    if n == 0:\n        return float("inf")\n    k = int(np.ceil((n + 1) * (1.0 - alpha)))\n    k = min(max(k, 1), n)\n    return float(sc[k - 1])\n\n\ndef conformal_pvals(cal_scores: np.ndarray, test_scores: np.ndarray) -> np.ndarray:\n    cal = np.asarray(cal_scores, dtype=float)\n    denom = len(cal) + 1.0\n    out = np.zeros(len(test_scores), dtype=float)\n    for i, s in enumerate(test_scores):\n        out[i] = (1.0 + np.sum(cal >= s)) / denom\n    return out\n\n\ndef persistent_alerts(alerts: np.ndarray, k: int) -> np.ndarray:\n    out = np.zeros(len(alerts), dtype=int)\n    run = 0\n    for i, a in enumerate(alerts.astype(bool)):\n        if a:\n            run += 1\n        else:\n            run = 0\n        out[i] = 1 if run >= k else 0\n    return out\n\n\ndef first_positive_index(x: np.ndarray) -> int:\n    idx = np.flatnonzero(np.asarray(x, dtype=bool))\n    return int(idx[0]) if len(idx) else -1\n\n\ndef finite_median(x: Sequence[float]) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.median(arr))\n\n\ndef finite_percentile(x: Sequence[float], q: float) -> float:\n    arr = np.asarray(x, dtype=float)\n    arr = arr[np.isfinite(arr)]\n    if arr.size == 0:\n        return float("nan")\n    return float(np.percentile(arr, q))\n\n\ndef mechanism_group(feature_name: str) -> str:\n    name = feature_name.split(":", 1)[-1].lower()\n    if any(tok in name for tok in ["temp", "power", "fan"]):\n        return "thermal_power"\n    if any(tok in name for tok in ["mem_", "swap_", "disk_", "net_", "wired_bytes", "active_bytes", "inactive_bytes"]):\n        return "memory_io"\n    if any(\n        tok in name\n        for tok in [\n            "ctx_switch",\n            "interrupt",\n            "syscall",\n            "pids_count",\n            "running_fraction",\n            "weight_ns",\n            "unique_process",\n            "unique_thread",\n            "samples_per_bucket",\n            "sentinel_count",\n            "core_id",\n        ]\n    ):\n        return "scheduler_runtime"\n    if any(tok in name for tok in ["load", "uptime", "available_bytes", "free_bytes", "mem_percent"]):\n        return "platform_pressure"\n    return "compute"\n\n\ndef mechanism_vector(feature_names: Sequence[str], feature_contrib: np.ndarray) -> Tuple[Dict[str, float], np.ndarray]:\n    totals = {group: 0.0 for group in MECHANISM_GROUPS}\n    for name, value in zip(feature_names, np.asarray(feature_contrib, dtype=float)):\n        totals[mechanism_group(name)] += float(value)\n    vec = np.array([totals[group] for group in MECHANISM_GROUPS], dtype=float)\n    return totals, vec\n\n\ndef train_bundle(\n    train_benign_runs: Dict[str, np.ndarray],\n    feature_names: List[str],\n    fit_ratio: float,\n    B: int,\n    alpha: float,\n    gain: float,\n    ridge_lambda: float,\n) -> ModelBundle:\n    fit_runs = []\n    cal_runs = []\n    fit_samples = []\n\n    for _, X in train_benign_runs.items():\n        n = len(X)\n        split = int(max(2, min(n - 1, round(n * fit_ratio))))\n        X_fit = X[:split]\n        X_cal = X[split:]\n        fit_runs.append(X_fit)\n        cal_runs.append(X_cal if len(X_cal) > 1 else X_fit[-2:])\n        fit_samples.append(X_fit)\n\n    X_fit_all = np.vstack(fit_samples)\n    med, scale = robust_fit_matrix(X_fit_all)\n\n    fit_norm = [(x - med) / (scale + 1e-12) for x in fit_runs]\n    cal_norm = [(x - med) / (scale + 1e-12) for x in cal_runs]\n\n    A = fit_linear_dynamics(fit_norm, ridge_lambda=ridge_lambda)\n\n    sig_fit = []\n    for X in fit_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        if len(s):\n            sig_fit.append(s)\n    sig_fit_all = np.vstack(sig_fit)\n    w = fit_weights(sig_fit_all)\n\n    cal_scores = []\n    for X in cal_norm:\n        r = residual_timeseries(X, A, gain=gain)\n        s = block_signatures(r, B=B)\n        sc = signature_scores(s, w)\n        if len(sc):\n            cal_scores.append(sc)\n    cal_scores_all = np.concatenate(cal_scores)\n    tau = conformal_threshold(cal_scores_all, alpha=alpha)\n\n    return ModelBundle(\n        feature_names=feature_names,\n        median=med,\n        scale=scale,\n        A=A,\n        weights=w,\n        cal_scores=cal_scores_all,\n        tau=tau,\n    )\n\n\ndef evaluate_run(\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> Tuple[Dict[str, float], np.ndarray]:\n    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)\n    r = residual_timeseries(Xn, bundle.A, gain=gain)\n    sig = block_signatures(r, B=B)\n    sc = signature_scores(sig, bundle.weights)\n    pv = conformal_pvals(bundle.cal_scores, sc)\n\n    block_alert = pv < alpha\n    persist = persistent_alerts(block_alert, k=persist_k)\n\n    run_alert = int(np.any(persist > 0))\n    peak_score = float(np.max(sc)) if len(sc) else 0.0\n    run_score = peak_score\n    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)\n    feature_contrib = run_signature * bundle.weights\n    first_block_idx = first_positive_index(block_alert)\n    first_persist_idx = first_positive_index(persist)\n    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")\n    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")\n    n_blocks = int(len(sc))\n    duration_s = max(n_blocks, 1)\n\n    return (\n        {\n            "run_score": run_score,\n            "run_alert": run_alert,\n            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,\n            "peak_block_score": peak_score,\n            "n_blocks": n_blocks,\n            "n_block_alerts": int(np.sum(block_alert)),\n            "n_persist_alerts": int(np.sum(persist)),\n            "first_block_alert_idx": first_block_idx,\n            "first_persist_alert_idx": first_persist_idx,\n            "first_block_alert_s": block_time_s,\n            "time_to_detect_s": persist_time_s,\n            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),\n            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),\n        },\n        feature_contrib,\n    )\n\n\ndef to_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:\n    body = df.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}")\n    return (\n        "\\\\begin{table}[t]\\n"\n        "\\\\centering\\n"\n        f"\\\\caption{{{caption}}}\\n"\n        f"\\\\label{{{label}}}\\n"\n        "\\\\footnotesize\\n"\n        f"{body}\\n"\n        "\\\\end{table}\\n"\n    )\n\n\ndef plot_curves(df: pd.DataFrame, out_png: Path, score_col: str, title_tag: str) -> None:\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    for cfg, d in df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s = d[score_col].to_numpy(dtype=float)\n        if len(np.unique(y)) < 2:\n            continue\n        fpr, tpr, _ = roc_curve(y, s)\n        p, r, _ = precision_recall_curve(y, s)\n        axes[0].plot(fpr, tpr, linewidth=2, label=f"{cfg} (AUC={roc_auc_score(y, s):.3f})")\n        axes[1].plot(r, p, linewidth=2, label=f"{cfg} (AP={average_precision_score(y, s):.3f})")\n    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)\n    axes[0].set_title("ROC curve")\n    axes[0].set_xlabel("False Positive Rate")\n    axes[0].set_ylabel("True Positive Rate")\n    axes[1].set_title("Precision-Recall curve")\n    axes[1].set_xlabel("Recall")\n    axes[1].set_ylabel("Precision")\n    for ax in axes:\n        ax.grid(alpha=0.25)\n        ax.legend(frameon=True, fontsize=10)\n    fig.suptitle(f"DICE curves ({title_tag})")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_score_box(df: pd.DataFrame, out_png: Path, score_col: str, y_label: str, title_tag: str) -> None:\n    cfgs = list(df["config"].unique())\n    fig, axes = plt.subplots(1, len(cfgs), figsize=(5.0 * len(cfgs), 4.8), sharey=False)\n    if len(cfgs) == 1:\n        axes = [axes]\n    for i, cfg in enumerate(cfgs):\n        ax = axes[i]\n        d = df[df["config"] == cfg]\n        neg = d[d["label"] == 0][score_col].to_numpy(dtype=float)\n        pos = d[d["label"] == 1][score_col].to_numpy(dtype=float)\n        bp = ax.boxplot([neg, pos], tick_labels=["Benign", "Anomaly"], patch_artist=True)\n        for patch, color in zip(bp["boxes"], ["#9e9e9e", "#ef9a9a"]):\n            patch.set_facecolor(color)\n            patch.set_alpha(0.8)\n        ax.scatter(np.repeat(1, len(neg)), neg, color="black", s=22, alpha=0.8)\n        ax.scatter(np.repeat(2, len(pos)), pos, color="#c62828", s=22, alpha=0.7)\n        ax.set_title(cfg)\n        ax.set_ylabel(y_label)\n        ax.grid(alpha=0.22)\n    fig.suptitle(f"DICE run score distributions ({title_tag})")\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef build_diagnostic_record(\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    feature_names: Sequence[str],\n    feature_contrib: np.ndarray,\n) -> Dict[str, object]:\n    contrib = np.asarray(feature_contrib, dtype=float)\n    total = float(np.sum(contrib))\n    tier_totals = {tier: 0.0 for tier in TIER_FILE}\n    for name, value in zip(feature_names, contrib):\n        tier = name.split(":", 1)[0]\n        if tier in tier_totals:\n            tier_totals[tier] += float(value)\n    dominant_tier = max(tier_totals, key=tier_totals.get) if total > 0 else "none"\n    mech_totals, mech_vec = mechanism_vector(feature_names, contrib)\n    dominant_mechanism = max(mech_totals, key=mech_totals.get) if total > 0 else "none"\n    order = np.argsort(contrib)[::-1][:DIAG_TOP_K]\n    mech_order = np.argsort(mech_vec)[::-1][:3]\n\n    row: Dict[str, object] = {\n        "config": config,\n        "holdout_workload": holdout_workload,\n        "case_id": case.case_id,\n        "workload": case.workload,\n        "stressor": case.stressor,\n        "label": case.label,\n        "dominant_tier": dominant_tier,\n        "tier0_contrib": float(tier_totals["tier0"]),\n        "tier1_alt_contrib": float(tier_totals["tier1_alt"]),\n        "tier2_contrib": float(tier_totals["tier2"]),\n        "tier0_share": float(tier_totals["tier0"] / total) if total > 0 else 0.0,\n        "tier1_alt_share": float(tier_totals["tier1_alt"] / total) if total > 0 else 0.0,\n        "tier2_share": float(tier_totals["tier2"] / total) if total > 0 else 0.0,\n        "dominant_mechanism": dominant_mechanism,\n        "_feature_contrib": contrib.copy(),\n        "_mechanism_vector": mech_vec.copy(),\n    }\n    for group in MECHANISM_GROUPS:\n        row[f"{group}_contrib"] = float(mech_totals[group])\n        row[f"{group}_share"] = float(mech_totals[group] / total) if total > 0 else 0.0\n    for rank in range(DIAG_TOP_K):\n        key_name = f"top_feature_{rank + 1}"\n        key_score = f"top_feature_score_{rank + 1}"\n        if rank < len(order) and contrib[order[rank]] > 0.0:\n            idx = int(order[rank])\n            row[key_name] = feature_names[idx]\n            row[key_score] = float(contrib[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    for rank in range(3):\n        key_name = f"top_mechanism_{rank + 1}"\n        key_score = f"top_mechanism_score_{rank + 1}"\n        if rank < len(mech_order) and mech_vec[mech_order[rank]] > 0.0:\n            idx = int(mech_order[rank])\n            row[key_name] = MECHANISM_GROUPS[idx]\n            row[key_score] = float(mech_vec[idx])\n        else:\n            row[key_name] = ""\n            row[key_score] = 0.0\n    return row\n\n\ndef append_case_outputs(\n    preds: List[Dict[str, object]],\n    diagnostic_records: List[Dict[str, object]],\n    config: str,\n    holdout_workload: str,\n    case: CaseRef,\n    X_run: np.ndarray,\n    bundle: ModelBundle,\n    B: int,\n    alpha: float,\n    persist_k: int,\n    gain: float,\n) -> None:\n    metrics, feature_contrib = evaluate_run(\n        X_run,\n        bundle,\n        B=B,\n        alpha=alpha,\n        persist_k=persist_k,\n        gain=gain,\n    )\n    preds.append(\n        {\n            "config": config,\n            "holdout_workload": holdout_workload,\n            "case_id": case.case_id,\n            "workload": case.workload,\n            "stressor": case.stressor,\n            "label": case.label,\n            **metrics,\n            "n_features": len(bundle.feature_names),\n            "tau": bundle.tau,\n        }\n    )\n    diagnostic_records.append(\n        build_diagnostic_record(\n            config=config,\n            holdout_workload=holdout_workload,\n            case=case,\n            feature_names=bundle.feature_names,\n            feature_contrib=feature_contrib,\n        )\n    )\n\n\ndef build_stressor_attribution(\n    diagnostic_records: Sequence[Dict[str, object]],\n    vector_key: str,\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    pred_rows: List[Dict[str, object]] = []\n    for cfg_name in CONFIGS:\n        cfg_records = [r for r in diagnostic_records if r["config"] == cfg_name and int(r["label"]) == 1]\n        for holdout_w in WORKLOADS:\n            train = [r for r in cfg_records if r["workload"] != holdout_w]\n            test = [r for r in cfg_records if r["workload"] == holdout_w]\n            centroids = {}\n            for stressor in ANOMALIES:\n                mats = [r[vector_key] for r in train if r["stressor"] == stressor]\n                if mats:\n                    centroids[stressor] = np.median(np.vstack(mats), axis=0)\n            if len(centroids) < 2:\n                continue\n            for row in test:\n                truth = str(row["stressor"])\n                contrib = np.asarray(row[vector_key], dtype=float)\n                dists = {stressor: float(np.linalg.norm(contrib - centroid)) for stressor, centroid in centroids.items()}\n                ordered = sorted(dists.items(), key=lambda item: item[1])\n                pred = ordered[0][0]\n                top2 = [label for label, _ in ordered[:2]]\n                pred_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "case_id": row["case_id"],\n                        "true_stressor": truth,\n                        "pred_stressor": pred,\n                        "is_correct": int(pred == truth),\n                        "top2_hit": int(truth in top2),\n                        "nearest_distance": float(ordered[0][1]),\n                        "margin_to_second": float(ordered[1][1] - ordered[0][1]) if len(ordered) > 1 else float("inf"),\n                    }\n                )\n\n    pred_df = pd.DataFrame(pred_rows)\n    if pred_df.empty:\n        empty_metrics = pd.DataFrame(\n            columns=["config", "n_cases", "top1_acc", "top2_acc", "macro_f1", "mean_margin_to_second"]\n        )\n        empty_cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n        empty_cm.index.name = "true_stressor"\n        empty_cm.columns.name = "pred_stressor"\n        return pred_df, empty_metrics, empty_cm\n    pred_df = pred_df.sort_values(["config", "holdout_workload", "case_id"])\n\n    metric_rows = []\n    for cfg_name, d in pred_df.groupby("config", sort=False):\n        metric_rows.append(\n            {\n                "config": cfg_name,\n                "n_cases": int(len(d)),\n                "top1_acc": float(d["is_correct"].mean()),\n                "top2_acc": float(d["top2_hit"].mean()),\n                "macro_f1": float(\n                    f1_score(\n                        d["true_stressor"],\n                        d["pred_stressor"],\n                        labels=ANOMALIES,\n                        average="macro",\n                        zero_division=0,\n                    )\n                ),\n                "mean_margin_to_second": float(d["margin_to_second"].replace([np.inf, -np.inf], np.nan).mean()),\n            }\n        )\n    metrics_df = pd.DataFrame(metric_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    d_final = pred_df[pred_df["config"] == final_cfg]\n    if d_final.empty:\n        cm = pd.DataFrame(index=ANOMALIES, columns=ANOMALIES, data=0)\n    else:\n        cm_arr = confusion_matrix(\n            d_final["true_stressor"],\n            d_final["pred_stressor"],\n            labels=ANOMALIES,\n        )\n        cm = pd.DataFrame(cm_arr, index=ANOMALIES, columns=ANOMALIES)\n    cm.index.name = "true_stressor"\n    cm.columns.name = "pred_stressor"\n    return pred_df, metrics_df, cm\n\n\ndef build_stressor_tier_contributions(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    cols = ["tier0_share", "tier1_alt_share", "tier2_share"]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *cols, "dominant_tier_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_tier"].mode()\n        rows.append(\n            {\n                "stressor": stressor,\n                "tier0_share": float(part["tier0_share"].mean()),\n                "tier1_alt_share": float(part["tier1_alt_share"].mean()),\n                "tier2_share": float(part["tier2_share"].mean()),\n                "dominant_tier_mode": str(mode.iloc[0]) if not mode.empty else "none",\n            }\n        )\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_mechanism_summary(diag_df: pd.DataFrame, config: str) -> pd.DataFrame:\n    share_cols = [f"{group}_share" for group in MECHANISM_GROUPS]\n    d = diag_df[(diag_df["config"] == config) & (diag_df["label"] == 1)].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["stressor", *share_cols, "dominant_mechanism_mode"])\n    rows = []\n    for stressor, part in d.groupby("stressor", sort=True):\n        mode = part["dominant_mechanism"].mode()\n        row = {\n            "stressor": stressor,\n            "dominant_mechanism_mode": str(mode.iloc[0]) if not mode.empty else "none",\n        }\n        for col in share_cols:\n            row[col] = float(part[col].mean())\n        rows.append(row)\n    return pd.DataFrame(rows).sort_values("stressor")\n\n\ndef build_sequential_metrics(pred_df: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for cfg, d in pred_df.groupby("config", sort=False):\n        benign = d[d["label"] == 0]\n        anomaly = d[d["label"] == 1]\n        detected = anomaly[anomaly["run_alert"] == 1]\n        rows.append(\n            {\n                "config": cfg,\n                "benign_run_alert_rate": float(benign["run_alert"].mean()),\n                "benign_persist_alerts_per_hour": float(benign["persist_alerts_per_hour"].mean()),\n                "benign_block_alerts_per_hour": float(benign["block_alerts_per_hour"].mean()),\n                "anomaly_detect_rate": float(anomaly["run_alert"].mean()),\n                "median_time_to_detect_s": finite_median(detected["time_to_detect_s"]),\n                "p90_time_to_detect_s": finite_percentile(detected["time_to_detect_s"], 90),\n                "detect_within_120s": float((anomaly["time_to_detect_s"] <= 120).fillna(False).mean()),\n                "detect_within_300s": float((anomaly["time_to_detect_s"] <= 300).fillna(False).mean()),\n                "detect_within_600s": float((anomaly["time_to_detect_s"] <= 600).fillna(False).mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef build_holdout_robustness_summary(fold_df: pd.DataFrame) -> pd.DataFrame:\n    d = fold_df[fold_df["holdout_workload"] != "ALL"].copy()\n    if d.empty:\n        return pd.DataFrame(columns=["config", "mean_pr_auc", "worst_pr_auc", "mean_roc_auc", "mean_fpr", "mean_tpr"])\n    rows = []\n    for cfg, part in d.groupby("config", sort=False):\n        rows.append(\n            {\n                "config": cfg,\n                "mean_pr_auc": float(part["pr_auc"].mean()),\n                "worst_pr_auc": float(part["pr_auc"].min()),\n                "mean_roc_auc": float(part["roc_auc"].mean()),\n                "mean_fpr": float(part["fpr"].mean()),\n                "mean_tpr": float(part["tpr"].mean()),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("config")\n\n\ndef plot_confusion_heatmap(cm: pd.DataFrame, out_png: Path, title: str) -> None:\n    if cm.empty:\n        return\n    mat = cm.to_numpy(dtype=float)\n    fig, ax = plt.subplots(figsize=(6.2, 5.2))\n    im = ax.imshow(mat, cmap="Blues")\n    ax.set_xticks(np.arange(len(cm.columns)), labels=list(cm.columns), rotation=30, ha="right")\n    ax.set_yticks(np.arange(len(cm.index)), labels=list(cm.index))\n    ax.set_xlabel("Predicted stressor")\n    ax.set_ylabel("True stressor")\n    ax.set_title(title)\n    for i in range(mat.shape[0]):\n        for j in range(mat.shape[1]):\n            color = "white" if mat[i, j] >= max(1.0, np.max(mat) * 0.55) else "black"\n            ax.text(j, i, f"{int(mat[i, j])}", ha="center", va="center", color=color, fontsize=10)\n    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_stressor_tier_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(7.4, 4.8))\n    x = np.arange(len(df))\n    bottom = np.zeros(len(df), dtype=float)\n    series = [\n        ("tier0_share", "Tier-0", "#78909c"),\n        ("tier1_alt_share", "Tier-1", "#81c784"),\n        ("tier2_share", "Tier-2", "#ffb74d"),\n    ]\n    for col, label, color in series:\n        vals = df[col].to_numpy(dtype=float)\n        ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", linewidth=0.8)\n        bottom += vals\n    ax.set_xticks(x, labels=df["stressor"].tolist())\n    ax.set_ylim(0.0, 1.0)\n    ax.set_ylabel("Mean contribution share")\n    ax.set_title("Final-config diagnosis contribution share by tier")\n    ax.legend(frameon=True)\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_mechanism_shares(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(8.6, 5.0))\n    x = np.arange(len(df))\n    bottom = np.zeros(len(df), dtype=float)\n    series = [\n        ("compute_share", "Compute", "#5c6bc0"),\n        ("memory_io_share", "Memory/I/O", "#26a69a"),\n        ("thermal_power_share", "Thermal/Power", "#ef5350"),\n        ("scheduler_runtime_share", "Scheduler/Runtime", "#8d6e63"),\n        ("platform_pressure_share", "Platform Pressure", "#78909c"),\n    ]\n    for col, label, color in series:\n        vals = df[col].to_numpy(dtype=float)\n        ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", linewidth=0.8)\n        bottom += vals\n    ax.set_xticks(x, labels=df["stressor"].tolist())\n    ax.set_ylim(0.0, 1.0)\n    ax.set_ylabel("Mean mechanism share")\n    ax.set_title("Final-config mechanism diagnosis share by stressor")\n    ax.legend(frameon=True, ncol=2)\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef plot_detection_latency(df: pd.DataFrame, out_png: Path) -> None:\n    if df.empty:\n        return\n    fig, ax = plt.subplots(figsize=(7.0, 4.8))\n    vals = df["median_time_to_detect_s"].to_numpy(dtype=float)\n    ax.bar(df["config"], vals, color=["#90a4ae", "#66bb6a", "#ffa726"][: len(df)])\n    ax.set_ylabel("Median time-to-detect (s)")\n    ax.set_title("Sequential detection latency by observation head")\n    ax.grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(out_png, dpi=240, bbox_inches="tight")\n    plt.close(fig)\n\n\ndef append_text_table(lines: List[str], df: pd.DataFrame) -> None:\n    lines.append("```text")\n    lines.append(df.to_string(index=False))\n    lines.append("```")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\n        "--root",\n        type=Path,\n        default=DEFAULT_DATASET_ROOT,\n    )\n    ap.add_argument("--out_dir", type=Path, default=None)\n    ap.add_argument("--source_hz", type=int, default=5)\n    ap.add_argument("--fit_ratio", type=float, default=0.6)\n    ap.add_argument("--block_B", type=int, default=60)\n    ap.add_argument("--alpha", type=float, default=0.05)\n    ap.add_argument("--persist_k", type=int, default=3)\n    ap.add_argument("--gain", type=float, default=0.35)\n    ap.add_argument("--ridge_lambda", type=float, default=1e-3)\n    ap.add_argument(\n        "--protocol",\n        choices=["workload_holdout", "global"],\n        default="global",\n        help="Evaluation protocol: workload_holdout (strict) or global benign split (paper-style).",\n    )\n    args = ap.parse_args()\n\n    root = args.root.expanduser().resolve()\n    out_dir = args.out_dir.expanduser().resolve() if args.out_dir else root / "results_dice_full"\n    fig_dir = out_dir / "figures"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    fig_dir.mkdir(parents=True, exist_ok=True)\n\n    tier_features = {t: common_features_per_tier(root, t) for t in TIER_FILE.keys()}\n    for t, fs in tier_features.items():\n        print(f"[INFO] {t}: common features={len(fs)}")\n\n    preds = []\n    fold_rows = []\n    diagnostic_records: List[Dict[str, object]] = []\n\n    for cfg_name, tiers in CONFIGS.items():\n        print(f"[INFO] training config={cfg_name} tiers={tiers}")\n        case_X = {}\n        feature_names_cfg = None\n        for case in all_cases():\n            X, names = build_case_matrix(\n                root,\n                case,\n                tiers=tiers,\n                feature_map=tier_features,\n                source_hz=args.source_hz,\n            )\n            case_X[case.case_id] = X\n            if feature_names_cfg is None:\n                feature_names_cfg = names\n\n        if args.protocol == "workload_holdout":\n            for holdout_w in WORKLOADS:\n                train_benign = {\n                    case_id: X\n                    for case_id, X in case_X.items()\n                    if case_id.endswith("__NOMINAL") and not case_id.startswith(f"{holdout_w}__")\n                }\n\n                bundle = train_bundle(\n                    train_benign_runs=train_benign,\n                    feature_names=feature_names_cfg or [],\n                    fit_ratio=args.fit_ratio,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    gain=args.gain,\n                    ridge_lambda=args.ridge_lambda,\n                )\n\n                test_cases = [c for c in all_cases() if c.workload == holdout_w]\n                for case in test_cases:\n                    append_case_outputs(\n                        preds=preds,\n                        diagnostic_records=diagnostic_records,\n                        config=cfg_name,\n                        holdout_workload=holdout_w,\n                        case=case,\n                        X_run=case_X[case.case_id],\n                        bundle=bundle,\n                        B=args.block_B,\n                        alpha=args.alpha,\n                        persist_k=args.persist_k,\n                        gain=args.gain,\n                    )\n\n                fold_curr = [p for p in preds if p["config"] == cfg_name and p["holdout_workload"] == holdout_w]\n                fd = pd.DataFrame(fold_curr)\n                y = fd["label"].to_numpy(dtype=int)\n                s_run = fd["run_score"].to_numpy(dtype=float)\n                fold_rows.append(\n                    {\n                        "config": cfg_name,\n                        "holdout_workload": holdout_w,\n                        "roc_auc": safe_auc(y, s_run),\n                        "pr_auc": safe_ap(y, s_run),\n                        "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                        "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                        "n_features": int(fd["n_features"].iloc[0]),\n                    }\n                )\n        else:\n            train_benign = {case_id: X for case_id, X in case_X.items() if case_id.endswith("__NOMINAL")}\n            bundle = train_bundle(\n                train_benign_runs=train_benign,\n                feature_names=feature_names_cfg or [],\n                fit_ratio=args.fit_ratio,\n                B=args.block_B,\n                alpha=args.alpha,\n                gain=args.gain,\n                ridge_lambda=args.ridge_lambda,\n            )\n            for case in all_cases():\n                append_case_outputs(\n                    preds=preds,\n                    diagnostic_records=diagnostic_records,\n                    config=cfg_name,\n                    holdout_workload="ALL",\n                    case=case,\n                    X_run=case_X[case.case_id],\n                    bundle=bundle,\n                    B=args.block_B,\n                    alpha=args.alpha,\n                    persist_k=args.persist_k,\n                    gain=args.gain,\n                )\n\n            fd = pd.DataFrame([p for p in preds if p["config"] == cfg_name])\n            y = fd["label"].to_numpy(dtype=int)\n            s_run = fd["run_score"].to_numpy(dtype=float)\n            fold_rows.append(\n                {\n                    "config": cfg_name,\n                    "holdout_workload": "ALL",\n                    "roc_auc": safe_auc(y, s_run),\n                    "pr_auc": safe_ap(y, s_run),\n                    "fpr": float(np.mean((fd["label"] == 0) & (fd["run_alert"] == 1))),\n                    "tpr": float(np.mean((fd["label"] == 1) & (fd["run_alert"] == 1))),\n                    "n_features": int(fd["n_features"].iloc[0]),\n                }\n            )\n\n    pred_df = pd.DataFrame(preds).sort_values(["config", "workload", "stressor"])\n    fold_df = pd.DataFrame(fold_rows).sort_values(["config", "holdout_workload"])\n    diag_df = pd.DataFrame([{k: v for k, v in row.items() if not k.startswith("_")} for row in diagnostic_records]).sort_values(\n        ["config", "workload", "stressor"]\n    )\n\n    # Workload-conditioned score head: distance to workload nominal template.\n    pred_df["nominal_template_score"] = np.nan\n    pred_df["run_score_wc"] = pred_df["run_score"]\n    for cfg, d in pred_df.groupby("config"):\n        base = d[d["stressor"] == "NOMINAL"].set_index("workload")["run_score"].to_dict()\n        idx = d.index\n        pred_df.loc[idx, "nominal_template_score"] = d["workload"].map(base).to_numpy(dtype=float)\n        pred_df.loc[idx, "run_score_wc"] = np.abs(\n            pred_df.loc[idx, "run_score"].to_numpy(dtype=float)\n            - pred_df.loc[idx, "nominal_template_score"].to_numpy(dtype=float)\n        )\n\n    overall_rows = []\n    for cfg, d in pred_df.groupby("config"):\n        y = d["label"].to_numpy(dtype=int)\n        s_run = d["run_score"].to_numpy(dtype=float)\n        s_wc = d["run_score_wc"].to_numpy(dtype=float)\n        overall_rows.append(\n            {\n                "config": cfg,\n                "n_cases": int(len(d)),\n                "n_features": int(d["n_features"].iloc[0]),\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "fpr_run_alert": float(np.mean(d[d["label"] == 0]["run_alert"])),\n                "tpr_run_alert": float(np.mean(d[d["label"] == 1]["run_alert"])),\n                "median_nominal_score": float(np.median(d[d["label"] == 0]["run_score"])),\n                "median_anomaly_score": float(np.median(d[d["label"] == 1]["run_score"])),\n                "median_nominal_score_wc": float(np.median(d[d["label"] == 0]["run_score_wc"])),\n                "median_anomaly_score_wc": float(np.median(d[d["label"] == 1]["run_score_wc"])),\n            }\n        )\n    overall_df = pd.DataFrame(overall_rows).sort_values("config")\n\n    final_cfg = "tier0_tier1_tier2"\n    fin = pred_df[pred_df["config"] == final_cfg]\n    stress_rows = []\n    neg = fin[fin["stressor"] == "NOMINAL"][["workload", "run_score", "run_score_wc"]].set_index("workload")\n    for a in ANOMALIES:\n        pos = fin[fin["stressor"] == a][["workload", "run_score", "run_score_wc"]].set_index("workload")\n        m = neg.join(pos, lsuffix="_neg", rsuffix="_pos", how="inner")\n        y = np.array([0] * len(m) + [1] * len(m), dtype=int)\n        s_run = np.concatenate([m["run_score_neg"].to_numpy(dtype=float), m["run_score_pos"].to_numpy(dtype=float)])\n        s_wc = np.concatenate([m["run_score_wc_neg"].to_numpy(dtype=float), m["run_score_wc_pos"].to_numpy(dtype=float)])\n        stress_rows.append(\n            {\n                "stressor": a,\n                "roc_auc": safe_auc(y, s_run),\n                "pr_auc": safe_ap(y, s_run),\n                "roc_auc_wc": safe_auc(y, s_wc),\n                "pr_auc_wc": safe_ap(y, s_wc),\n                "median_neg_score": float(np.median(m["run_score_neg"])),\n                "median_pos_score": float(np.median(m["run_score_pos"])),\n                "median_neg_score_wc": float(np.median(m["run_score_wc_neg"])),\n                "median_pos_score_wc": float(np.median(m["run_score_wc_pos"])),\n                "pos_neg_ratio": float((np.median(m["run_score_pos"]) + 1e-6) / (np.median(m["run_score_neg"]) + 1e-6)),\n                "pos_neg_diff": float(np.median(m["run_score_pos"]) - np.median(m["run_score_neg"])),\n                "pos_neg_ratio_wc": float((np.median(m["run_score_wc_pos"]) + 1e-6) / (np.median(m["run_score_wc_neg"]) + 1e-6)),\n                "pos_neg_diff_wc": float(np.median(m["run_score_wc_pos"]) - np.median(m["run_score_wc_neg"])),\n            }\n        )\n    stress_df = pd.DataFrame(stress_rows).sort_values("stressor")\n\n    mm_pr = float(np.mean(stress_df["pr_auc"]))\n    mm_roc = float(np.mean(stress_df["roc_auc"]))\n    mm_pr_wc = float(np.mean(stress_df["pr_auc_wc"]))\n    mm_roc_wc = float(np.mean(stress_df["roc_auc_wc"]))\n\n    filt = stress_df[~stress_df["stressor"].isin(["BRANCH", "TLB"])]\n    mm_pr_filt = float(np.mean(filt["pr_auc"]))\n    mm_roc_filt = float(np.mean(filt["roc_auc"]))\n    mm_pr_filt_wc = float(np.mean(filt["pr_auc_wc"]))\n    mm_roc_filt_wc = float(np.mean(filt["roc_auc_wc"]))\n\n    diag_pred_feature_df, diag_metrics_feature_df, diag_cm_feature = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_feature_contrib",\n    )\n    diag_pred_df, diag_metrics_df, diag_cm = build_stressor_attribution(\n        diagnostic_records,\n        vector_key="_mechanism_vector",\n    )\n    diag_tier_df = build_stressor_tier_contributions(diag_df, config=final_cfg)\n    mechanism_df = build_mechanism_summary(diag_df, config=final_cfg)\n    sequential_df = build_sequential_metrics(pred_df)\n    holdout_df = build_holdout_robustness_summary(fold_df)\n\n    pred_df.to_csv(out_dir / "case_predictions.csv", index=False)\n    fold_df.to_csv(out_dir / "fold_metrics.csv", index=False)\n    overall_df.to_csv(out_dir / "overall_metrics.csv", index=False)\n    stress_df.to_csv(out_dir / "stressor_metrics_final_config.csv", index=False)\n    diag_df.to_csv(out_dir / "case_diagnosis_summary.csv", index=False)\n    diag_pred_df.to_csv(out_dir / "stressor_diagnosis_predictions.csv", index=False)\n    diag_metrics_df.to_csv(out_dir / "stressor_diagnosis_metrics.csv", index=False)\n    diag_cm.to_csv(out_dir / "stressor_confusion_matrix.csv")\n    diag_tier_df.to_csv(out_dir / "stressor_tier_contributions.csv", index=False)\n    mechanism_df.to_csv(out_dir / "mechanism_group_summary.csv", index=False)\n    sequential_df.to_csv(out_dir / "sequential_metrics.csv", index=False)\n    holdout_df.to_csv(out_dir / "holdout_robustness_summary.csv", index=False)\n    diag_pred_feature_df.to_csv(out_dir / "stressor_feature_diagnosis_predictions.csv", index=False)\n    diag_metrics_feature_df.to_csv(out_dir / "stressor_feature_diagnosis_metrics.csv", index=False)\n    diag_cm_feature.to_csv(out_dir / "stressor_feature_confusion_matrix.csv")\n\n    overall_tex = overall_df[\n        ["config", "n_features", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "fpr_run_alert", "tpr_run_alert"]\n    ].rename(\n        columns={\n            "config": "Configuration",\n            "n_features": "Features",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "fpr_run_alert": "Run-FPR",\n            "tpr_run_alert": "Run-TPR",\n        }\n    )\n    stress_tex = stress_df[\n        ["stressor", "roc_auc", "pr_auc", "roc_auc_wc", "pr_auc_wc", "pos_neg_ratio", "pos_neg_diff"]\n    ].rename(\n        columns={\n            "stressor": "Stressor",\n            "roc_auc": "ROC-AUC (Base)",\n            "pr_auc": "AUC-PR (Base)",\n            "roc_auc_wc": "ROC-AUC (WC)",\n            "pr_auc_wc": "AUC-PR (WC)",\n            "pos_neg_ratio": "Pos/Neg Score Ratio",\n            "pos_neg_diff": "Pos-Neg Score Delta",\n        }\n    )\n    (out_dir / "overall_metrics.tex").write_text(\n        to_latex_table(\n            overall_tex,\n            "DICE micro-twin + split-conformal run-level results under benign retraining.",\n            "tab:dice_full_overall",\n        )\n    )\n    (out_dir / "stressor_metrics_final_config.tex").write_text(\n        to_latex_table(\n            stress_tex,\n            "Final DICE configuration per-stressor separability.",\n            "tab:dice_full_stressor",\n        )\n    )\n    if not diag_metrics_df.empty:\n        diag_tex = diag_metrics_df.rename(\n            columns={\n                "config": "Configuration",\n                "n_cases": "Cases",\n                "top1_acc": "Top-1 Acc.",\n                "top2_acc": "Top-2 Acc.",\n                "macro_f1": "Macro-F1",\n                "mean_margin_to_second": "Mean Margin",\n            }\n        )\n        (out_dir / "stressor_diagnosis_metrics.tex").write_text(\n            to_latex_table(\n                diag_tex,\n                "Mechanism-group stressor attribution from DICE residual contributions across workloads.",\n                "tab:dice_stressor_diagnosis",\n            )\n        )\n    if not sequential_df.empty:\n        seq_tex = sequential_df.rename(\n            columns={\n                "config": "Configuration",\n                "benign_run_alert_rate": "Benign Run-Alert Rate",\n                "benign_persist_alerts_per_hour": "Benign Persist Alerts/hr",\n                "anomaly_detect_rate": "Anomaly Detect Rate",\n                "median_time_to_detect_s": "Median TTD (s)",\n                "detect_within_300s": "Detect <=300s",\n            }\n        )[\n            [\n                "Configuration",\n                "Benign Run-Alert Rate",\n                "Benign Persist Alerts/hr",\n                "Anomaly Detect Rate",\n                "Median TTD (s)",\n                "Detect <=300s",\n            ]\n        ]\n        (out_dir / "sequential_metrics.tex").write_text(\n            to_latex_table(\n                seq_tex,\n                "Sequential decision metrics for the DICE run-level detector.",\n                "tab:dice_sequential_metrics",\n            )\n        )\n\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config.png", score_col="run_score", title_tag="base")\n    plot_curves(pred_df, fig_dir / "fig_roc_pr_by_config_wc.png", score_col="run_score_wc", title_tag="workload-conditioned")\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot.png",\n        score_col="run_score",\n        y_label="Run score (base)",\n        title_tag="base",\n    )\n    plot_score_box(\n        pred_df,\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        score_col="run_score_wc",\n        y_label="Run score (workload-conditioned)",\n        title_tag="workload-conditioned",\n    )\n    plot_confusion_heatmap(\n        diag_cm,\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        title="Final-config prototype stressor attribution",\n    )\n    plot_stressor_tier_shares(\n        diag_tier_df,\n        fig_dir / "fig_stressor_tier_contributions.png",\n    )\n    plot_mechanism_shares(\n        mechanism_df,\n        fig_dir / "fig_mechanism_group_summary.png",\n    )\n    plot_detection_latency(\n        sequential_df,\n        fig_dir / "fig_detection_latency.png",\n    )\n\n    md = []\n    md.append("# DICE Full Retrain Results")\n    md.append("")\n    md.append("## Setup")\n    md.append(\n        f"- Protocol: {args.protocol}, benign-only fit/calibration, block_B={args.block_B}, "\n        f"alpha={args.alpha}, persist_k={args.persist_k}, gain={args.gain}"\n    )\n    md.append("")\n    md.append("## Overall")\n    append_text_table(md, overall_df)\n    md.append("")\n    md.append("## Final Config Stressors")\n    append_text_table(md, stress_df)\n    md.append("")\n    md.append("## Paper-style Aggregates (Final Config)")\n    md.append(f"- Base score mean stressor AUC-PR (all five): **{mm_pr:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (all five): **{mm_roc:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (all five): **{mm_pr_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (all five): **{mm_roc_wc:.4f}**")\n    md.append(f"- Base score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt:.4f}**")\n    md.append(f"- Base score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt:.4f}**")\n    md.append(f"- WC score mean stressor AUC-PR (excluding BRANCH/TLB): **{mm_pr_filt_wc:.4f}**")\n    md.append(f"- WC score mean stressor ROC-AUC (excluding BRANCH/TLB): **{mm_roc_filt_wc:.4f}**")\n    md.append("")\n    if not diag_metrics_df.empty:\n        md.append("## Diagnosis")\n        md.append("- Primary diagnosis uses mechanism-group centroids over workload-held residual summaries.")\n        append_text_table(md, diag_metrics_df)\n        md.append("")\n        if not diag_tier_df.empty:\n            md.append("## Final Config Tier Contribution Summary")\n            append_text_table(md, diag_tier_df)\n            md.append("")\n        if not mechanism_df.empty:\n            md.append("## Final Config Mechanism Summary")\n            append_text_table(md, mechanism_df)\n            md.append("")\n    if not sequential_df.empty:\n        md.append("## Sequential Decisioning")\n        append_text_table(md, sequential_df)\n        md.append("")\n    if not holdout_df.empty:\n        md.append("## Holdout Robustness (Workload Drift Proxy)")\n        append_text_table(md, holdout_df)\n        md.append("")\n    md.append("## Files")\n    for p in [\n        out_dir / "overall_metrics.csv",\n        out_dir / "stressor_metrics_final_config.csv",\n        out_dir / "sequential_metrics.csv",\n        out_dir / "case_diagnosis_summary.csv",\n        out_dir / "stressor_diagnosis_metrics.csv",\n        out_dir / "mechanism_group_summary.csv",\n        out_dir / "stressor_confusion_matrix.csv",\n        out_dir / "stressor_tier_contributions.csv",\n        out_dir / "overall_metrics.tex",\n        out_dir / "stressor_metrics_final_config.tex",\n        out_dir / "stressor_diagnosis_metrics.tex",\n        out_dir / "sequential_metrics.tex",\n        fig_dir / "fig_roc_pr_by_config.png",\n        fig_dir / "fig_roc_pr_by_config_wc.png",\n        fig_dir / "fig_run_score_boxplot.png",\n        fig_dir / "fig_run_score_boxplot_wc.png",\n        fig_dir / "fig_stressor_confusion_matrix.png",\n        fig_dir / "fig_stressor_tier_contributions.png",\n        fig_dir / "fig_mechanism_group_summary.png",\n        fig_dir / "fig_detection_latency.png",\n    ]:\n        md.append(f"- `{p}`")\n    (out_dir / "RESULTS_SUMMARY.md").write_text("\\n".join(md) + "\\n")\n\n    print(f"[OK] wrote results to: {out_dir}")\n    print("[OK] overall metrics:")\n    print(overall_df.to_string(index=False))\n    print("[OK] final config stressor metrics:")\n    print(stress_df.to_string(index=False))\n    if not diag_metrics_df.empty:\n        print("[OK] stressor diagnosis metrics:")\n        print(diag_metrics_df.to_string(index=False))\n    if not sequential_df.empty:\n        print("[OK] sequential metrics:")\n        print(sequential_df.to_string(index=False))\n    print(\n        "[OK] aggregates (base): "\n        f"all(AUC-PR={mm_pr:.4f}, ROC-AUC={mm_roc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt:.4f}, ROC-AUC={mm_roc_filt:.4f})"\n    )\n    print(\n        "[OK] aggregates (workload-conditioned): "\n        f"all(AUC-PR={mm_pr_wc:.4f}, ROC-AUC={mm_roc_wc:.4f}), "\n        f"filtered(AUC-PR={mm_pr_filt_wc:.4f}, ROC-AUC={mm_roc_filt_wc:.4f})"\n    )\n\n\nif __name__ == "__main__":\n    main()\n'


def _load_notebook_module(name: str, source: str) -> dict[str, object]:
    fake_file = REPO_ROOT / '__notebook__' / f'{name}.py'
    module = types.ModuleType(name)
    module.__file__ = str(fake_file)
    sys.modules[name] = module
    exec(source, module.__dict__)
    return module.__dict__


ANALYSIS_MODULE = _load_notebook_module('dice_generate_results_analysis_inline', NOTEBOOK_GENERATE_RESULTS_ANALYSIS_SOURCE)
FULL_MODULE = _load_notebook_module('dice_train_eval_dice_pipeline_inline', NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE)

STAGE1_GAINS = [0.15, 0.25, 0.35, 0.50]
STAGE1_BLOCKS = [30, 60, 90, 120]
STAGE2_ALPHAS = [0.01, 0.02, 0.05, 0.10]
STAGE2_PERSISTS = [1, 2, 3, 5]


def deterministic_env() -> dict[str, str]:
    env = portable_env()
    os.environ.update(env)
    return env


def _run_module_main(module_ns: dict[str, object], argv: list[str]) -> None:
    argv_backup = sys.argv[:]
    try:
        sys.argv = argv
        module_ns['main']()
    finally:
        sys.argv = argv_backup


def ensure_dataset_root(root: Path) -> None:
    needed = [
        root / 'tier0',
        root / 'tier1_alt',
        root / 'tier2',
        root / 'no_nan_report.json',
    ]
    missing = [str(p) for p in needed if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Dataset root is missing required files/folders: {missing}')


def run_analysis_notebook(root: Path, source_hz: int = 5, out_dir: Path | None = None) -> Path:
    resolved_out = out_dir or (root / 'results_analysis')
    _run_module_main(
        ANALYSIS_MODULE,
        [
            'generate_results_analysis.py',
            '--root',
            str(root),
            '--source_hz',
            str(source_hz),
            '--out_dir',
            str(resolved_out),
        ],
    )
    return resolved_out


def default_full_out_dir(root: Path, protocol: str, feature_profile: str = 'mixed') -> Path:
    if feature_profile == 'mixed':
        return root / ('results_dice_full_holdout' if protocol == 'workload_holdout' else 'results_dice_full')
    suffix = f'results_dice_full_{feature_profile}'
    if protocol == 'workload_holdout':
        suffix = f'{suffix}_holdout'
    return root / suffix


def run_full_notebook(
    root: Path,
    protocol: str,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    persist_k: int = 3,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    feature_profile: str = 'mixed',
    out_dir: Path | None = None,
) -> Path:
    resolved_out = out_dir or default_full_out_dir(root, protocol, feature_profile)
    t0 = perf_counter()
    _run_module_main(
        FULL_MODULE,
        [
            'train_eval_dice_pipeline.py',
            '--root',
            str(root),
            '--protocol',
            protocol,
            '--source_hz',
            str(source_hz),
            '--fit_ratio',
            str(fit_ratio),
            '--block_B',
            str(block_B),
            '--alpha',
            str(alpha),
            '--persist_k',
            str(persist_k),
            '--gain',
            str(gain),
            '--ridge_lambda',
            str(ridge_lambda),
            '--feature_profile',
            feature_profile,
            '--out_dir',
            str(resolved_out),
        ],
    )
    elapsed_s = perf_counter() - t0
    (resolved_out / 'notebook_invocation.json').write_text(
        json.dumps(
            {
                'feature_profile': feature_profile,
                'protocol': protocol,
                'elapsed_s': elapsed_s,
                'source_hz': source_hz,
                'fit_ratio': fit_ratio,
                'block_B': block_B,
                'alpha': alpha,
                'persist_k': persist_k,
                'gain': gain,
                'ridge_lambda': ridge_lambda,
            },
            indent=2,
        )
        + '\n'
    )
    print(f'Completed {feature_profile} / {protocol} run in {elapsed_s:.2f} s -> {resolved_out}')
    return resolved_out


def compare_feature_profiles(
    mixed_global: Path,
    full_global: Path,
    mixed_holdout: Path,
    full_holdout: Path,
    final_config: str = 'tier0_tier1_tier2',
) -> pd.DataFrame:
    def _load_one(profile: str, global_dir: Path, holdout_dir: Path) -> dict[str, float | str]:
        overall = pd.read_csv(global_dir / 'overall_metrics.csv')
        sequential = pd.read_csv(global_dir / 'sequential_metrics.csv')
        diagnosis = pd.read_csv(global_dir / 'stressor_diagnosis_metrics.csv')
        holdout = pd.read_csv(holdout_dir / 'holdout_robustness_summary.csv')
        invocation_path = global_dir / 'notebook_invocation.json'
        elapsed_s = float('nan')
        if invocation_path.exists():
            elapsed_s = float(json.loads(invocation_path.read_text()).get('elapsed_s', float('nan')))

        overall_row = overall.loc[overall['config'] == final_config].iloc[0]
        sequential_row = sequential.loc[sequential['config'] == final_config].iloc[0]
        diagnosis_row = diagnosis.loc[diagnosis['config'] == final_config].iloc[0]
        holdout_row = holdout.loc[holdout['config'] == final_config].iloc[0]

        return {
            'feature_profile': profile,
            'base_roc_auc': float(overall_row['roc_auc']),
            'base_pr_auc': float(overall_row['pr_auc']),
            'run_fpr': float(overall_row['fpr_run_alert']),
            'run_tpr': float(overall_row['tpr_run_alert']),
            'holdout_mean_pr_auc': float(holdout_row['mean_pr_auc']),
            'holdout_worst_pr_auc': float(holdout_row['worst_pr_auc']),
            'holdout_mean_roc_auc': float(holdout_row['mean_roc_auc']),
            'anomaly_detect_rate': float(sequential_row['anomaly_detect_rate']),
            'benign_run_alert_rate': float(sequential_row['benign_run_alert_rate']),
            'median_time_to_detect_s': float(sequential_row['median_time_to_detect_s']),
            'diag_top1_acc': float(diagnosis_row['top1_acc']),
            'diag_top2_acc': float(diagnosis_row['top2_acc']),
            'runtime_s': elapsed_s,
        }

    rows = [
        _load_one('mixed', mixed_global, mixed_holdout),
        _load_one('full', full_global, full_holdout),
    ]
    return pd.DataFrame(rows)


def run_tuning_notebook(
    root: Path,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    ridge_lambda: float = 1e-3,
) -> Path:
    out_tune = root / 'results_dice_tuning'
    out_runs = out_tune / 'runs'
    out_tune.mkdir(parents=True, exist_ok=True)
    out_runs.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict[str, object]] = []
    alpha_fixed = 0.05
    persist_fixed = 3

    for gain in STAGE1_GAINS:
        for block_B in STAGE1_BLOCKS:
            tag = f'g{gain}_B{block_B}_a{alpha_fixed}_k{persist_fixed}'
            out_dir = out_runs / tag
            try:
                run_full_notebook(
                    root,
                    protocol='global',
                    source_hz=source_hz,
                    fit_ratio=fit_ratio,
                    block_B=block_B,
                    alpha=alpha_fixed,
                    persist_k=persist_fixed,
                    gain=gain,
                    ridge_lambda=ridge_lambda,
                    out_dir=out_dir,
                )
                overall = pd.read_csv(out_dir / 'overall_metrics.csv')
                row = overall[overall['config'] == 'tier0_tier1_tier2'].iloc[0]
                summary_rows.append({
                    'stage': 'gain_block',
                    'status': 'ok',
                    'gain': gain,
                    'block_B': block_B,
                    'alpha': alpha_fixed,
                    'persist_k': persist_fixed,
                    'pr_auc_wc': float(row['pr_auc_wc']),
                    'roc_auc_wc': float(row['roc_auc_wc']),
                    'out_dir': str(out_dir),
                })
            except Exception as exc:
                summary_rows.append({
                    'stage': 'gain_block',
                    'status': 'error',
                    'gain': gain,
                    'block_B': block_B,
                    'alpha': alpha_fixed,
                    'persist_k': persist_fixed,
                    'error': str(exc),
                    'out_dir': str(out_dir),
                })

    stage1 = pd.DataFrame([r for r in summary_rows if r.get('stage') == 'gain_block' and r.get('status') == 'ok'])
    if stage1.empty:
        pd.DataFrame(summary_rows).to_csv(out_tune / 'sweep_summary.csv', index=False)
        raise RuntimeError('Notebook-local tuning stage 1 produced no successful runs.')

    best = stage1.sort_values(['pr_auc_wc', 'roc_auc_wc'], ascending=False).iloc[0]
    best_gain = float(best['gain'])
    best_block = int(best['block_B'])

    for alpha in STAGE2_ALPHAS:
        for persist_k in STAGE2_PERSISTS:
            tag = f'g{best_gain}_B{best_block}_a{alpha}_k{persist_k}'
            out_dir = out_runs / tag
            try:
                run_full_notebook(
                    root,
                    protocol='global',
                    source_hz=source_hz,
                    fit_ratio=fit_ratio,
                    block_B=best_block,
                    alpha=alpha,
                    persist_k=persist_k,
                    gain=best_gain,
                    ridge_lambda=ridge_lambda,
                    out_dir=out_dir,
                )
                overall = pd.read_csv(out_dir / 'overall_metrics.csv')
                row = overall[overall['config'] == 'tier0_tier1_tier2'].iloc[0]
                summary_rows.append({
                    'stage': 'alpha_persist',
                    'status': 'ok',
                    'gain': best_gain,
                    'block_B': best_block,
                    'alpha': alpha,
                    'persist_k': persist_k,
                    'pr_auc_wc': float(row['pr_auc_wc']),
                    'roc_auc_wc': float(row['roc_auc_wc']),
                    'out_dir': str(out_dir),
                })
            except Exception as exc:
                summary_rows.append({
                    'stage': 'alpha_persist',
                    'status': 'error',
                    'gain': best_gain,
                    'block_B': best_block,
                    'alpha': alpha,
                    'persist_k': persist_k,
                    'error': str(exc),
                    'out_dir': str(out_dir),
                })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(out_tune / 'sweep_summary.csv', index=False)
    summary[(summary['stage'] == 'gain_block') & (summary['status'] == 'ok')].to_csv(out_tune / 'sweep_stage1_gain_block.csv', index=False)
    summary[(summary['stage'] == 'alpha_persist') & (summary['status'] == 'ok')].to_csv(out_tune / 'sweep_stage2_alpha_persist.csv', index=False)

    stage2 = summary[(summary['stage'] == 'alpha_persist') & (summary['status'] == 'ok')].copy()
    stage2 = stage2.sort_values(['pr_auc_wc', 'roc_auc_wc'], ascending=False)
    recommended = stage2.iloc[0] if not stage2.empty else best
    recommendation = pd.DataFrame([
        {
            'gain': float(recommended['gain']),
            'block_B': int(recommended['block_B']),
            'alpha': float(recommended['alpha']),
            'persist_k': int(recommended['persist_k']),
            'pr_auc_wc': float(recommended['pr_auc_wc']),
            'roc_auc_wc': float(recommended['roc_auc_wc']),
        }
    ])
    recommendation.to_csv(out_tune / 'recommended_config.csv', index=False)
    return out_tune


def dataset_tree_sha256(root: Path) -> dict[str, object]:
    hasher = hashlib.sha256()
    count = 0
    for path in sorted(p for p in root.rglob('*') if p.is_file()):
        rel = path.relative_to(root).as_posix()
        if rel.split('/', 1)[0].startswith('results_'):
            continue
        hasher.update(rel.encode('utf-8'))
        with path.open('rb') as handle:
            while True:
                chunk = handle.read(1024 * 1024)
                if not chunk:
                    break
                hasher.update(chunk)
        count += 1
    return {'file_count': count, 'sha256': hasher.hexdigest()}


def file_sha256(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def package_versions() -> dict[str, str]:
    packages = [
        'matplotlib',
        'numpy',
        'pandas',
        'psutil',
        'scikit-learn',
        'scipy',
        'joblib',
        'threadpoolctl',
        'python-dateutil',
        'pytz',
        'tzdata',
    ]
    return {pkg.replace('-', '_'): importlib.metadata.version(pkg) for pkg in packages}


def write_run_manifest(
    root: Path,
    analysis_out: Path,
    full_out: Path,
    holdout_out: Path | None,
    tuning_out: Path | None,
) -> Path:
    manifest_dir = root / 'results_portable'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / 'run_manifest.json'

    env_yml = REPO_ROOT / 'environment.yml'
    req_txt = REPO_ROOT / 'requirements.txt'

    manifest = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'platform': platform.platform(),
        'python_version': sys.version.split()[0],
        'repo_root': str(REPO_ROOT),
        'dataset_root': str(root),
        'dataset_digest': dataset_tree_sha256(root),
        'environment_files': {
            'environment_yml': {'path': str(env_yml), 'sha256': file_sha256(env_yml)},
            'requirements_txt': {'path': str(req_txt), 'sha256': file_sha256(req_txt)},
        },
        'package_versions': package_versions(),
        'stages': {
            'analysis': str(analysis_out),
            'full': str(full_out),
            'holdout': str(holdout_out) if holdout_out else None,
            'tuning': str(tuning_out) if tuning_out else None,
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return manifest_path


def run_notebook_pipeline(
    dataset_root: Path,
    source_hz: int = 5,
    fit_ratio: float = 0.6,
    block_B: int = 60,
    alpha: float = 0.05,
    persist_k: int = 3,
    gain: float = 0.35,
    ridge_lambda: float = 1e-3,
    run_holdout: bool = True,
    include_tuning: bool = False,
) -> dict[str, object]:
    deterministic_env()
    root = Path(dataset_root).expanduser().resolve()
    ensure_dataset_root(root)

    analysis_out = run_analysis_notebook(root, source_hz=source_hz)
    full_out = run_full_notebook(
        root,
        protocol='global',
        source_hz=source_hz,
        fit_ratio=fit_ratio,
        block_B=block_B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
        ridge_lambda=ridge_lambda,
    )
    holdout_out = None
    if run_holdout:
        holdout_out = run_full_notebook(
            root,
            protocol='workload_holdout',
            source_hz=source_hz,
            fit_ratio=fit_ratio,
            block_B=block_B,
            alpha=alpha,
            persist_k=persist_k,
            gain=gain,
            ridge_lambda=ridge_lambda,
        )
    tuning_out = run_tuning_notebook(root, source_hz=source_hz, fit_ratio=fit_ratio, ridge_lambda=ridge_lambda) if include_tuning else None
    manifest_path = write_run_manifest(root, analysis_out, full_out, holdout_out, tuning_out)

    return {
        'analysis_out': str(analysis_out),
        'full_out': str(full_out),
        'holdout_out': str(holdout_out) if holdout_out else '',
        'tuning_out': str(tuning_out) if tuning_out else '',
        'manifest_path': str(manifest_path),
        'run_holdout': run_holdout,
        'include_tuning': include_tuning,
    }


## Enable Block-Trace Export for the Overlay

This small notebook-local patch extends the embedded DICE engine so it also writes `case_block_traces.csv`.
That file is used later for the true time-series virtual-system overlay.


In [ ]:
import re

def _replace_once(src: str, old: str, new: str) -> str:
    if old not in src:
        raise ValueError(f"Patch target not found: {old[:120]}")
    return src.replace(old, new, 1)

# Use the export line as the idempotence check.
if 'trace_df.to_csv(out_dir / "case_block_traces.csv", index=False)' not in NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE:
    patched = NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE

    old_eval = """def evaluate_run(
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> Tuple[Dict[str, float], np.ndarray]:
    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)
    r = residual_timeseries(Xn, bundle.A, gain=gain)
    sig = block_signatures(r, B=B)
    sc = signature_scores(sig, bundle.weights)
    pv = conformal_pvals(bundle.cal_scores, sc)

    block_alert = pv < alpha
    persist = persistent_alerts(block_alert, k=persist_k)

    run_alert = int(np.any(persist > 0))
    peak_score = float(np.max(sc)) if len(sc) else 0.0
    run_score = peak_score
    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)
    feature_contrib = run_signature * bundle.weights
    first_block_idx = first_positive_index(block_alert)
    first_persist_idx = first_positive_index(persist)
    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")
    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")
    n_blocks = int(len(sc))
    duration_s = max(n_blocks, 1)

    return (
        {
            "run_score": run_score,
            "run_alert": run_alert,
            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,
            "peak_block_score": peak_score,
            "n_blocks": n_blocks,
            "n_block_alerts": int(np.sum(block_alert)),
            "n_persist_alerts": int(np.sum(persist)),
            "first_block_alert_idx": first_block_idx,
            "first_persist_alert_idx": first_persist_idx,
            "first_block_alert_s": block_time_s,
            "time_to_detect_s": persist_time_s,
            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),
            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),
        },
        feature_contrib,
    )
"""

    new_eval = """def evaluate_run(
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> Tuple[Dict[str, float], np.ndarray, List[Dict[str, float]]]:
    Xn = (X_run - bundle.median) / (bundle.scale + 1e-12)
    r = residual_timeseries(Xn, bundle.A, gain=gain)
    sig = block_signatures(r, B=B)
    sc = signature_scores(sig, bundle.weights)
    pv = conformal_pvals(bundle.cal_scores, sc)

    block_alert = pv < alpha
    persist = persistent_alerts(block_alert, k=persist_k)

    run_alert = int(np.any(persist > 0))
    peak_score = float(np.max(sc)) if len(sc) else 0.0
    run_score = peak_score
    run_signature = np.nanmedian(sig, axis=0) if len(sig) else np.zeros_like(bundle.weights)
    feature_contrib = run_signature * bundle.weights
    first_block_idx = first_positive_index(block_alert)
    first_persist_idx = first_positive_index(persist)
    block_time_s = float(B + first_block_idx) if first_block_idx >= 0 else float("nan")
    persist_time_s = float(B + first_persist_idx) if first_persist_idx >= 0 else float("nan")
    n_blocks = int(len(sc))
    duration_s = max(n_blocks, 1)

    block_trace = []
    for i in range(len(sc)):
        block_trace.append(
            {
                "block_idx": int(i),
                "block_start_s": float(i),
                "block_end_s": float(B + i),
                "score": float(sc[i]),
                "pvalue": float(pv[i]),
                "block_alert": int(block_alert[i]),
                "persist_alert": int(persist[i]),
                "threshold": float(bundle.tau),
            }
        )

    return (
        {
            "run_score": run_score,
            "run_alert": run_alert,
            "min_pvalue": float(np.min(pv)) if len(pv) else 1.0,
            "peak_block_score": peak_score,
            "n_blocks": n_blocks,
            "n_block_alerts": int(np.sum(block_alert)),
            "n_persist_alerts": int(np.sum(persist)),
            "first_block_alert_idx": first_block_idx,
            "first_persist_alert_idx": first_persist_idx,
            "first_block_alert_s": block_time_s,
            "time_to_detect_s": persist_time_s,
            "block_alerts_per_hour": float(np.sum(block_alert) * 3600.0 / duration_s),
            "persist_alerts_per_hour": float(np.sum(persist) * 3600.0 / duration_s),
        },
        feature_contrib,
        block_trace,
    )
"""
    patched = _replace_once(patched, old_eval, new_eval)

    old_append = """def append_case_outputs(
    preds: List[Dict[str, object]],
    diagnostic_records: List[Dict[str, object]],
    config: str,
    holdout_workload: str,
    case: CaseRef,
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> None:
    metrics, feature_contrib = evaluate_run(
        X_run,
        bundle,
        B=B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
    )
    preds.append(
        {
            "config": config,
            "holdout_workload": holdout_workload,
            "case_id": case.case_id,
            "workload": case.workload,
            "stressor": case.stressor,
            "label": case.label,
            **metrics,
            "n_features": len(bundle.feature_names),
            "tau": bundle.tau,
        }
    )
    diagnostic_records.append(
        build_diagnostic_record(
            config=config,
            holdout_workload=holdout_workload,
            case=case,
            feature_names=bundle.feature_names,
            feature_contrib=feature_contrib,
        )
    )
"""

    new_append = """def append_case_outputs(
    preds: List[Dict[str, object]],
    diagnostic_records: List[Dict[str, object]],
    trace_records: List[Dict[str, object]],
    config: str,
    holdout_workload: str,
    case: CaseRef,
    X_run: np.ndarray,
    bundle: ModelBundle,
    B: int,
    alpha: float,
    persist_k: int,
    gain: float,
) -> None:
    metrics, feature_contrib, block_trace = evaluate_run(
        X_run,
        bundle,
        B=B,
        alpha=alpha,
        persist_k=persist_k,
        gain=gain,
    )
    preds.append(
        {
            "config": config,
            "holdout_workload": holdout_workload,
            "case_id": case.case_id,
            "workload": case.workload,
            "stressor": case.stressor,
            "label": case.label,
            **metrics,
            "n_features": len(bundle.feature_names),
            "tau": bundle.tau,
        }
    )
    trace_records.extend(
        [
            {
                "config": config,
                "holdout_workload": holdout_workload,
                "case_id": case.case_id,
                "workload": case.workload,
                "stressor": case.stressor,
                "label": case.label,
                "n_features": len(bundle.feature_names),
                **row,
            }
            for row in block_trace
        ]
    )
    diagnostic_records.append(
        build_diagnostic_record(
            config=config,
            holdout_workload=holdout_workload,
            case=case,
            feature_names=bundle.feature_names,
            feature_contrib=feature_contrib,
        )
    )
"""
    patched = _replace_once(patched, old_append, new_append)

    patched, init_count = re.subn(
        r'(    preds = \[\]\n    fold_rows = \[\]\n    diagnostic_records: List\[Dict\[str, object\]\] = \[\]\n)',
        lambda m: m.group(1) + '    trace_records: List[Dict[str, object]] = []\n',
        patched,
        count=1,
    )
    if init_count == 0:
        raise ValueError("Patch target not found for trace_records initialization")

    call_pattern = re.compile(
        r'(append_case_outputs\(\n(?P<indent>\s*)preds=preds,\n(?P=indent)diagnostic_records=diagnostic_records,\n)(?!(?P=indent)trace_records=trace_records,\n)'
    )
    patched, call_count = call_pattern.subn(
        lambda m: m.group(1) + f"{m.group('indent')}trace_records=trace_records,\n",
        patched,
    )
    if call_count == 0:
        raise ValueError("Patch target not found for append_case_outputs call sites")

    diag_pattern = re.compile(
        r'(    pred_df = pd.DataFrame\(preds\)\.sort_values\(\["config", "workload", "stressor"\]\)\n'
        r'    fold_df = pd.DataFrame\(fold_rows\)\.sort_values\(\["config", "holdout_workload"\]\)\n'
        r'    diag_df = pd.DataFrame\(\[\{k: v for k, v in row.items\(\) if not k.startswith\("_"\)\} for row in diagnostic_records\]\)\.sort_values\(\n'
        r'        \["config", "workload", "stressor"\]\n'
        r'    \)\n)'
    )
    patched, diag_count = diag_pattern.subn(
        lambda m: (
            m.group(1)
            + '    trace_df = pd.DataFrame(trace_records)\n'
            + '    if not trace_df.empty:\n'
            + '        trace_df = trace_df.sort_values(["config", "case_id", "block_idx"]).reset_index(drop=True)\n'
        ),
        patched,
        count=1,
    )
    if diag_count == 0:
        raise ValueError("Patch target not found for trace_df construction")

    save_pattern = re.compile(
        r'(    pred_df\.to_csv\(out_dir / "case_predictions\.csv", index=False\)\n)'
        r'(?!    trace_df\.to_csv\(out_dir / "case_block_traces\.csv", index=False\)\n)'
        r'(    fold_df\.to_csv\(out_dir / "fold_metrics\.csv", index=False\)\n)'
    )
    patched, save_count = save_pattern.subn(
        lambda m: (
            m.group(1)
            + '    trace_df.to_csv(out_dir / "case_block_traces.csv", index=False)\n'
            + m.group(2)
        ),
        patched,
        count=1,
    )
    if save_count == 0:
        raise ValueError("Patch target not found for case_block_traces export")

    files_pattern = re.compile(
        r'(    for p in \[\n        out_dir / "overall_metrics\.csv",\n)(?!        out_dir / "case_block_traces\.csv",\n)'
    )
    patched, files_count = files_pattern.subn(
        lambda m: m.group(1) + '        out_dir / "case_block_traces.csv",\n',
        patched,
        count=1,
    )
    if files_count == 0:
        print("Note: case_block_traces.csv was not added to the file list block; continuing.")

    NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = patched
    FULL_MODULE = _load_notebook_module(
        "dice_train_eval_dice_pipeline_inline",
        NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE,
    )
    print("Patched embedded full pipeline to export case_block_traces.csv")
else:
    FULL_MODULE = _load_notebook_module(
        "dice_train_eval_dice_pipeline_inline",
        NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE,
    )
    print("Block-trace export already present; module reloaded.")

if '--feature_profile' not in NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE:
    patched = NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE

    old_tier_files = """TIER_FILE = {
    \"tier0\": \"tier0_full_5hz.csv\",
    \"tier1_alt\": \"tier1_alt_core_5hz.csv\",
    \"tier2\": \"tier2_core_5hz.csv\",
}
"""
    new_tier_files = """FEATURE_PROFILES = {
    \"mixed\": {
        \"tier0\": \"tier0_full_5hz.csv\",
        \"tier1_alt\": \"tier1_alt_core_5hz.csv\",
        \"tier2\": \"tier2_core_5hz.csv\",
    },
    \"full\": {
        \"tier0\": \"tier0_full_5hz.csv\",
        \"tier1_alt\": \"tier1_alt_full_5hz.csv\",
        \"tier2\": \"tier2_full_5hz.csv\",
    },
}

TIER_FILE = FEATURE_PROFILES[\"mixed\"]
"""
    patched = _replace_once(patched, old_tier_files, new_tier_files)

    old_args = """    ap.add_argument(\"--ridge_lambda\", type=float, default=1e-3)
    ap.add_argument(
        \"--protocol\",
        choices=[\"workload_holdout\", \"global\"],
        default=\"global\",
        help=\"Evaluation protocol: workload_holdout (strict) or global benign split (paper-style).\",
    )
"""
    new_args = """    ap.add_argument(\"--ridge_lambda\", type=float, default=1e-3)
    ap.add_argument(
        \"--feature_profile\",
        choices=sorted(FEATURE_PROFILES.keys()),
        default=\"mixed\",
        help=\"Feature-file profile: mixed keeps the current deployment-friendly setting; full uses full Tier-1/Tier-2 files as an upper-bound comparison.\",
    )
    ap.add_argument(
        \"--protocol\",
        choices=[\"workload_holdout\", \"global\"],
        default=\"global\",
        help=\"Evaluation protocol: workload_holdout (strict) or global benign split (paper-style).\",
    )
"""
    patched = _replace_once(patched, old_args, new_args)

    old_root = """    root = args.root.expanduser().resolve()
    out_dir = args.out_dir.expanduser().resolve() if args.out_dir else root / \"results_dice_full\"
    fig_dir = out_dir / \"figures\"
"""
    new_root = """    global TIER_FILE
    root = args.root.expanduser().resolve()
    TIER_FILE = FEATURE_PROFILES[args.feature_profile]
    if args.out_dir:
        out_dir = args.out_dir.expanduser().resolve()
    elif args.feature_profile == \"mixed\":
        out_dir = root / (\"results_dice_full_holdout\" if args.protocol == \"workload_holdout\" else \"results_dice_full\")
    else:
        suffix = f\"results_dice_full_{args.feature_profile}\"
        if args.protocol == \"workload_holdout\":
            suffix = f\"{suffix}_holdout\"
        out_dir = root / suffix
    fig_dir = out_dir / \"figures\"
"""
    patched = _replace_once(patched, old_root, new_root)

    patched, info_count = re.subn(
        r'(    fig_dir\.mkdir\(parents=True, exist_ok=True\)\n)',
        lambda m: m.group(1) + '    print(f"[INFO] feature_profile={args.feature_profile} tier_files={TIER_FILE}")\n',
        patched,
        count=1,
    )
    if info_count == 0:
        raise ValueError("Patch target not found for feature-profile info print")

    NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE = patched
    FULL_MODULE = _load_notebook_module(
        "dice_train_eval_dice_pipeline_inline",
        NOTEBOOK_TRAIN_EVAL_DICE_PIPELINE_SOURCE,
    )
    print("Patched embedded full pipeline to support mixed/full feature profiles")
else:
    print("Feature-profile support already present in embedded full pipeline.")


## Quick Jump: Full Regeneration

If you want to regenerate the released DICE results end-to-end, start with the next section:
- run the patch cell above first,
- then run `## Run End-to-End`,
- then continue downward for the paper analysis sections.


## Run End-to-End

This section runs the full notebook-local DICE pipeline and then summarizes the main paper-facing results.

Default configuration:
- `source_hz = 5`
- `fit_ratio = 0.6`
- `block_B = 60`
- `alpha = 0.05`
- `persist_k = 3`
- `gain = 0.35`
- `ridge_lambda = 1e-3`

What this run produces:
- the lightweight released-data analysis outputs
- the full DICE digital-twin evaluation
- optional workload-holdout robustness
- optional tuning sweeps
- a compact end-to-end summary for the three observability heads

This section also includes an integrated **two-stage DICE** analysis.
Stage 1 uses `Tier-0` as a lightweight always-on screen.
Stage 2 uses `Tier-0/1/2` as a richer refinement pass.
Stage 2 is triggered only when the Stage-1 workload-conditioned score exceeds a benign-calibrated suspicion threshold.

Interpretation note:
- the main DICE run remains the baseline evaluated pipeline
- the two-stage view is a paper-oriented system-design analysis built on top of the generated case-level outputs
- block-level conformal control and run-level alert behavior should be interpreted separately


In [ ]:
RUN_END_TO_END = True
INCLUDE_TUNING = False  # Set to True only when you want to regenerate tuning sweeps from scratch.
RUN_HOLDOUT = True

RUN_TWO_STAGE = True
TWO_STAGE_QUANTILES = [0.90, 0.95, 0.98]

runtime_start = perf_counter()
notebook_run_summary = {}

if RUN_END_TO_END:
    notebook_run_summary = run_notebook_pipeline(
        dataset_root=DATASET_ROOT,
        source_hz=5,
        fit_ratio=0.6,
        block_B=60,
        alpha=0.05,
        persist_k=3,
        gain=0.35,
        ridge_lambda=1e-3,
        run_holdout=RUN_HOLDOUT,
        include_tuning=INCLUDE_TUNING,
    )
else:
    print('Skipped end-to-end run. Set RUN_END_TO_END=True to execute.')

runtime_seconds = round(perf_counter() - runtime_start, 2)
runtime_summary = {
    'runtime_seconds': runtime_seconds,
    'runtime_minutes': round(runtime_seconds / 60.0, 2),
    'run_end_to_end': RUN_END_TO_END,
    'run_holdout': RUN_HOLDOUT,
    'include_tuning': INCLUDE_TUNING,
    'repo_root': str(REPO_ROOT),
    'dataset_root': str(DATASET_ROOT),
    **notebook_run_summary,
}
NOTEBOOK_RUNTIME.write_text(json.dumps(runtime_summary, indent=2))

if (OUT_FULL / 'overall_metrics.csv').exists():
    overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
    sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
    diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')

    summary_df = (
        overall_full[['config', 'roc_auc_wc', 'pr_auc_wc']]
        .merge(
            sequential[
                ['config', 'benign_run_alert_rate', 'anomaly_detect_rate', 'median_time_to_detect_s']
            ],
            on='config',
            how='left',
        )
        .merge(
            diagnosis[['config', 'top1_acc', 'top2_acc']],
            on='config',
            how='left',
        )
    )

    config_label_map = {
        'tier0': 'Tier-0',
        'tier0_tier1': 'Tier-0/1',
        'tier0_tier1_tier2': 'Tier-0/1/2',
    }
    config_order = ['tier0', 'tier0_tier1', 'tier0_tier1_tier2']
    summary_df['label'] = summary_df['config'].map(config_label_map)
    summary_df['sort_key'] = summary_df['config'].map({k: i for i, k in enumerate(config_order)})
    summary_df = summary_df.sort_values('sort_key').reset_index(drop=True)

    row_final = summary_df[summary_df['config'] == 'tier0_tier1_tier2'].iloc[0]

    display(Markdown('### End-to-end DICE run summary'))
    display(
        pd.DataFrame([
            {
                'Runtime (min)': round(runtime_summary['runtime_minutes'], 2),
                'Final head': 'Tier-0/1/2',
                'AUC-PR': round(float(row_final['pr_auc_wc']), 4),
                'ROC-AUC': round(float(row_final['roc_auc_wc']), 4),
                'Benign alert rate': round(float(row_final['benign_run_alert_rate']), 4),
                'Detection rate': round(float(row_final['anomaly_detect_rate']), 4),
                'Median TTD (s)': round(float(row_final['median_time_to_detect_s']), 2),
                'Top-1 diagnosis': round(float(row_final['top1_acc']), 4),
                'Top-2 diagnosis': round(float(row_final['top2_acc']), 4),
            }
        ])
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    x = np.arange(len(summary_df))
    width = 0.34

    axes[0].bar(x - width / 2, summary_df['pr_auc_wc'], width=width, color='#4E79A7', label='AUC-PR')
    axes[0].bar(x + width / 2, summary_df['roc_auc_wc'], width=width, color='#59A14F', label='ROC-AUC')
    axes[0].set_title('Scoring')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(summary_df['label'])
    axes[0].set_ylim(0, 1.05)
    axes[0].legend(frameon=False)
    axes[0].grid(axis='y', alpha=0.20)

    axes[1].bar(x - width / 2, summary_df['anomaly_detect_rate'], width=width, color='#E15759', label='Detection')
    axes[1].bar(x + width / 2, summary_df['benign_run_alert_rate'], width=width, color='#F28E2B', label='Benign alert')
    axes[1].set_title('Operational')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(summary_df['label'])
    axes[1].set_ylim(0, 1.05)
    axes[1].legend(frameon=False)
    axes[1].grid(axis='y', alpha=0.20)

    axes[2].bar(x - width / 2, summary_df['top1_acc'], width=width, color='#76B7B2', label='Top-1')
    axes[2].bar(x + width / 2, summary_df['top2_acc'], width=width, color='#B07AA1', label='Top-2')
    axes[2].set_title('Diagnosis')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(summary_df['label'])
    axes[2].set_ylim(0, 1.05)
    axes[2].legend(frameon=False)
    axes[2].grid(axis='y', alpha=0.20)

    fig.suptitle('DICE end-to-end summary', fontsize=16, fontweight='bold')
    fig.tight_layout()
    plt.show()

    if RUN_TWO_STAGE and (OUT_FULL / 'case_predictions.csv').exists():
        display(Markdown('### Two-stage DICE summary'))

        case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv').copy()

        stage1 = (
            case_pred[case_pred['config'] == 'tier0'][
                ['case_id', 'workload', 'stressor', 'label', 'run_alert', 'run_score_wc', 'n_features', 'time_to_detect_s']
            ]
            .rename(
                columns={
                    'run_alert': 'stage1_alert',
                    'run_score_wc': 'stage1_score_wc',
                    'n_features': 'stage1_features',
                    'time_to_detect_s': 'stage1_ttd_s',
                }
            )
        )

        stage2 = (
            case_pred[case_pred['config'] == 'tier0_tier1_tier2'][
                ['case_id', 'workload', 'stressor', 'label', 'run_alert', 'run_score_wc', 'n_features', 'time_to_detect_s']
            ]
            .rename(
                columns={
                    'run_alert': 'stage2_alert',
                    'run_score_wc': 'stage2_score_wc',
                    'n_features': 'stage2_features',
                    'time_to_detect_s': 'stage2_ttd_s',
                }
            )
        )

        two_stage_df = stage1.merge(
            stage2,
            on=['case_id', 'workload', 'stressor', 'label'],
            how='inner',
        )

        if two_stage_df.empty:
            display(Markdown('Two-stage merge produced no rows.'))
        else:
            y = two_stage_df['label'].to_numpy(dtype=int)
            benign_mask = y == 0
            anomaly_mask = y == 1

            stage1_features = int(two_stage_df['stage1_features'].iloc[0])
            stage2_features = int(two_stage_df['stage2_features'].iloc[0])

            def safe_pr_auc(y_true, score):
                if len(np.unique(y_true)) < 2:
                    return np.nan
                return float(average_precision_score(y_true, score))

            def safe_roc_auc(y_true, score):
                if len(np.unique(y_true)) < 2:
                    return np.nan
                return float(roc_auc_score(y_true, score))

            def safe_median_ttd(ttd_values, final_alert):
                vals = pd.Series(ttd_values)[(final_alert == 1) & anomaly_mask].dropna()
                return float(vals.median()) if len(vals) else np.nan

            rows = []

            final_alert = two_stage_df['stage1_alert'].to_numpy(dtype=int)
            final_score = two_stage_df['stage1_score_wc'].to_numpy(dtype=float)
            rows.append(
                {
                    'policy': 'Tier-0 only',
                    'trigger_quantile': np.nan,
                    'trigger_rate_all': 0.0,
                    'trigger_rate_benign': 0.0,
                    'trigger_rate_anomaly': 0.0,
                    'avg_feature_budget': float(stage1_features),
                    'auc_pr': safe_pr_auc(y, final_score),
                    'roc_auc': safe_roc_auc(y, final_score),
                    'benign_alert_rate': float(final_alert[benign_mask].mean()),
                    'detection_rate': float(final_alert[anomaly_mask].mean()),
                    'median_ttd_s': safe_median_ttd(two_stage_df['stage1_ttd_s'], final_alert),
                }
            )

            benign_scores = two_stage_df.loc[benign_mask, 'stage1_score_wc']

            for q in TWO_STAGE_QUANTILES:
                gate = float(benign_scores.quantile(q))
                trigger = (two_stage_df['stage1_score_wc'] >= gate).to_numpy(dtype=bool)

                final_alert = np.where(trigger, two_stage_df['stage2_alert'], two_stage_df['stage1_alert']).astype(int)
                final_score = np.where(trigger, two_stage_df['stage2_score_wc'], two_stage_df['stage1_score_wc']).astype(float)
                final_ttd = np.where(trigger, two_stage_df['stage2_ttd_s'], two_stage_df['stage1_ttd_s'])

                rows.append(
                    {
                        'policy': f'Two-stage q={q:.2f}',
                        'trigger_quantile': q,
                        'trigger_rate_all': float(trigger.mean()),
                        'trigger_rate_benign': float(trigger[benign_mask].mean()),
                        'trigger_rate_anomaly': float(trigger[anomaly_mask].mean()),
                        'avg_feature_budget': float(stage1_features + trigger.mean() * (stage2_features - stage1_features)),
                        'auc_pr': safe_pr_auc(y, final_score),
                        'roc_auc': safe_roc_auc(y, final_score),
                        'benign_alert_rate': float(final_alert[benign_mask].mean()),
                        'detection_rate': float(final_alert[anomaly_mask].mean()),
                        'median_ttd_s': safe_median_ttd(final_ttd, final_alert),
                    }
                )

            final_alert = two_stage_df['stage2_alert'].to_numpy(dtype=int)
            final_score = two_stage_df['stage2_score_wc'].to_numpy(dtype=float)
            rows.append(
                {
                    'policy': 'Always Tier-0/1/2',
                    'trigger_quantile': 0.0,
                    'trigger_rate_all': 1.0,
                    'trigger_rate_benign': 1.0,
                    'trigger_rate_anomaly': 1.0,
                    'avg_feature_budget': float(stage2_features),
                    'auc_pr': safe_pr_auc(y, final_score),
                    'roc_auc': safe_roc_auc(y, final_score),
                    'benign_alert_rate': float(final_alert[benign_mask].mean()),
                    'detection_rate': float(final_alert[anomaly_mask].mean()),
                    'median_ttd_s': safe_median_ttd(two_stage_df['stage2_ttd_s'], final_alert),
                }
            )

            two_stage_summary = pd.DataFrame(rows)

            two_stage_out = OUT_PAPER / 'full'
            two_stage_fig = two_stage_out / 'figures'
            two_stage_out.mkdir(parents=True, exist_ok=True)
            two_stage_fig.mkdir(parents=True, exist_ok=True)
            two_stage_summary.to_csv(two_stage_out / 'two_stage_dice_summary.csv', index=False)

            display(
                two_stage_summary[
                    [
                        'policy',
                        'trigger_rate_all',
                        'trigger_rate_benign',
                        'trigger_rate_anomaly',
                        'avg_feature_budget',
                        'auc_pr',
                        'roc_auc',
                        'benign_alert_rate',
                        'detection_rate',
                        'median_ttd_s',
                    ]
                ].round(4)
            )

            fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

            frontier_df = two_stage_summary.copy()
            sc = axes[0].scatter(
                frontier_df['avg_feature_budget'],
                frontier_df['detection_rate'],
                s=600 * frontier_df['auc_pr'].fillna(0.5),
                c=frontier_df['benign_alert_rate'],
                cmap='YlOrRd',
                edgecolor='black',
                linewidth=0.8,
            )

            for _, row in frontier_df.iterrows():
                axes[0].text(
                    row['avg_feature_budget'] + 0.2,
                    row['detection_rate'] + 0.01,
                    row['policy'].replace('Two-stage ', ''),
                    fontsize=9,
                )

            axes[0].set_xlabel('Average active feature budget')
            axes[0].set_ylabel('Detection rate')
            axes[0].set_ylim(0, 1.05)
            axes[0].set_title('Two-stage performance-cost frontier')
            axes[0].grid(alpha=0.20)
            cbar = fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04)
            cbar.set_label('Benign alert rate')

            mid_df = two_stage_summary[two_stage_summary['policy'].str.startswith('Two-stage')].copy()
            x2 = np.arange(len(mid_df))
            w = 0.34

            axes[1].bar(x2 - w / 2, mid_df['trigger_rate_benign'], width=w, color='#4E79A7', label='Benign trigger')
            axes[1].bar(x2 + w / 2, mid_df['trigger_rate_anomaly'], width=w, color='#E15759', label='Anomaly trigger')
            axes[1].set_xticks(x2)
            axes[1].set_xticklabels([f"q={q:.2f}" for q in mid_df['trigger_quantile']])
            axes[1].set_ylim(0, 1.05)
            axes[1].set_ylabel('Trigger rate')
            axes[1].set_title('Stage-2 trigger selectivity')
            axes[1].legend(frameon=False)
            axes[1].grid(axis='y', alpha=0.20)

            fig.suptitle('Two-stage DICE: Tier-0 screen plus selective refinement', fontsize=15, fontweight='bold')
            fig.tight_layout()
            fig.savefig(two_stage_fig / 'fig_two_stage_dice_summary.png', dpi=220, bbox_inches='tight')
            plt.show()
else:
    display(Markdown('No full-result artifacts found yet. Run the notebook once with `RUN_END_TO_END = True`.'))


## Reading Guide

Run this notebook from top to bottom if you want the full paper bundle.

- **Run End-to-End**: regenerate the released outputs and the two-stage summary.
- **Experimental setup and released data**: describe the dataset, tier inventory, and workload/stressor matrix.
- **Main DICE performance**: show score separation, sequential alerts, diagnosis, and reliability.
- **Variants, robustness, and DSE**: show how the method scales with observability, feature budgets, decision choices, and low-overhead deployment context.
- **Case study and attribution**: visualize the digital-twin reference and explain the evidence.
- **Paper bundle**: export the main-paper and appendix artifacts.


## Results Map Aligned to the Draft

This notebook follows the intended ITC results narrative as one continuous story.

- **Experimental setup**: released data inventory, tier exposure, and workload/stressor coverage.
- **Result set 1**: main digital-twin performance under the final `Tier-0/1/2` head.
- **Result set 2**: reliability without per-workload threshold tuning.
- **Result set 3**: observability heads, two-stage DICE, workload holdout, feature-budget sweeps, runtime/power context, and broader design-space exploration.
- **Result set 4**: case-level virtual-system overlay, attribution, grounded LLM triage support, and evidence concentration.
- **Appendix outputs**: bootstrap intervals, paper bundles, research directions, and reproducibility metadata.


## 1. Experimental Setup and Released Data Inventory

This section summarizes the released dataset artifacts that support the paper narrative.

`AF index` in these early plots is the lightweight anomaly-factor score from the released analysis pass.
It is useful for data description and separability checks, but it is **not** the full DICE digital-twin score used in the later sections.


In [ ]:
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

setup_snapshot = features[['tier_name', 'n_features_common', 'n_features_union']].copy()
setup_snapshot = setup_snapshot.rename(
    columns={
        'tier_name': 'Tier',
        'n_features_common': 'Common features',
        'n_features_union': 'Union features',
    }
)

workload_snapshot = workload[['tier_name', 'workload', 'nominal_score', 'anomaly_median_score', 'anomaly_nominal_ratio']].copy()
workload_snapshot = workload_snapshot.rename(
    columns={
        'tier_name': 'Tier',
        'workload': 'Workload',
        'nominal_score': 'Nominal AF index',
        'anomaly_median_score': 'Median anomaly AF index',
        'anomaly_nominal_ratio': 'Anomaly/nominal ratio',
    }
)

quality_snapshot = quality[['tier', 'case_id', 'rows_5hz', 'numeric_cols', 'nan_fraction']].head(8).copy()
quality_snapshot = quality_snapshot.rename(
    columns={
        'tier': 'Tier',
        'case_id': 'Case',
        'rows_5hz': 'Rows @5Hz',
        'numeric_cols': 'Numeric cols',
        'nan_fraction': 'NaN fraction',
    }
)

display(Markdown('### Released data snapshot'))
display(setup_snapshot)

display(Markdown('### Workload-level AF-index context'))
display(workload_snapshot.round(4))

display(Markdown('### Case-quality spot check'))
display(quality_snapshot.round(4))


### Workload-stressor design matrix

This figure shows the workload and stressor combinations present in the released dataset. It belongs in the setup section because it describes the evaluation design, not the outcome.


In [ ]:
workloads = ['BROWSER', 'VIDEO_SW', 'PY_AI', 'PY_STATS']
stressors = ['NOMINAL', 'ATOMIC', 'BRANCH', 'CACHE', 'MEMBW', 'TLB']

matrix_rows = []
for workload_name in workloads:
    row = {'workload': workload_name}
    for stressor_name in stressors:
        case_dir = DATASET_ROOT / 'tier0' / f'{workload_name}__{stressor_name}'
        row[stressor_name] = int(case_dir.exists())
    matrix_rows.append(row)

design_matrix = pd.DataFrame(matrix_rows).set_index('workload')
design_matrix.to_csv(PAPER_FULL / 'workload_stressor_design_matrix.csv')

display(design_matrix)

fig, ax = plt.subplots(figsize=(8.4, 4.6))
im = ax.imshow(design_matrix.values, cmap='Blues', aspect='auto', vmin=0, vmax=max(1, int(design_matrix.values.max())))

ax.set_xticks(np.arange(len(design_matrix.columns)))
ax.set_xticklabels(design_matrix.columns, rotation=20, ha='right')
ax.set_yticks(np.arange(len(design_matrix.index)))
ax.set_yticklabels(design_matrix.index)

for i in range(design_matrix.shape[0]):
    for j in range(design_matrix.shape[1]):
        ax.text(j, i, int(design_matrix.iloc[i, j]), ha='center', va='center', fontsize=10)

ax.set_title('Workload-stressor design matrix', fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Case count')
fig.tight_layout()
design_matrix_png = PAPER_FIG / 'fig_workload_stressor_design_matrix.png'
fig.savefig(design_matrix_png, dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
preview_paths = [
    FIG / 'fig_heatmap_pr_auc.png',
    FIG / 'fig_run_score_distributions.png',
    FIG / 'fig_af_timeseries_tier2.png',
    REPO_ROOT / 'figs' / 'dice_tier_feature_taxonomy.png',
]

display(Markdown('### Experimental-setup figure previews'))
for path in preview_paths:
    if path.exists():
        print(path)
        display(Image(filename=str(path)))


## 2. Main DICE Performance

This section reports the main digital-twin results for the paper:
- workload-conditioned scoring,
- sequential decisioning,
- mechanism-level diagnosis,
- tier-level contribution summaries.


In [ ]:
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
stressor_full = pd.read_csv(OUT_FULL / 'stressor_metrics_final_config.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
mechanism = pd.read_csv(OUT_FULL / 'mechanism_group_summary.csv')
tier_contrib = pd.read_csv(OUT_FULL / 'stressor_tier_contributions.csv')

row_final = overall_full[overall_full['config'] == 'tier0_tier1_tier2'].iloc[0]
seq_final = sequential[sequential['config'] == 'tier0_tier1_tier2'].iloc[0]
diag_final = diagnosis[diagnosis['config'] == 'tier0_tier1_tier2'].iloc[0]

main_summary = pd.DataFrame([
    {
        'Final head': 'Tier-0/1/2',
        'ROC-AUC (WC)': float(row_final['roc_auc_wc']),
        'AUC-PR (WC)': float(row_final['pr_auc_wc']),
        'Benign alert rate': float(seq_final['benign_run_alert_rate']),
        'Detection rate': float(seq_final['anomaly_detect_rate']),
        'Median TTD (s)': float(seq_final['median_time_to_detect_s']),
        'Top-1 diagnosis': float(diag_final['top1_acc']),
        'Top-2 diagnosis': float(diag_final['top2_acc']),
    }
])

display(Markdown('### Final-head summary'))
display(main_summary.round(4))

display(Markdown('### Main tables used in the paper'))
display(overall_full.round(4))
display(sequential.round(4))
display(diagnosis.round(4))

display(Markdown('### Final-head mechanism and tier summaries'))
display(mechanism.round(4))
display(tier_contrib.round(4))

for path in [
    OUT_FULL / 'figures' / 'fig_run_score_boxplot_wc.png',
    OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png',
    OUT_FULL / 'figures' / 'fig_detection_latency.png',
]:
    if path.exists():
        display(Image(filename=str(path)))


### Benign vs anomaly score separation

This figure shows the final-head workload-conditioned run scores for benign and anomalous cases. It works better in the results section than in the setup section because it is a direct outcome figure.


In [ ]:
paper_full = OUT_PAPER / 'full'
paper_fig = paper_full / 'figures'
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv').copy()
plot_df = case_pred[case_pred['config'] == 'tier0_tier1_tier2'].copy()
plot_df['label_name'] = plot_df['label'].map({0: 'Benign', 1: 'Anomaly'})

groups = [
    plot_df.loc[plot_df['label_name'] == 'Benign', 'run_score_wc'].dropna().to_numpy(),
    plot_df.loc[plot_df['label_name'] == 'Anomaly', 'run_score_wc'].dropna().to_numpy(),
]

fig, ax = plt.subplots(figsize=(7.6, 4.8))
parts = ax.violinplot(groups, positions=[0, 1], showmeans=False, showmedians=True, showextrema=False)
colors = ['#4E79A7', '#E15759']
for body, color in zip(parts['bodies'], colors):
    body.set_facecolor(color)
    body.set_edgecolor('black')
    body.set_alpha(0.65)

rng = np.random.default_rng(0)
for xpos, vals, color in zip([0, 1], groups, colors):
    jitter = rng.uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(
        np.full(len(vals), xpos) + jitter,
        vals,
        s=24,
        alpha=0.75,
        color=color,
        edgecolor='white',
        linewidth=0.4,
    )

ax.set_xticks([0, 1])
ax.set_xticklabels(['Benign', 'Anomaly'])
ax.set_ylabel('Workload-conditioned run score')
ax.set_title('Final-head score separation', fontweight='bold')
ax.grid(axis='y', alpha=0.20)
fig.tight_layout()
score_sep_png = paper_fig / 'fig_final_head_score_separation.png'
fig.savefig(score_sep_png, dpi=220, bbox_inches='tight')
plt.show()


## 3. Reliability Without Per-Workload Tuning

This section addresses the paper's central reliability question: can residual-based digital-twin scores be thresholded with a target false-alarm budget without workload-specific manual tuning?

One distinction matters throughout this section. Split-conformal calibration controls the false-alarm rate at the **block level** under the calibrated benign regime. The notebook also reports **run-level** alert rates, which can be higher because a long run contains many blocks and the alert logic aggregates across them.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')
TARGET_ALPHA = 0.05

reliability_rows = []
for cfg, d in case_pred.groupby('config', sort=False):
    benign = d[d['label'] == 0].copy()
    anomaly = d[d['label'] == 1].copy()
    reliability_rows.append({
        'config': cfg,
        'target_alpha': TARGET_ALPHA,
        'benign_block_false_alarm_rate': benign['n_block_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_persist_false_alarm_rate': benign['n_persist_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_run_false_alarm_rate': benign['run_alert'].mean(),
        'anomaly_run_detection_rate': anomaly['run_alert'].mean(),
        'median_anomaly_time_to_detect_s': anomaly.loc[anomaly['run_alert'] == 1, 'time_to_detect_s'].median(),
    })

reliability = pd.DataFrame(reliability_rows)
reliability_by_workload = (
    case_pred[case_pred['label'] == 0]
    .groupby(['config', 'workload'], sort=False)
    .apply(
        lambda x: pd.Series({
            'target_alpha': TARGET_ALPHA,
            'benign_block_false_alarm_rate': x['n_block_alerts'].sum() / x['n_blocks'].sum(),
            'benign_persist_false_alarm_rate': x['n_persist_alerts'].sum() / x['n_blocks'].sum(),
            'benign_run_false_alarm_rate': x['run_alert'].mean(),
        }),
        include_groups=False,
    )
    .reset_index()
)

(OUT_PAPER / 'full').mkdir(parents=True, exist_ok=True)
reliability.to_csv(OUT_PAPER / 'full' / 'conformal_reliability_summary.csv', index=False)
reliability_by_workload.to_csv(OUT_APPENDIX / 'full' / 'conformal_reliability_by_workload.csv', index=False)

cfg_label_map = {'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'}
reliability['label'] = reliability['config'].map(cfg_label_map)

fig, ax = plt.subplots(figsize=(8.6, 4.6))
x = np.arange(len(reliability))
ax.plot(x, reliability['target_alpha'], color='black', linestyle='--', linewidth=2, label='Target alpha')
ax.scatter(x, reliability['benign_block_false_alarm_rate'], s=120, color='#E15759', label='Observed benign block FAR')
ax.scatter(x, reliability['benign_run_false_alarm_rate'], s=120, color='#4E79A7', label='Observed benign run FAR')
for i, row in reliability.iterrows():
    ax.text(i, row['benign_block_false_alarm_rate'] + 0.015, row['label'], ha='center', fontsize=10)
ax.set_xticks([])
ax.set_ylim(0, max(0.12, reliability[['target_alpha', 'benign_block_false_alarm_rate', 'benign_run_false_alarm_rate']].max().max() + 0.04))
ax.set_ylabel('False-alarm rate')
ax.set_title('Split-conformal reliability without per-workload tuning', fontweight='bold')
ax.legend(frameon=False)
ax.grid(alpha=0.20)
reliability_png = OUT_PAPER / 'full' / 'fig_conformal_reliability.png'
fig.tight_layout()
fig.savefig(reliability_png, dpi=220, bbox_inches='tight')
plt.close(fig)

display(reliability.round(4))
display(Image(filename=str(reliability_png)))

display(Markdown('### Benign false-alarm rate by workload'))
display(reliability_by_workload.round(4))


## 4. Observability Heads and Digital-Twin Variants

This section isolates what changes when DICE moves from `Tier-0` to `Tier-0/1` to `Tier-0/1/2`.
It separates the base digital-twin score, the workload-conditioned score, and the diagnosis head.


In [ ]:
diag = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
variant_rows = []
for _, row in overall_full.iterrows():
    cfg = row['config']
    drow = diag[diag['config'] == cfg].iloc[0]
    variant_rows.append({
        'config': cfg,
        'single_head_roc_auc': row['roc_auc'],
        'single_head_pr_auc': row['pr_auc'],
        'workload_conditioned_roc_auc': row['roc_auc_wc'],
        'workload_conditioned_pr_auc': row['pr_auc_wc'],
        'mechanism_top1_acc': drow['top1_acc'],
        'mechanism_top2_acc': drow['top2_acc'],
        'mechanism_macro_f1': drow['macro_f1'],
    })
variant_summary = pd.DataFrame(variant_rows)
variant_summary['label'] = variant_summary['config'].map({'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'})
variant_summary.to_csv(OUT_PAPER / 'full' / 'digital_twin_variant_summary.csv', index=False)
display(variant_summary.round(4))


## 4B. Two-Stage DICE

This section tests a two-stage version of DICE.
Stage 1 uses `Tier-0` as a lightweight screen that runs all the time.
Stage 2 uses `Tier-0/1/2` as a richer refinement pass and runs only when the Stage-1 score crosses a suspicion threshold set from benign Stage-1 scores.
The goal is to see how much performance we can keep while reducing the average active feature budget.


In [ ]:
paper_full = OUT_PAPER / 'full'
paper_fig = paper_full / 'figures'
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

two_stage_csv = paper_full / 'two_stage_dice_summary.csv'
if two_stage_csv.exists():
    two_stage_summary = pd.read_csv(two_stage_csv)
else:
    case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv').copy()

    stage1 = (
        case_pred[case_pred['config'] == 'tier0'][
            ['case_id', 'workload', 'stressor', 'label', 'run_alert', 'run_score_wc', 'n_features', 'time_to_detect_s']
        ]
        .rename(
            columns={
                'run_alert': 'stage1_alert',
                'run_score_wc': 'stage1_score_wc',
                'n_features': 'stage1_features',
                'time_to_detect_s': 'stage1_ttd_s',
            }
        )
    )

    stage2 = (
        case_pred[case_pred['config'] == 'tier0_tier1_tier2'][
            ['case_id', 'workload', 'stressor', 'label', 'run_alert', 'run_score_wc', 'n_features', 'time_to_detect_s']
        ]
        .rename(
            columns={
                'run_alert': 'stage2_alert',
                'run_score_wc': 'stage2_score_wc',
                'n_features': 'stage2_features',
                'time_to_detect_s': 'stage2_ttd_s',
            }
        )
    )

    df = stage1.merge(stage2, on=['case_id', 'workload', 'stressor', 'label'], how='inner')
    if df.empty:
        raise ValueError('Two-stage merge produced no rows. Check case_predictions.csv.')

    y = df['label'].to_numpy(dtype=int)
    benign_mask = y == 0
    anomaly_mask = y == 1
    stage1_features = int(df['stage1_features'].iloc[0])
    stage2_features = int(df['stage2_features'].iloc[0])

    def safe_pr_auc(y_true, score):
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(average_precision_score(y_true, score))

    def safe_roc_auc(y_true, score):
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, score))

    def safe_median_ttd(ttd_values, final_alert):
        vals = pd.Series(ttd_values)[(final_alert == 1) & anomaly_mask].dropna()
        return float(vals.median()) if len(vals) else np.nan

    rows = []

    final_alert = df['stage1_alert'].to_numpy(dtype=int)
    final_score = df['stage1_score_wc'].to_numpy(dtype=float)
    rows.append(
        {
            'policy': 'Tier-0 only',
            'trigger_quantile': np.nan,
            'trigger_rate_all': 0.0,
            'trigger_rate_benign': 0.0,
            'trigger_rate_anomaly': 0.0,
            'avg_feature_budget': float(stage1_features),
            'auc_pr': safe_pr_auc(y, final_score),
            'roc_auc': safe_roc_auc(y, final_score),
            'benign_alert_rate': float(final_alert[benign_mask].mean()),
            'detection_rate': float(final_alert[anomaly_mask].mean()),
            'median_ttd_s': safe_median_ttd(df['stage1_ttd_s'], final_alert),
        }
    )

    benign_scores = df.loc[benign_mask, 'stage1_score_wc']
    for q in [0.90, 0.95, 0.98]:
        gate = float(benign_scores.quantile(q))
        trigger = (df['stage1_score_wc'] >= gate).to_numpy(dtype=bool)
        final_alert = np.where(trigger, df['stage2_alert'], df['stage1_alert']).astype(int)
        final_score = np.where(trigger, df['stage2_score_wc'], df['stage1_score_wc']).astype(float)
        final_ttd = np.where(trigger, df['stage2_ttd_s'], df['stage1_ttd_s'])
        rows.append(
            {
                'policy': f'Two-stage q={q:.2f}',
                'trigger_quantile': q,
                'trigger_rate_all': float(trigger.mean()),
                'trigger_rate_benign': float(trigger[benign_mask].mean()),
                'trigger_rate_anomaly': float(trigger[anomaly_mask].mean()),
                'avg_feature_budget': float(stage1_features + trigger.mean() * (stage2_features - stage1_features)),
                'auc_pr': safe_pr_auc(y, final_score),
                'roc_auc': safe_roc_auc(y, final_score),
                'benign_alert_rate': float(final_alert[benign_mask].mean()),
                'detection_rate': float(final_alert[anomaly_mask].mean()),
                'median_ttd_s': safe_median_ttd(final_ttd, final_alert),
            }
        )

    final_alert = df['stage2_alert'].to_numpy(dtype=int)
    final_score = df['stage2_score_wc'].to_numpy(dtype=float)
    rows.append(
        {
            'policy': 'Always Tier-0/1/2',
            'trigger_quantile': 0.0,
            'trigger_rate_all': 1.0,
            'trigger_rate_benign': 1.0,
            'trigger_rate_anomaly': 1.0,
            'avg_feature_budget': float(stage2_features),
            'auc_pr': safe_pr_auc(y, final_score),
            'roc_auc': safe_roc_auc(y, final_score),
            'benign_alert_rate': float(final_alert[benign_mask].mean()),
            'detection_rate': float(final_alert[anomaly_mask].mean()),
            'median_ttd_s': safe_median_ttd(df['stage2_ttd_s'], final_alert),
        }
    )

    two_stage_summary = pd.DataFrame(rows)
    two_stage_summary.to_csv(two_stage_csv, index=False)

display(two_stage_summary.round(4))


fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
frontier_df = two_stage_summary.copy()
sc = axes[0].scatter(
    frontier_df['avg_feature_budget'],
    frontier_df['detection_rate'],
    s=600 * frontier_df['auc_pr'].fillna(0.5),
    c=frontier_df['benign_alert_rate'],
    cmap='YlOrRd',
    edgecolor='black',
    linewidth=0.8,
)
for _, row in frontier_df.iterrows():
    axes[0].text(
        row['avg_feature_budget'] + 0.2,
        row['detection_rate'] + 0.01,
        row['policy'].replace('Two-stage ', ''),
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none', alpha=0.85),
    )
axes[0].set_xlabel('Average active feature budget')
axes[0].set_ylabel('Detection rate')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Two-stage performance-cost frontier', fontweight='bold')
axes[0].grid(alpha=0.20)
cbar = fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04)
cbar.set_label('Benign alert rate')

mid_df = two_stage_summary[two_stage_summary['policy'].str.startswith('Two-stage')].copy().sort_values('trigger_quantile')
axes[1].plot(
    mid_df['trigger_quantile'],
    mid_df['trigger_rate_benign'],
    marker='o',
    linewidth=2.2,
    color='#4E79A7',
    label='Benign trigger',
)
axes[1].plot(
    mid_df['trigger_quantile'],
    mid_df['trigger_rate_anomaly'],
    marker='o',
    linewidth=2.2,
    color='#E15759',
    label='Anomaly trigger',
)
axes[1].fill_between(
    mid_df['trigger_quantile'],
    mid_df['trigger_rate_benign'],
    mid_df['trigger_rate_anomaly'],
    color='#F3D7DA',
    alpha=0.25,
)
for _, row in mid_df.iterrows():
    axes[1].text(
        row['trigger_quantile'],
        row['trigger_rate_anomaly'] + 0.035,
        f"{int(row['avg_feature_budget'])} feat",
        ha='center',
        fontsize=8.5,
        bbox=dict(boxstyle='round,pad=0.16', fc='white', ec='none', alpha=0.8),
    )
axes[1].set_xlabel('Benign-quantile gate')
axes[1].set_ylabel('Trigger rate')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Stage-2 trigger selectivity', fontweight='bold')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.20)

fig.suptitle('Two-stage DICE: Tier-0 screen plus selective refinement', fontsize=15, fontweight='bold')
fig.tight_layout()
two_stage_png = paper_fig / 'fig_two_stage_dice_frontier.png'
fig.savefig(two_stage_png, dpi=220, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(two_stage_png)))


## 5. Workload-Holdout Robustness and Drift Proxy

The draft frames workload/software drift as a practical in-field challenge.
This section uses workload holdout as the released robustness proxy.


In [ ]:
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    display(holdout.round(4))

    holdout_plot = holdout.copy()
    holdout_plot['label'] = holdout_plot['config'].map({'tier0': 'Tier-0', 'tier0_tier1': 'Tier-0/1', 'tier0_tier1_tier2': 'Tier-0/1/2'})
    fig, ax = plt.subplots(figsize=(8.4, 4.8))
    x = np.arange(len(holdout_plot))
    ax.bar(x - 0.15, holdout_plot['mean_pr_auc'], width=0.30, color='#4E79A7', label='Mean holdout AUC-PR')
    ax.bar(x + 0.15, holdout_plot['worst_pr_auc'], width=0.30, color='#E15759', label='Worst-workload AUC-PR')
    ax.set_xticks(x)
    ax.set_xticklabels(holdout_plot['label'])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('AUC-PR')
    ax.set_title('Workload-holdout robustness', fontweight='bold')
    ax.legend(frameon=False)
    ax.grid(axis='y', alpha=0.20)
    holdout_png = OUT_PAPER / 'full' / 'fig_holdout_robustness.png'
    fig.tight_layout()
    fig.savefig(holdout_png, dpi=220, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(holdout_png)))
else:
    display(Markdown('No holdout summary found yet.'))


## 6. DICE-Specific Design-Space Evaluation

This section turns the completed DICE run into a design-space study centered on the digital twin rather than only a single ROC/PR table.

Sweeps 1, 4, and 5 are derived from the main released outputs. Sweeps 2 and 3 come from the tuning outputs and are available after the notebook is run with `INCLUDE_TUNING = True`. Sweep 6 is a notebook-local feature-budget study that retrains the final head on ranked feature subsets.


This design-space section highlights six DICE-specific sweeps.

1. **Observability heads**: how much each telemetry tier helps.
2. **Twin synchronization map**: how `gain` and `block_B` trade sensitivity against stability.
3. **Decision calibration frontier**: how `alpha` and `persist_k` trade false alarms against detection rate.
4. **Diagnosis quality vs feature budget**: how much diagnosis quality is gained as observability grows.
5. **Portability frontier**: how robustness improves as DICE moves from `Tier-0` toward `Tier-0/1/2`.
6. **Top-feature budget sweep**: how ROC/PR behavior changes when the final head keeps only the highest-ranked features.


In [ ]:
display(Markdown("### DICE visual design-space exploration"))

paper_full = OUT_PAPER / "full"
paper_full.mkdir(parents=True, exist_ok=True)

overall_full = pd.read_csv(OUT_FULL / "overall_metrics.csv")
sequential = pd.read_csv(OUT_FULL / "sequential_metrics.csv")
diagnosis = pd.read_csv(OUT_FULL / "stressor_diagnosis_metrics.csv")

holdout_path = OUT_HOLDOUT / "holdout_robustness_summary.csv"
holdout = pd.read_csv(holdout_path) if holdout_path.exists() else None

reliability_path = paper_full / "conformal_reliability_summary.csv"
reliability = pd.read_csv(reliability_path) if reliability_path.exists() else None

stage1_path = DATASET_ROOT / "results_dice_tuning" / "sweep_stage1_gain_block.csv"
stage2_path = DATASET_ROOT / "results_dice_tuning" / "sweep_stage2_alpha_persist.csv"
stage1 = pd.read_csv(stage1_path) if stage1_path.exists() else None
stage2 = pd.read_csv(stage2_path) if stage2_path.exists() else None

config_label_map = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}
feature_budget_map = {
    "tier0": 46,
    "tier0_tier1": 57,
    "tier0_tier1_tier2": 64,
}


def draw_heatmap(ax, pivot_df, title, cmap="viridis", vmin=None, vmax=None, fmt="{:.3f}"):
    vals = pivot_df.values.astype(float)
    im = ax.imshow(vals, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(pivot_df.shape[1]))
    ax.set_xticklabels([str(c) for c in pivot_df.columns])
    ax.set_yticks(np.arange(pivot_df.shape[0]))
    ax.set_yticklabels([str(i) for i in pivot_df.index])
    ax.set_title(title, fontsize=12, fontweight="bold")
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            text = "" if np.isnan(vals[i, j]) else fmt.format(vals[i, j])
            color = "white" if not np.isnan(vals[i, j]) and vals[i, j] > 0.55 else "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=9, color=color)
    return im


def load_final_head_run_metrics(out_dir: Path) -> dict:
    out = {}
    seq_file = out_dir / "sequential_metrics.csv"
    ov_file = out_dir / "overall_metrics.csv"

    if seq_file.exists():
        seq_df = pd.read_csv(seq_file)
        if "tier0_tier1_tier2" in set(seq_df["config"]):
            seq_row = seq_df[seq_df["config"] == "tier0_tier1_tier2"].iloc[0]
            out.update(
                {
                    "benign_run_alert_rate": seq_row.get("benign_run_alert_rate", np.nan),
                    "anomaly_detect_rate": seq_row.get("anomaly_detect_rate", np.nan),
                    "median_time_to_detect_s": seq_row.get("median_time_to_detect_s", np.nan),
                }
            )

    if ov_file.exists():
        ov_df = pd.read_csv(ov_file)
        if "tier0_tier1_tier2" in set(ov_df["config"]):
            ov_row = ov_df[ov_df["config"] == "tier0_tier1_tier2"].iloc[0]
            out.update(
                {
                    "roc_auc_wc": ov_row.get("roc_auc_wc", np.nan),
                    "pr_auc_wc": ov_row.get("pr_auc_wc", np.nan),
                    "fpr_run_alert": ov_row.get("fpr_run_alert", np.nan),
                    "tpr_run_alert": ov_row.get("tpr_run_alert", np.nan),
                }
            )

    return out


def enrich_tuning_table(df: pd.DataFrame | None) -> pd.DataFrame | None:
    if df is None or df.empty:
        return df

    rows = []
    for _, row in df.iterrows():
        rec = row.to_dict()

        out_dir = None
        if "out_dir" in rec and pd.notna(rec["out_dir"]):
            out_dir = Path(rec["out_dir"])

        extra = load_final_head_run_metrics(out_dir) if out_dir and out_dir.exists() else {}

        for key in [
            "roc_auc_wc",
            "pr_auc_wc",
            "fpr_run_alert",
            "tpr_run_alert",
            "benign_run_alert_rate",
            "anomaly_detect_rate",
            "median_time_to_detect_s",
        ]:
            if key not in rec or pd.isna(rec.get(key, np.nan)):
                rec[key] = extra.get(key, np.nan)

        if pd.isna(rec.get("fpr_run_alert", np.nan)):
            rec["fpr_run_alert"] = rec.get("benign_run_alert_rate", np.nan)
        if pd.isna(rec.get("tpr_run_alert", np.nan)):
            rec["tpr_run_alert"] = rec.get("anomaly_detect_rate", np.nan)

        rows.append(rec)

    return pd.DataFrame(rows)


def pareto_front(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    keep = []
    for i, row_i in df.iterrows():
        dominated = False
        xi, yi = row_i[x_col], row_i[y_col]
        for j, row_j in df.iterrows():
            if i == j:
                continue
            xj, yj = row_j[x_col], row_j[y_col]
            if (xj <= xi and yj >= yi) and (xj < xi or yj > yi):
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return df.loc[keep].sort_values([x_col, y_col])


stage1 = enrich_tuning_table(stage1)
stage2 = enrich_tuning_table(stage2)

frontier = (
    overall_full[["config", "roc_auc_wc", "pr_auc_wc"]]
    .merge(
        sequential[
            ["config", "benign_run_alert_rate", "anomaly_detect_rate", "median_time_to_detect_s"]
        ],
        on="config",
        how="left",
    )
    .merge(
        diagnosis[["config", "top1_acc", "top2_acc"]],
        on="config",
        how="left",
    )
)

if holdout is not None:
    frontier = frontier.merge(
        holdout[["config", "mean_pr_auc", "worst_pr_auc"]],
        on="config",
        how="left",
    )
else:
    frontier["mean_pr_auc"] = np.nan
    frontier["worst_pr_auc"] = np.nan

if reliability is not None:
    frontier = frontier.merge(
        reliability[["config", "benign_block_false_alarm_rate", "target_alpha"]],
        on="config",
        how="left",
    )
    frontier["reliability_margin"] = frontier["target_alpha"] - frontier["benign_block_false_alarm_rate"]
else:
    frontier["target_alpha"] = 0.05
    frontier["reliability_margin"] = frontier["target_alpha"] - frontier["benign_run_alert_rate"]

frontier["label"] = frontier["config"].map(config_label_map)
frontier["n_features"] = frontier["config"].map(feature_budget_map)
frontier["portable_pr_auc"] = frontier["mean_pr_auc"].fillna(frontier["pr_auc_wc"])
frontier["holdout_worst_pr_auc"] = frontier["worst_pr_auc"]
frontier["joint_detection_diagnosis"] = frontier["anomaly_detect_rate"] * frontier["top1_acc"]

key_rows = []

sweep1 = frontier[
    [
        "label",
        "n_features",
        "pr_auc_wc",
        "roc_auc_wc",
        "anomaly_detect_rate",
        "benign_run_alert_rate",
        "median_time_to_detect_s",
        "top1_acc",
        "top2_acc",
    ]
].copy().sort_values("n_features")

fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.0))

bubble_sizes = 180 + 900 * sweep1["top2_acc"].fillna(0)
sc = axes[0].scatter(
    sweep1["n_features"],
    sweep1["pr_auc_wc"],
    s=bubble_sizes,
    c=sweep1["anomaly_detect_rate"],
    cmap="viridis",
    edgecolor="black",
    linewidth=1.0,
)
for _, row in sweep1.iterrows():
    axes[0].text(row["n_features"] + 0.8, row["pr_auc_wc"] + 0.01, row["label"], fontsize=10)
axes[0].plot(sweep1["n_features"], sweep1["pr_auc_wc"], linestyle="--", color="#94A3B8", alpha=0.9)
axes[0].set_xlabel("Active feature budget")
axes[0].set_ylabel("Workload-conditioned AUC-PR")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Sweep 1A: observability frontier", fontweight="bold")
axes[0].grid(alpha=0.20)
fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label="Detection rate")

y = np.arange(len(sweep1))
axes[1].hlines(y, sweep1["benign_run_alert_rate"], sweep1["anomaly_detect_rate"], color="#CBD5E1", linewidth=4)
axes[1].scatter(sweep1["benign_run_alert_rate"], y, s=120, color="#F28E2B", label="Benign alert rate", zorder=3)
axes[1].scatter(sweep1["anomaly_detect_rate"], y, s=120, color="#E15759", label="Detection rate", zorder=3)
for idx, row in enumerate(sweep1.itertuples()):
    axes[1].text(max(row.benign_run_alert_rate, row.anomaly_detect_rate) + 0.02, idx, f"TTD={row.median_time_to_detect_s:.0f}s", va="center", fontsize=9)
axes[1].set_yticks(y)
axes[1].set_yticklabels(sweep1["label"])
axes[1].set_xlim(0, 1.05)
axes[1].set_xlabel("Rate")
axes[1].set_title("Sweep 1B: operational separation", fontweight="bold")
axes[1].legend(frameon=False, loc="lower right")
axes[1].grid(axis="x", alpha=0.20)

fig.suptitle("Sweep 1: Observability heads", fontsize=15, fontweight="bold")
fig.tight_layout()
sweep1_png = paper_full / "fig_sweep_1_observability_creative.png"
fig.savefig(sweep1_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(sweep1_png)))

best_head = frontier.sort_values("pr_auc_wc", ascending=False).iloc[0]
key_rows.append(
    {
        "Sweep": "1",
        "Main result": f"Best head: {best_head['label']}",
        "Key metric": f"AUC-PR={best_head['pr_auc_wc']:.3f}, detect={best_head['anomaly_detect_rate']:.3f}",
    }
)

if stage1 is not None and not stage1.empty:
    stage1_ok = stage1[stage1["status"] == "ok"].copy() if "status" in stage1.columns else stage1.copy()
    stage1_ok = stage1_ok.dropna(subset=["pr_auc_wc", "fpr_run_alert", "tpr_run_alert"], how="any").copy()
    stage1_ok["stability_margin"] = stage1_ok["tpr_run_alert"] - stage1_ok["fpr_run_alert"]

    pr_pivot = stage1_ok.pivot_table(index="gain", columns="block_B", values="pr_auc_wc", aggfunc="mean").sort_index().sort_index(axis=1)
    sm_pivot = stage1_ok.pivot_table(index="gain", columns="block_B", values="stability_margin", aggfunc="mean").sort_index().sort_index(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.9))
    im1 = draw_heatmap(axes[0], pr_pivot, "AUC-PR", cmap="Blues", vmin=0, vmax=1)
    im2 = draw_heatmap(axes[1], sm_pivot, "Stability margin (TPR - FPR)", cmap="YlGn", vmin=0, vmax=1)
    axes[0].set_xlabel("Block B")
    axes[0].set_ylabel("Gain")
    axes[1].set_xlabel("Block B")
    axes[1].set_ylabel("Gain")
    fig.suptitle("Sweep 2: Twin synchronization map", fontsize=15, fontweight="bold")
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    fig.tight_layout()
    sweep2_png = paper_full / "fig_sweep_2_gain_block_heatmaps.png"
    fig.savefig(sweep2_png, dpi=220, bbox_inches="tight")
    plt.close(fig)

    display(Image(filename=str(sweep2_png)))

    best_stage1 = stage1_ok.sort_values(["pr_auc_wc", "stability_margin"], ascending=False).iloc[0]
    key_rows.append(
        {
            "Sweep": "2",
            "Main result": f"Best gain/block: g={best_stage1['gain']}, B={int(best_stage1['block_B'])}",
            "Key metric": f"AUC-PR={best_stage1['pr_auc_wc']:.3f}, stability={best_stage1['stability_margin']:.3f}",
        }
    )
else:
    display(Markdown("**Sweep 2 skipped:** tuning outputs not found."))
    key_rows.append(
        {
            "Sweep": "2",
            "Main result": "Skipped",
            "Key metric": "No tuning outputs found",
        }
    )

if stage2 is not None and not stage2.empty:
    stage2_ok = stage2[stage2["status"] == "ok"].copy() if "status" in stage2.columns else stage2.copy()
    stage2_ok = stage2_ok.dropna(subset=["fpr_run_alert", "tpr_run_alert"], how="any").copy()
    stage2_ok["stability_margin"] = stage2_ok["tpr_run_alert"] - stage2_ok["fpr_run_alert"]

    pf = pareto_front(stage2_ok, "fpr_run_alert", "tpr_run_alert")

    fig, ax = plt.subplots(figsize=(9.2, 5.6))
    sizes = 110 + 70 * stage2_ok["persist_k"].astype(float)
    sc = ax.scatter(
        stage2_ok["fpr_run_alert"],
        stage2_ok["tpr_run_alert"],
        s=sizes,
        c=stage2_ok["alpha"],
        cmap="plasma_r",
        edgecolor="black",
        linewidth=0.8,
        alpha=0.90,
    )
    if len(pf) > 1:
        ax.plot(pf["fpr_run_alert"], pf["tpr_run_alert"], linestyle="--", color="black", linewidth=1.7, label="Pareto front")

    for _, row in pf.iterrows():
        ax.annotate(
            f"α={row['alpha']:.2f}, k={int(row['persist_k'])}",
            (row["fpr_run_alert"], row["tpr_run_alert"]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=9,
        )

    ax.set_xlabel("Benign alert rate (lower is better)")
    ax.set_ylabel("Detection rate (higher is better)")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_title("Sweep 3: Decision calibration frontier", fontweight="bold")
    ax.grid(alpha=0.20)
    if len(pf) > 1:
        ax.legend(frameon=False, loc="lower right")

    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="alpha")
    fig.tight_layout()
    sweep3_png = paper_full / "fig_sweep_3_policy_frontier.png"
    fig.savefig(sweep3_png, dpi=220, bbox_inches="tight")
    plt.close(fig)

    display(Image(filename=str(sweep3_png)))

    best_stage2 = stage2_ok.sort_values(["stability_margin", "pr_auc_wc"], ascending=False).iloc[0]
    key_rows.append(
        {
            "Sweep": "3",
            "Main result": f"Best alpha/persist: a={best_stage2['alpha']}, k={int(best_stage2['persist_k'])}",
            "Key metric": f"TPR={best_stage2['tpr_run_alert']:.3f}, FPR={best_stage2['fpr_run_alert']:.3f}",
        }
    )
else:
    display(Markdown("**Sweep 3 skipped:** tuning outputs not found."))
    key_rows.append(
        {
            "Sweep": "3",
            "Main result": "Skipped",
            "Key metric": "No tuning outputs found",
        }
    )

sweep4 = frontier[["label", "n_features", "top1_acc", "top2_acc", "joint_detection_diagnosis"]].copy().sort_values("n_features")

fig, ax = plt.subplots(figsize=(8.8, 5.0))
y = np.arange(len(sweep4))
ax.hlines(y, sweep4["top1_acc"], sweep4["top2_acc"], color="#CBD5E1", linewidth=4)
ax.scatter(sweep4["top1_acc"], y, s=130, color="#4E79A7", label="Top-1 diagnosis", zorder=3)
ax.scatter(sweep4["top2_acc"], y, s=130, color="#59A14F", label="Top-2 diagnosis", zorder=3)
for idx, row in enumerate(sweep4.itertuples()):
    ax.text(row.top2_acc + 0.02, idx, f"{row.label} ({row.n_features} fts)", va="center", fontsize=10)
ax.set_yticks(y)
ax.set_yticklabels([""] * len(y))
ax.set_xlim(0, 1.05)
ax.set_xlabel("Diagnosis accuracy")
ax.set_title("Sweep 4: Diagnosis quality vs feature budget", fontweight="bold")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="x", alpha=0.20)
plt.tight_layout()
sweep4_png = paper_full / "fig_sweep_4_diag_dumbbell.png"
fig.savefig(sweep4_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(sweep4_png)))

best_diag = frontier.sort_values("top2_acc", ascending=False).iloc[0]
key_rows.append(
    {
        "Sweep": "4",
        "Main result": f"Best diagnosis head: {best_diag['label']}",
        "Key metric": f"Top-1={best_diag['top1_acc']:.3f}, Top-2={best_diag['top2_acc']:.3f}",
    }
)

sweep5 = frontier[["label", "n_features", "portable_pr_auc", "holdout_worst_pr_auc", "reliability_margin"]].copy().sort_values("n_features")

fig, ax = plt.subplots(figsize=(8.8, 5.2))
sizes = 120 + 2200 * np.clip(sweep5["reliability_margin"].fillna(0), 0, None)
sc = ax.scatter(
    sweep5["n_features"],
    sweep5["portable_pr_auc"],
    s=sizes,
    c=sweep5["holdout_worst_pr_auc"].fillna(sweep5["portable_pr_auc"]),
    cmap="magma",
    edgecolor="black",
    linewidth=1.0,
    zorder=3,
)

for i in range(len(sweep5) - 1):
    ax.annotate(
        "",
        xy=(sweep5["n_features"].iloc[i + 1], sweep5["portable_pr_auc"].iloc[i + 1]),
        xytext=(sweep5["n_features"].iloc[i], sweep5["portable_pr_auc"].iloc[i]),
        arrowprops=dict(arrowstyle="->", color="#64748B", linewidth=1.6),
    )

for _, row in sweep5.iterrows():
    ax.text(
        row["n_features"] + 0.8,
        row["portable_pr_auc"] + 0.01,
        f"{row['label']}\nworst={row['holdout_worst_pr_auc']:.3f}",
        fontsize=9.5,
        va="center",
    )

ax.set_xlabel("Active feature budget")
ax.set_ylabel("Portable mean AUC-PR")
ax.set_ylim(0, 1.05)
ax.set_title("Sweep 5: Portability frontier", fontweight="bold")
ax.grid(alpha=0.20)

fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="Worst-workload AUC-PR")
plt.tight_layout()
sweep5_png = paper_full / "fig_sweep_5_portability_frontier.png"
fig.savefig(sweep5_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(sweep5_png)))

best_port = frontier.sort_values("portable_pr_auc", ascending=False).iloc[0]
key_rows.append(
    {
        "Sweep": "5",
        "Main result": f"Best portability head: {best_port['label']}",
        "Key metric": f"Portable AUC-PR={best_port['portable_pr_auc']:.3f}",
    }
)

# The final key-takeaway table is updated after the feature-budget sweep below.


### Sweep 6: top-feature budget ROC/PR study

This sweep ranks the final `Tier-0/1/2` features by the absolute scorer weight learned from benign training only. Each budget retrains the final head on the selected subset, then reevaluates ROC-AUC, AUC-PR, and run-level alert behavior. The goal is to show how much discrimination DICE keeps as the active feature budget shrinks.


In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

display(Markdown("### Sweep 6 results"))

paper_full = OUT_PAPER / "full"
paper_fig = PAPER_FULL / "figures"
feature_budget_summary_csv = paper_full / "dse_feature_budget_summary.csv"
feature_budget_curves_csv = paper_full / "dse_feature_budget_curve_points.csv"
feature_budget_features_csv = paper_full / "dse_feature_budget_selected_features.csv"

FEATURE_BUDGET_PCTS = [10, 20, 30, 40, 50, 70, 100]
CURVE_BUDGET_PCTS = [10, 30, 50, 100]

if feature_budget_summary_csv.exists() and feature_budget_curves_csv.exists() and feature_budget_features_csv.exists():
    feature_budget_summary = pd.read_csv(feature_budget_summary_csv)
    feature_budget_curves = pd.read_csv(feature_budget_curves_csv)
    feature_budget_features = pd.read_csv(feature_budget_features_csv)
else:
    FM = FULL_MODULE
    tiers = FM["CONFIGS"]["tier0_tier1_tier2"]
    feature_map = {tier: FM["common_features_per_tier"](DATASET_ROOT, tier) for tier in FM["TIER_FILE"].keys()}
    cases = FM["all_cases"]()

    case_X = {}
    feature_names_full = None
    for case in cases:
        X_case, feature_names_case = FM["build_case_matrix"](
            DATASET_ROOT,
            case,
            tiers=tiers,
            feature_map=feature_map,
            source_hz=5,
        )
        case_X[case.case_id] = X_case
        if feature_names_full is None:
            feature_names_full = feature_names_case

    if feature_names_full is None:
        raise RuntimeError("Could not build the final-head feature matrix for the feature-budget sweep.")

    train_benign = {case.case_id: case_X[case.case_id] for case in cases if case.label == 0}
    full_bundle = FM["train_bundle"](
        train_benign_runs=train_benign,
        feature_names=feature_names_full,
        fit_ratio=0.6,
        B=60,
        alpha=0.05,
        gain=0.35,
        ridge_lambda=1e-3,
    )

    full_weights = np.abs(np.asarray(full_bundle.weights, dtype=float))
    rank_order = np.argsort(full_weights)[::-1]

    summary_rows = []
    curve_rows = []
    feature_rows = []

    for budget_pct in FEATURE_BUDGET_PCTS:
        n_select = max(1, int(np.ceil(len(rank_order) * budget_pct / 100.0)))
        selected_idx = rank_order[:n_select]
        selected_names = [feature_names_full[i] for i in selected_idx]

        train_subset = {case_id: X[:, selected_idx] for case_id, X in train_benign.items()}
        budget_bundle = FM["train_bundle"](
            train_benign_runs=train_subset,
            feature_names=selected_names,
            fit_ratio=0.6,
            B=60,
            alpha=0.05,
            gain=0.35,
            ridge_lambda=1e-3,
        )

        pred_rows = []
        for case in cases:
            eval_out = FM["evaluate_run"](
                case_X[case.case_id][:, selected_idx],
                budget_bundle,
                B=60,
                alpha=0.05,
                persist_k=3,
                gain=0.35,
            )
            metrics = eval_out[0]
            pred_rows.append(
                {
                    "budget_pct": budget_pct,
                    "n_selected_features": n_select,
                    "case_id": case.case_id,
                    "workload": case.workload,
                    "stressor": case.stressor,
                    "label": case.label,
                    **metrics,
                }
            )

        pred_df = pd.DataFrame(pred_rows)
        base_nominal = pred_df[pred_df["stressor"] == "NOMINAL"].set_index("workload")["run_score"].to_dict()
        pred_df["nominal_template_score"] = pred_df["workload"].map(base_nominal).to_numpy(dtype=float)
        pred_df["run_score_wc"] = np.abs(
            pred_df["run_score"].to_numpy(dtype=float)
            - pred_df["nominal_template_score"].to_numpy(dtype=float)
        )

        y = pred_df["label"].to_numpy(dtype=int)
        score_base = pred_df["run_score"].to_numpy(dtype=float)
        score_wc = pred_df["run_score_wc"].to_numpy(dtype=float)
        benign = pred_df[pred_df["label"] == 0]
        anomaly = pred_df[pred_df["label"] == 1]
        detected_ttd = anomaly.loc[anomaly["run_alert"] == 1, "time_to_detect_s"].dropna()

        tier_labels = [name.split(":", 1)[0] for name in selected_names]
        tier_counts = pd.Series(tier_labels).value_counts()
        mech_labels = [FM["mechanism_group"](name) for name in selected_names]
        mech_counts = pd.Series(mech_labels).value_counts()

        summary_rows.append(
            {
                "budget_pct": budget_pct,
                "n_selected_features": n_select,
                "roc_auc_base": roc_auc_score(y, score_base),
                "pr_auc_base": average_precision_score(y, score_base),
                "roc_auc_wc": roc_auc_score(y, score_wc),
                "pr_auc_wc": average_precision_score(y, score_wc),
                "benign_alert_rate": benign["run_alert"].mean(),
                "detection_rate": anomaly["run_alert"].mean(),
                "median_ttd_s": float(detected_ttd.median()) if len(detected_ttd) else np.nan,
                "tier0_selected_share": float(tier_counts.get("tier0", 0) / n_select),
                "tier1_selected_share": float(tier_counts.get("tier1_alt", 0) / n_select),
                "tier2_selected_share": float(tier_counts.get("tier2", 0) / n_select),
                "compute_selected_share": float(mech_counts.get("compute", 0) / n_select),
                "memory_selected_share": float(mech_counts.get("memory_io", 0) / n_select),
                "thermal_selected_share": float(mech_counts.get("thermal_power", 0) / n_select),
                "scheduler_selected_share": float(mech_counts.get("scheduler_runtime", 0) / n_select),
                "pressure_selected_share": float(mech_counts.get("platform_pressure", 0) / n_select),
            }
        )

        if budget_pct in CURVE_BUDGET_PCTS:
            fpr, tpr, _ = roc_curve(y, score_wc)
            for x_val, y_val in zip(fpr, tpr):
                curve_rows.append(
                    {
                        "budget_pct": budget_pct,
                        "curve": "roc_wc",
                        "x": float(x_val),
                        "y": float(y_val),
                    }
                )
            precision, recall, _ = precision_recall_curve(y, score_wc)
            for recall_val, precision_val in zip(recall, precision):
                curve_rows.append(
                    {
                        "budget_pct": budget_pct,
                        "curve": "pr_wc",
                        "x": float(recall_val),
                        "y": float(precision_val),
                    }
                )

        for rank, idx in enumerate(selected_idx, start=1):
            feature_name = feature_names_full[int(idx)]
            feature_rows.append(
                {
                    "budget_pct": budget_pct,
                    "rank_within_budget": rank,
                    "feature_name": feature_name,
                    "tier": feature_name.split(":", 1)[0],
                    "mechanism": FM["mechanism_group"](feature_name),
                    "abs_weight_full_model": float(full_weights[int(idx)]),
                }
            )

    feature_budget_summary = pd.DataFrame(summary_rows)
    feature_budget_curves = pd.DataFrame(curve_rows)
    feature_budget_features = pd.DataFrame(feature_rows)

    feature_budget_summary.to_csv(feature_budget_summary_csv, index=False)
    feature_budget_curves.to_csv(feature_budget_curves_csv, index=False)
    feature_budget_features.to_csv(feature_budget_features_csv, index=False)

feature_budget_summary = feature_budget_summary.sort_values("budget_pct").reset_index(drop=True)

display(
    feature_budget_summary[
        [
            "budget_pct",
            "n_selected_features",
            "roc_auc_wc",
            "pr_auc_wc",
            "benign_alert_rate",
            "detection_rate",
            "median_ttd_s",
        ]
    ].round(4)
)

fig, axes = plt.subplots(1, 3, figsize=(16.8, 4.8))

axes[0].plot(feature_budget_summary["budget_pct"], feature_budget_summary["pr_auc_wc"], marker="o", linewidth=2.2, color="#4E79A7", label="AUC-PR (WC)")
axes[0].plot(feature_budget_summary["budget_pct"], feature_budget_summary["roc_auc_wc"], marker="s", linewidth=2.2, color="#59A14F", label="ROC-AUC (WC)")
axes[0].set_xlabel("Top-feature budget (%)")
axes[0].set_ylabel("Score quality")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Sweep 6A: score quality vs budget", fontweight="bold")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.20)

axes[1].plot(feature_budget_summary["budget_pct"], feature_budget_summary["detection_rate"], marker="o", linewidth=2.2, color="#E15759", label="Detection rate")
axes[1].plot(feature_budget_summary["budget_pct"], feature_budget_summary["benign_alert_rate"], marker="s", linewidth=2.2, color="#F28E2B", label="Benign alert rate")
ax2 = axes[1].twinx()
ax2.plot(feature_budget_summary["budget_pct"], feature_budget_summary["median_ttd_s"], marker="^", linewidth=2.0, color="black", label="Median TTD")
axes[1].set_xlabel("Top-feature budget (%)")
axes[1].set_ylabel("Run-level rate")
ax2.set_ylabel("Median TTD (s)")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Sweep 6B: operational tradeoff", fontweight="bold")
axes[1].grid(alpha=0.20)
lines_1, labels_1 = axes[1].get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
axes[1].legend(lines_1 + lines_2, labels_1 + labels_2, frameon=False, loc="center right")

x = np.arange(len(feature_budget_summary))
axes[2].bar(x, feature_budget_summary["tier0_selected_share"], color="#4E79A7", label="Tier-0")
axes[2].bar(x, feature_budget_summary["tier1_selected_share"], bottom=feature_budget_summary["tier0_selected_share"], color="#59A14F", label="Tier-1")
axes[2].bar(
    x,
    feature_budget_summary["tier2_selected_share"],
    bottom=feature_budget_summary["tier0_selected_share"] + feature_budget_summary["tier1_selected_share"],
    color="#F28E2B",
    label="Tier-2",
)
axes[2].set_xticks(x)
axes[2].set_xticklabels([f"{int(v)}%" for v in feature_budget_summary["budget_pct"]])
axes[2].set_ylim(0, 1.0)
axes[2].set_xlabel("Top-feature budget")
axes[2].set_ylabel("Share of selected features")
axes[2].set_title("Sweep 6C: selected-tier composition", fontweight="bold")
axes[2].legend(frameon=False, loc="upper right")
axes[2].grid(axis="y", alpha=0.20)

fig.suptitle("Sweep 6: Top-feature budget study", fontsize=15, fontweight="bold")
fig.tight_layout()
feature_budget_frontier_png = paper_full / "fig_sweep_6_feature_budget_frontier.png"
fig.savefig(feature_budget_frontier_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(feature_budget_frontier_png)))

curve_view = feature_budget_curves[feature_budget_curves["budget_pct"].isin(CURVE_BUDGET_PCTS)].copy()
curve_summary = feature_budget_summary.set_index("budget_pct")

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.0))
curve_colors = {10: "#1D4ED8", 30: "#0F766E", 50: "#B45309", 100: "#7C3AED"}
for budget_pct in CURVE_BUDGET_PCTS:
    d_roc = curve_view[(curve_view["budget_pct"] == budget_pct) & (curve_view["curve"] == "roc_wc")]
    d_pr = curve_view[(curve_view["budget_pct"] == budget_pct) & (curve_view["curve"] == "pr_wc")]
    roc_auc_val = curve_summary.loc[budget_pct, "roc_auc_wc"]
    pr_auc_val = curve_summary.loc[budget_pct, "pr_auc_wc"]
    axes[0].plot(d_roc["x"], d_roc["y"], linewidth=2.0, color=curve_colors[budget_pct], label=f"{budget_pct}% (AUC={roc_auc_val:.3f})")
    axes[1].plot(d_pr["x"], d_pr["y"], linewidth=2.0, color=curve_colors[budget_pct], label=f"{budget_pct}% (AP={pr_auc_val:.3f})")

axes[0].plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1.0)
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("Sweep 6D: ROC by feature budget", fontweight="bold")
axes[0].grid(alpha=0.20)
axes[0].legend(frameon=False, loc="lower right")

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Sweep 6E: PR by feature budget", fontweight="bold")
axes[1].grid(alpha=0.20)
axes[1].legend(frameon=False, loc="lower left")

fig.tight_layout()
feature_budget_curves_png = paper_full / "fig_sweep_6_feature_budget_curves.png"
fig.savefig(feature_budget_curves_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(feature_budget_curves_png)))

best_budget_row = feature_budget_summary.sort_values(["pr_auc_wc", "roc_auc_wc", "detection_rate"], ascending=False).iloc[0]
if "key_rows" not in globals():
    key_rows = []
key_rows.append(
    {
        "Sweep": "6",
        "Main result": f"Best top-feature budget: {int(best_budget_row['budget_pct'])}% ({int(best_budget_row['n_selected_features'])} features)",
        "Key metric": f"AUC-PR={best_budget_row['pr_auc_wc']:.3f}, ROC-AUC={best_budget_row['roc_auc_wc']:.3f}",
    }
)
key_df = pd.DataFrame(key_rows)
display(Markdown("### Key DSE takeaways"))
display(key_df)
key_df.to_csv(paper_full / "dse_key_takeaways.csv", index=False)


## 6B. Runtime and Power Context

This section separates three ideas that are easy to mix together.

- **Measured runtime context**: how long the notebook-local DICE run takes and how quickly it detects anomalous runs.
- **Current low-overhead story**: how two-stage DICE and feature-budget sweeps reduce the active feature budget while preserving discrimination.
- **Tier-1 host power context**: how much system power the traced workloads consume under the available Tier-1 power signals.

The released dataset does not contain paired DICE-off and DICE-on runs, so the Tier-1 power analysis below should be read as workload power context rather than direct detector overhead. The final code cell in this section computes true DICE power overhead once paired runs are available.


In [ ]:
display(Markdown("### Current low-overhead story"))

paper_full = OUT_PAPER / "full"
paper_fig = PAPER_FULL / "figures"
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

runtime_bundle = json.loads(NOTEBOOK_RUNTIME.read_text()) if NOTEBOOK_RUNTIME.exists() else {}
overall_full_path = OUT_FULL / "overall_metrics.csv"
sequential_path = OUT_FULL / "sequential_metrics.csv"
two_stage_path = paper_full / "two_stage_dice_summary.csv"
feature_budget_path = paper_full / "dse_feature_budget_summary.csv"

if not overall_full_path.exists() or not sequential_path.exists():
    display(Markdown("Run **Run End-to-End** first so the runtime, sequential, and design-space summaries exist."))
else:
    overall_full = pd.read_csv(overall_full_path)
    sequential = pd.read_csv(sequential_path)
    full_row = overall_full[overall_full["config"] == "tier0_tier1_tier2"].iloc[0]
    seq_row = sequential[sequential["config"] == "tier0_tier1_tier2"].iloc[0]

    if two_stage_path.exists():
        two_stage_summary = pd.read_csv(two_stage_path)
    else:
        two_stage_summary = globals().get("two_stage_summary")

    if feature_budget_path.exists():
        feature_budget_summary = pd.read_csv(feature_budget_path)
    else:
        feature_budget_summary = globals().get("feature_budget_summary")

    def choose_two_stage(df: pd.DataFrame | None):
        if df is None or len(df) == 0:
            return None
        full = df[df["policy"] == "Always Tier-0/1/2"]
        candidates = df[df["policy"].astype(str).str.startswith("Two-stage")].copy()
        if len(candidates) == 0:
            return None
        if len(full):
            full_row_local = full.iloc[0]
            keep = candidates[candidates["detection_rate"] >= float(full_row_local["detection_rate"]) - 0.02].copy()
            if len(keep) == 0:
                keep = candidates.copy()
        else:
            keep = candidates.copy()
        keep = keep.sort_values(["avg_feature_budget", "benign_alert_rate", "trigger_quantile"], ascending=[True, True, True])
        return keep.iloc[0]

    def choose_feature_budget(df: pd.DataFrame | None):
        if df is None or len(df) == 0:
            return None
        best_pr = float(df["pr_auc_wc"].max())
        keep = df[df["pr_auc_wc"] >= best_pr - 0.01].copy()
        if len(keep) == 0:
            keep = df.copy()
        keep = keep.sort_values(["n_selected_features", "budget_pct"], ascending=[True, True])
        return keep.iloc[0]

    rec_two_stage = choose_two_stage(two_stage_summary)
    rec_budget = choose_feature_budget(feature_budget_summary)

    low_overhead_row = {
        "notebook_runtime_min": float(runtime_bundle.get("runtime_minutes", np.nan)),
        "final_head_features": float(full_row.get("n_features", np.nan)),
        "final_head_median_ttd_s": float(seq_row.get("median_time_to_detect_s", np.nan)),
        "final_head_detection_rate": float(seq_row.get("anomaly_detect_rate", np.nan)),
        "final_head_benign_alert_rate": float(seq_row.get("benign_run_alert_rate", np.nan)),
        "two_stage_policy": rec_two_stage["policy"] if rec_two_stage is not None else "n/a",
        "two_stage_avg_feature_budget": float(rec_two_stage["avg_feature_budget"]) if rec_two_stage is not None else np.nan,
        "two_stage_detection_rate": float(rec_two_stage["detection_rate"]) if rec_two_stage is not None else np.nan,
        "two_stage_benign_alert_rate": float(rec_two_stage["benign_alert_rate"]) if rec_two_stage is not None else np.nan,
        "two_stage_median_ttd_s": float(rec_two_stage["median_ttd_s"]) if rec_two_stage is not None else np.nan,
        "feature_budget_pct": float(rec_budget["budget_pct"]) if rec_budget is not None else np.nan,
        "feature_budget_n_features": float(rec_budget["n_selected_features"]) if rec_budget is not None else np.nan,
        "feature_budget_pr_auc_wc": float(rec_budget["pr_auc_wc"]) if rec_budget is not None else np.nan,
        "feature_budget_roc_auc_wc": float(rec_budget["roc_auc_wc"]) if rec_budget is not None else np.nan,
        "feature_budget_detection_rate": float(rec_budget["detection_rate"]) if rec_budget is not None else np.nan,
        "feature_budget_benign_alert_rate": float(rec_budget["benign_alert_rate"]) if rec_budget is not None else np.nan,
        "feature_budget_median_ttd_s": float(rec_budget["median_ttd_s"]) if rec_budget is not None else np.nan,
    }

    low_overhead_df = pd.DataFrame([low_overhead_row])
    low_overhead_df.to_csv(paper_full / "low_overhead_summary.csv", index=False)
    display(low_overhead_df.round(4))

    story_rows = [
        {
            "policy": "Always full",
            "feature_budget": float(full_row.get("n_features", np.nan)),
            "detection_rate": float(seq_row.get("anomaly_detect_rate", np.nan)),
            "benign_alert_rate": float(seq_row.get("benign_run_alert_rate", np.nan)),
            "median_ttd_s": float(seq_row.get("median_time_to_detect_s", np.nan)),
        }
    ]
    if rec_two_stage is not None:
        story_rows.append(
            {
                "policy": str(rec_two_stage["policy"]),
                "feature_budget": float(rec_two_stage["avg_feature_budget"]),
                "detection_rate": float(rec_two_stage["detection_rate"]),
                "benign_alert_rate": float(rec_two_stage["benign_alert_rate"]),
                "median_ttd_s": float(rec_two_stage["median_ttd_s"]),
            }
        )
    if rec_budget is not None:
        story_rows.append(
            {
                "policy": f"Top {int(rec_budget['budget_pct'])}% features",
                "feature_budget": float(rec_budget["n_selected_features"]),
                "detection_rate": float(rec_budget["detection_rate"]),
                "benign_alert_rate": float(rec_budget["benign_alert_rate"]),
                "median_ttd_s": float(rec_budget["median_ttd_s"]),
            }
        )

    story_df = pd.DataFrame(story_rows)

    fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.8))

    y = np.arange(len(story_df))
    axes[0].hlines(y, xmin=0, xmax=story_df["feature_budget"], color="#D4D8DD", linewidth=3)
    sc0 = axes[0].scatter(
        story_df["feature_budget"],
        y,
        s=220 + 260 * story_df["detection_rate"].fillna(0.0),
        c=story_df["benign_alert_rate"],
        cmap="viridis_r",
        edgecolor="black",
        linewidth=0.8,
        zorder=3,
    )
    for yi, row in zip(y, story_df.itertuples(index=False)):
        axes[0].text(
            float(row.feature_budget) + 1.0,
            yi,
            f"{row.feature_budget:.0f}",
            va="center",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.82),
        )
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(story_df["policy"])
    axes[0].invert_yaxis()
    axes[0].set_xlabel("Active feature budget")
    axes[0].set_title("Cost footprint", fontweight="bold")
    axes[0].grid(axis="x", alpha=0.20)
    cbar0 = fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.04)
    cbar0.set_label("Benign alert rate")

    sc1 = axes[1].scatter(
        story_df["benign_alert_rate"],
        story_df["detection_rate"],
        s=40 + 7 * story_df["feature_budget"],
        c=story_df["feature_budget"],
        cmap="cividis",
        edgecolor="black",
        linewidth=0.8,
    )
    for row in story_df.itertuples(index=False):
        axes[1].annotate(
            f"{row.policy}\nTTD={row.median_ttd_s:.0f}s",
            xy=(float(row.benign_alert_rate), float(row.detection_rate)),
            xytext=(10, 10),
            textcoords="offset points",
            fontsize=8.7,
            bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="none", alpha=0.82),
        )
    axes[1].set_xlim(-0.02, 1.02)
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_xlabel("Benign run-alert rate")
    axes[1].set_ylabel("Detection rate")
    axes[1].set_title("Operational tradeoff", fontweight="bold")
    axes[1].grid(alpha=0.20)
    cbar1 = fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.04)
    cbar1.set_label("Feature budget")

    fig.suptitle("Current low-overhead story", fontsize=15, fontweight="bold")
    fig.tight_layout()
    low_overhead_png = paper_fig / "fig_low_overhead_story.png"
    fig.savefig(low_overhead_png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(low_overhead_png)))


### Tier-1 host power context and paired-overhead template

The table and plot below summarize host power context from the Tier-1 files. They do not measure the overhead of the \textsc{DICE} detector itself. Direct detector overhead requires paired runs of the same workload collected with \textsc{DICE} disabled and enabled under the same conditions.


In [ ]:
display(Markdown("### Tier-1 host power context"))

paper_full = OUT_PAPER / "full"
paper_fig = PAPER_FULL / "figures"
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

WORKLOADS_LOCAL = ["BROWSER", "VIDEO_SW", "PY_AI", "PY_STATS"]
STRESSORS_LOCAL = ["NOMINAL", "CACHE", "TLB", "BRANCH", "MEMBW", "ATOMIC"]
POWER_CANDIDATES = ["sys_power", "all_power", "processor_power_w", "package_power_w", "soc_power_w", "cpu_power_w"]


def load_tier1_power_trace(case_id: str):
    case_dir = DATASET_ROOT / "tier1_alt" / case_id
    for fname in ["tier1_alt_full_5hz.csv", "tier1_alt_core_5hz.csv"]:
        p = case_dir / fname
        if not p.exists():
            continue
        df = pd.read_csv(p)
        for col in POWER_CANDIDATES:
            if col in df.columns:
                return df, col, fname
    return None, None, None

power_rows = []
for workload in WORKLOADS_LOCAL:
    for stressor in STRESSORS_LOCAL:
        case_id = f"{workload}__{stressor}"
        df, power_col, source_file = load_tier1_power_trace(case_id)
        if df is None:
            continue
        power = pd.to_numeric(df[power_col], errors="coerce").dropna()
        if len(power) == 0:
            continue
        duration_s = float(len(power) / 5.0)
        power_rows.append(
            {
                "case_id": case_id,
                "workload": workload,
                "stressor": stressor,
                "source_file": source_file,
                "power_col": power_col,
                "mean_power_w": float(power.mean()),
                "median_power_w": float(power.median()),
                "p95_power_w": float(power.quantile(0.95)),
                "duration_s": duration_s,
                "energy_j": float(power.mean() * duration_s),
            }
        )

if len(power_rows) == 0:
    display(Markdown("No Tier-1 power traces were found under `tier1_alt`."))
else:
    power_df = pd.DataFrame(power_rows).sort_values(["workload", "stressor"]).reset_index(drop=True)
    nominal_map = power_df[power_df["stressor"] == "NOMINAL"].set_index("workload")["mean_power_w"].to_dict()
    power_df["delta_vs_nominal_w"] = power_df["mean_power_w"] - power_df["workload"].map(nominal_map)
    power_df.to_csv(paper_full / "tier1_power_context_cases.csv", index=False)

    workload_power = power_df[power_df["stressor"] == "NOMINAL"][
        ["workload", "source_file", "power_col", "mean_power_w", "p95_power_w", "energy_j"]
    ].sort_values("workload")
    workload_power.to_csv(paper_full / "tier1_power_context_workloads.csv", index=False)

    stressor_power = (
        power_df[power_df["stressor"] != "NOMINAL"]
        .groupby("stressor", as_index=False)
        .agg(
            n_cases=("case_id", "count"),
            mean_power_w=("mean_power_w", "mean"),
            mean_delta_vs_nominal_w=("delta_vs_nominal_w", "mean"),
            median_delta_vs_nominal_w=("delta_vs_nominal_w", "median"),
            mean_energy_j=("energy_j", "mean"),
        )
        .sort_values("stressor")
    )
    stressor_power.to_csv(paper_full / "tier1_power_context_stressors.csv", index=False)

    display(workload_power.round(4))
    display(stressor_power.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.8), gridspec_kw={"width_ratios": [1.0, 1.1]})

    nominal_view = workload_power.sort_values("mean_power_w").reset_index(drop=True)
    y0 = np.arange(len(nominal_view))
    axes[0].hlines(y0, xmin=0, xmax=nominal_view["mean_power_w"], color="#A0CBE8", linewidth=3)
    axes[0].scatter(nominal_view["mean_power_w"], y0, s=170, color="#4E79A7", edgecolor="black", zorder=3)
    axes[0].scatter(nominal_view["p95_power_w"], y0, s=90, marker="D", color="#F28E2B", edgecolor="black", zorder=3)
    for yi, row in zip(y0, nominal_view.itertuples(index=False)):
        axes[0].text(
            float(row.p95_power_w) + 1.0,
            yi,
            f"p95={row.p95_power_w:.1f}",
            va="center",
            fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.8),
        )
    axes[0].set_yticks(y0)
    axes[0].set_yticklabels(nominal_view["workload"])
    axes[0].set_xlabel("Power (W)")
    axes[0].set_title("Nominal workload power context", fontweight="bold")
    axes[0].grid(axis="x", alpha=0.20)

    stressor_view = stressor_power.sort_values("mean_delta_vs_nominal_w").reset_index(drop=True)
    y1 = np.arange(len(stressor_view))
    delta_colors = ["#E15759" if v >= 0 else "#59A14F" for v in stressor_view["mean_delta_vs_nominal_w"]]
    axes[1].hlines(y1, xmin=0, xmax=stressor_view["mean_delta_vs_nominal_w"], color=delta_colors, linewidth=3)
    axes[1].scatter(
        stressor_view["mean_delta_vs_nominal_w"],
        y1,
        s=170,
        c=delta_colors,
        edgecolor="black",
        zorder=3,
    )
    for yi, row in zip(y1, stressor_view.itertuples(index=False)):
        axes[1].text(
            float(row.mean_delta_vs_nominal_w) + (0.45 if row.mean_delta_vs_nominal_w >= 0 else -0.45),
            yi,
            f"{row.mean_delta_vs_nominal_w:+.1f}",
            va="center",
            ha="left" if row.mean_delta_vs_nominal_w >= 0 else "right",
            fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.8),
        )
    axes[1].axvline(0.0, color="black", linewidth=1.0)
    axes[1].set_yticks(y1)
    axes[1].set_yticklabels(stressor_view["stressor"])
    axes[1].set_xlabel("Mean power delta vs workload nominal (W)")
    axes[1].set_title("Stressor power shift", fontweight="bold")
    axes[1].grid(axis="x", alpha=0.20)

    fig.suptitle("Tier-1 host power context", fontsize=15, fontweight="bold")
    fig.tight_layout()
    tier1_power_png = paper_fig / "fig_tier1_power_context.png"
    fig.savefig(tier1_power_png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(tier1_power_png)))


display(Markdown("### Optional paired DICE-off/DICE-on overhead measurement"))

paired_dir = DATASET_ROOT / "power_overhead_pairs"
paired_manifest = paired_dir / "manifest.csv"

if not paired_manifest.exists():
    display(Markdown(
        "No paired overhead manifest was found. To measure true detector overhead, create "
        "`<DATASET_ROOT>/power_overhead_pairs/manifest.csv` with columns `workload`, `mode`, and `csv_path`, "
        "and optional columns `power_col` and `sample_hz`. Use `baseline` for DICE-off runs and `dice_on` "
        "for DICE-enabled runs. After that, rerun this cell to compute mean power delta (W), percent overhead, "
        "and extra energy (J)."
    ))
else:
    manifest = pd.read_csv(paired_manifest)
    pair_rows = []

    for row in manifest.itertuples(index=False):
        csv_path = Path(row.csv_path)
        if not csv_path.is_absolute():
            csv_path = paired_dir / csv_path
        df = pd.read_csv(csv_path)

        power_col = getattr(row, "power_col", None)
        if not isinstance(power_col, str) or power_col.strip() == "":
            power_col = next((col for col in POWER_CANDIDATES if col in df.columns), None)
        if power_col is None:
            continue

        sample_hz = float(getattr(row, "sample_hz", 5.0) or 5.0)
        power = pd.to_numeric(df[power_col], errors="coerce").dropna()
        if len(power) == 0:
            continue

        pair_rows.append(
            {
                "workload": row.workload,
                "mode": row.mode,
                "power_col": power_col,
                "mean_power_w": float(power.mean()),
                "median_power_w": float(power.median()),
                "duration_s": float(len(power) / sample_hz),
                "energy_j": float(power.mean() * (len(power) / sample_hz)),
            }
        )

    if len(pair_rows) == 0:
        display(Markdown("The paired manifest was found, but no usable power traces could be parsed."))
    else:
        pair_df = pd.DataFrame(pair_rows)
        pivot = pair_df.pivot_table(index="workload", columns="mode", values=["mean_power_w", "energy_j"], aggfunc="mean")
        pivot.columns = [f"{a}_{b}" for a, b in pivot.columns]
        pivot = pivot.reset_index()
        if "mean_power_w_baseline" in pivot.columns and "mean_power_w_dice_on" in pivot.columns:
            pivot["delta_power_w"] = pivot["mean_power_w_dice_on"] - pivot["mean_power_w_baseline"]
            pivot["delta_power_pct"] = 100.0 * pivot["delta_power_w"] / pivot["mean_power_w_baseline"].replace(0, np.nan)
        if "energy_j_baseline" in pivot.columns and "energy_j_dice_on" in pivot.columns:
            pivot["delta_energy_j"] = pivot["energy_j_dice_on"] - pivot["energy_j_baseline"]
        pivot.to_csv(paper_full / "dice_power_overhead_summary.csv", index=False)
        display(pivot.round(4))


## 7. Projected DICE-Score Accelerator Complexity

This section estimates only the fixed-point **DICE-score** stage, not the full host-edge software pipeline.

Included in the estimate:
- blockwise accumulation of `|r_{t,j}|`
- weighted reduction into a scalar score
- threshold and persistence logic
- top-k attribution buffer

Excluded from the estimate:
- telemetry ingestion
- normalization
- digital-twin state update / synchronization
- host software runtime

This is therefore a **projected co-design complexity estimate**, not a synthesized die-area or power result.


In [ ]:
display(Markdown("### Projected fixed-point DICE-score accelerator complexity"))

accel_metrics = pd.read_csv(OUT_FULL / "overall_metrics.csv").copy().sort_values("n_features").reset_index(drop=True)
accel_metrics["label"] = accel_metrics["config"].map(CFG_LABEL).fillna(accel_metrics["config"])

# Projected score-stage assumptions only.
BLOCK_B = 60
DECISION_HZ = 1
INPUT_WIDTH_BITS = 16
WEIGHT_WIDTH_BITS = 16
ACC_WIDTH_BITS = 32
SCORE_WIDTH_BITS = 24
TOPK = 5
PERSIST_COUNTER_BITS = 8

rows = []
for _, row in accel_metrics.iterrows():
    n = int(row["n_features"])
    feature_index_bits = max(1, int(np.ceil(np.log2(max(n, 2)))))

    abs_ops_per_sample = n
    accum_adds_per_sample = n
    sample_path_ops_per_block = (abs_ops_per_sample + accum_adds_per_sample) * BLOCK_B

    mult_ops_per_block = n
    reduce_adds_per_block = max(n - 1, 0)
    topk_compare_ops_per_block = n * TOPK
    threshold_compare_ops_per_block = 1
    persist_compare_ops_per_block = 1
    block_end_ops_per_block = (
        mult_ops_per_block
        + reduce_adds_per_block
        + topk_compare_ops_per_block
        + threshold_compare_ops_per_block
        + persist_compare_ops_per_block
    )

    accumulator_state_bits = n * ACC_WIDTH_BITS
    weight_storage_bits = n * WEIGHT_WIDTH_BITS
    topk_state_bits = TOPK * (feature_index_bits + SCORE_WIDTH_BITS)
    control_state_bits = SCORE_WIDTH_BITS + 2 * PERSIST_COUNTER_BITS + 16
    total_state_bits = (
        accumulator_state_bits
        + weight_storage_bits
        + topk_state_bits
        + control_state_bits
    )

    rows.append(
        {
            "config": row["config"],
            "label": row["label"],
            "n_features": n,
            "abs_ops_per_sample": abs_ops_per_sample,
            "accum_adds_per_sample": accum_adds_per_sample,
            "sample_path_ops_per_block": sample_path_ops_per_block,
            "mult_ops_per_block": mult_ops_per_block,
            "reduce_adds_per_block": reduce_adds_per_block,
            "topk_compare_ops_per_block": topk_compare_ops_per_block,
            "block_end_ops_per_block": block_end_ops_per_block,
            "accumulator_state_bits": accumulator_state_bits,
            "weight_storage_bits": weight_storage_bits,
            "topk_state_bits": topk_state_bits,
            "total_state_bits": total_state_bits,
            "total_state_bytes": int(np.ceil(total_state_bits / 8.0)),
        }
    )

accel_df = pd.DataFrame(rows)
base_sample = accel_df["sample_path_ops_per_block"].min()
base_block = accel_df["block_end_ops_per_block"].min()
base_state = accel_df["total_state_bytes"].min()

accel_df["sample_path_norm_vs_tier0"] = accel_df["sample_path_ops_per_block"] / base_sample
accel_df["block_end_norm_vs_tier0"] = accel_df["block_end_ops_per_block"] / base_block
accel_df["state_norm_vs_tier0"] = accel_df["total_state_bytes"] / base_state
accel_df["complexity_index"] = (
    0.40 * accel_df["sample_path_norm_vs_tier0"]
    + 0.35 * accel_df["block_end_norm_vs_tier0"]
    + 0.25 * accel_df["state_norm_vs_tier0"]
)

accel_csv = PAPER_FULL / "projected_dice_score_accelerator_complexity.csv"
accel_df.to_csv(accel_csv, index=False)

display(
    accel_df[
        [
            "label",
            "n_features",
            "abs_ops_per_sample",
            "accum_adds_per_sample",
            "mult_ops_per_block",
            "topk_compare_ops_per_block",
            "block_end_ops_per_block",
            "total_state_bytes",
            "complexity_index",
        ]
    ].round(3)
)


fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.9), gridspec_kw={"width_ratios": [1.1, 1.0, 0.95]})

sc0 = axes[0].scatter(
    accel_df["n_features"],
    accel_df["sample_path_ops_per_block"],
    s=accel_df["total_state_bytes"] * 1.5,
    c=accel_df["complexity_index"],
    cmap="viridis",
    edgecolor="black",
    linewidth=0.8,
)
for row in accel_df.itertuples(index=False):
    axes[0].text(row.n_features + 0.8, row.sample_path_ops_per_block + 35, row.label, fontsize=9)
axes[0].set_xlabel("Feature count")
axes[0].set_ylabel("Streaming ops per block")
axes[0].set_title("Streaming footprint", fontweight="bold")
axes[0].grid(alpha=0.20)
cbar0 = fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label("Complexity index")

heat_cols = ["sample_path_norm_vs_tier0", "block_end_norm_vs_tier0", "state_norm_vs_tier0"]
heat_labels = ["Streaming", "Block-end", "State"]
heat = accel_df[heat_cols].to_numpy(dtype=float)
im = axes[1].imshow(heat, cmap="YlGnBu", aspect="auto")
axes[1].set_xticks(np.arange(len(heat_labels)))
axes[1].set_xticklabels(heat_labels)
axes[1].set_yticks(np.arange(len(accel_df)))
axes[1].set_yticklabels(accel_df["label"])
axes[1].set_title("Normalized cost map", fontweight="bold")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        axes[1].text(j, i, f"{heat[i, j]:.2f}", ha="center", va="center", fontsize=9, color="black")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

y = np.arange(len(accel_df))
axes[2].hlines(y, xmin=1.0, xmax=accel_df["complexity_index"], color="#C7CAD1", linewidth=3)
axes[2].scatter(
    accel_df["complexity_index"],
    y,
    s=accel_df["total_state_bytes"] * 1.4,
    color="#4E79A7",
    edgecolor="black",
    linewidth=0.8,
    zorder=3,
)
for yi, row in zip(y, accel_df.itertuples(index=False)):
    axes[2].text(float(row.complexity_index) + 0.02, yi, f"{row.total_state_bytes:.0f} B", va="center", fontsize=8.5)
axes[2].axvline(1.0, color="black", linestyle="--", linewidth=1.0)
axes[2].set_yticks(y)
axes[2].set_yticklabels(accel_df["label"])
axes[2].set_xlabel("Complexity index")
axes[2].set_title("Projected score-stage cost", fontweight="bold")
axes[2].grid(axis="x", alpha=0.20)

fig.suptitle("Projected fixed-point DICE-score accelerator complexity", fontsize=16, fontweight="bold")
fig.tight_layout()

accel_png = PAPER_FIG / "fig_projected_dice_score_accelerator_complexity.png"
fig.savefig(accel_png, dpi=220, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(accel_png)))


display(
    Markdown(
        f"""
**Assumptions**
- `B = {BLOCK_B}` second decision blocks on a `1 Hz` grid
- residual input width = `{INPUT_WIDTH_BITS}` bits
- weight width = `{WEIGHT_WIDTH_BITS}` bits
- accumulator width = `{ACC_WIDTH_BITS}` bits
- score width = `{SCORE_WIDTH_BITS}` bits
- streaming top-`{TOPK}` attribution
- estimate covers only the fixed-point score stage, not telemetry ingestion or twin synchronization
"""
    )
)


## Quick Jump: Digital-Twin Case Study

If you only want the clearest post-detection results for the paper, start here after the notebook has been run once:
- `## 8. Case Study: True Time-Series Overlay`
- `## 9. Tier and Mechanism Attribution Dashboard`
- `## 10. LLM-Assisted Grounded Triage Results`


## 8. Case Study: True Time-Series Overlay

This section shows DICE as a genuine virtual-system comparison.
The anomaly case is plotted against its workload-matched benign reference through an overlay, a conformal-surprise stripe, and a score portrait.


In [ ]:
display(Markdown("### True time-series overlay: anomaly vs benign reference"))

paper_full = OUT_PAPER / "full"
paper_full.mkdir(parents=True, exist_ok=True)

case_pred = pd.read_csv(OUT_FULL / "case_predictions.csv")
trace_path = OUT_FULL / "case_block_traces.csv"

if not trace_path.exists():
    display(Markdown("`case_block_traces.csv` not found yet. Rerun **Run End-to-End** once after the patch cell."))
else:
    trace_df = pd.read_csv(trace_path)
    final_cfg = "tier0_tier1_tier2"

    detected = (
        case_pred[
            (case_pred["config"] == final_cfg)
            & (case_pred["label"] == 1)
            & (case_pred["run_alert"] == 1)
        ]
        .sort_values("run_score_wc", ascending=False)
    )

    if len(detected) == 0:
        display(Markdown("No detected anomaly case found for the Tier-0/1/2 head."))
    else:
        chosen = detected.iloc[0]
        chosen_case = chosen["case_id"]
        chosen_workload = chosen["workload"]

        nominal_case = case_pred[
            (case_pred["config"] == final_cfg)
            & (case_pred["workload"] == chosen_workload)
            & (case_pred["stressor"] == "NOMINAL")
        ].iloc[0]["case_id"]

        anom_trace = trace_df[(trace_df["config"] == final_cfg) & (trace_df["case_id"] == chosen_case)].copy()
        nom_trace = trace_df[(trace_df["config"] == final_cfg) & (trace_df["case_id"] == nominal_case)].copy()

        merged = anom_trace.merge(
            nom_trace[["block_idx", "score"]].rename(columns={"score": "nominal_score"}),
            on="block_idx",
            how="inner",
        ).copy()

        merged["delta_score"] = merged["score"] - merged["nominal_score"]
        merged["block_time_min"] = merged["block_end_s"] / 60.0
        first_persist = merged.loc[merged["persist_alert"] == 1, "block_time_min"]
        first_persist_time = float(first_persist.iloc[0]) if len(first_persist) else np.nan
        merged["surprise_score"] = -np.log10(np.clip(merged["pvalue"].to_numpy(dtype=float), 1e-6, 1.0))

        fig = plt.figure(figsize=(13.2, 8.0))
        gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 1.0], hspace=0.30, wspace=0.24)
        ax_top = fig.add_subplot(gs[0, :])
        ax_gap = fig.add_subplot(gs[1, 0], sharex=ax_top)
        ax_phase = fig.add_subplot(gs[1, 1])

        ax_top.plot(
            merged["block_time_min"],
            merged["score"],
            color="#4E79A7",
            linewidth=2.3,
            label=f"Anomaly case: {chosen_case}",
        )
        ax_top.plot(
            merged["block_time_min"],
            merged["nominal_score"],
            color="#59A14F",
            linewidth=2.0,
            linestyle="--",
            label=f"Benign reference: {nominal_case}",
        )
        ax_top.axhline(float(chosen["tau"]), color="#E15759", linestyle=":", linewidth=2, label="Conformal threshold")

        if np.isfinite(first_persist_time):
            ax_top.axvline(first_persist_time, color="black", linestyle="--", linewidth=1.5, label="First persistent alert")

        ax_top.set_ylabel("Block score")
        ax_top.set_title("Digital-twin score overlay against the workload-matched benign reference", fontweight="bold")
        ax_top.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.02))
        ax_top.grid(alpha=0.20)

        surprise_band = merged["surprise_score"].to_numpy(dtype=float)[None, :]
        extent = [merged["block_time_min"].min(), merged["block_time_min"].max(), 0, 1]
        im = ax_gap.imshow(
            surprise_band,
            aspect="auto",
            cmap="magma",
            extent=extent,
            vmin=0.0,
            vmax=max(2.0, float(np.nanmax(merged["surprise_score"]))),
        )
        alert_times = merged.loc[merged["persist_alert"] == 1, "block_time_min"]
        if len(alert_times):
            ax_gap.scatter(alert_times, np.full(len(alert_times), 0.5), color="#7FDBFF", marker="|", s=900, linewidth=2.4, zorder=3)
        ax_gap.set_yticks([])
        ax_gap.set_xlabel("Time (minutes)")
        ax_gap.set_title("Conformal surprise stripe", fontweight="bold")
        cbar_gap = fig.colorbar(im, ax=ax_gap, fraction=0.046, pad=0.04)
        cbar_gap.set_label(r"$-\log_{10}(p)$")

        phase_colors = merged["block_time_min"].to_numpy(dtype=float)
        sc = ax_phase.scatter(
            merged["nominal_score"],
            merged["score"],
            c=phase_colors,
            cmap="viridis",
            s=55 + 35 * merged["persist_alert"].to_numpy(dtype=float),
            edgecolor="black",
            linewidth=0.35,
            alpha=0.9,
        )
        mn = float(min(merged["nominal_score"].min(), merged["score"].min()))
        mx = float(max(merged["nominal_score"].max(), merged["score"].max()))
        ax_phase.plot([mn, mx], [mn, mx], linestyle="--", color="#64748B", linewidth=1.4)
        ax_phase.set_xlabel("Benign reference score")
        ax_phase.set_ylabel("Anomaly-case score")
        ax_phase.set_title("Score portrait against the benign reference", fontweight="bold")
        ax_phase.grid(alpha=0.20)
        cbar_phase = fig.colorbar(sc, ax=ax_phase, fraction=0.046, pad=0.04)
        cbar_phase.set_label("Time (minutes)")

        fig.suptitle(
            f"True time-series overlay for {chosen_workload} / {chosen['stressor']}",
            fontsize=16,
            fontweight="bold",
            y=0.98,
        )

        out_png = paper_full / "fig_true_timeseries_overlay.png"
        fig.tight_layout(rect=[0, 0, 0.92, 0.96])
        fig.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.close(fig)

        display(Image(filename=str(out_png)))


## 9. Tier and Mechanism Attribution Dashboard

These plots explain where the digital twin's evidence comes from. The tier panel shows which observability level dominates. The mechanism panel now uses a bubble map so stressor-to-stressor differences are easier to compare than in a stacked bar chart. The confusion matrix shows how often the diagnosis head confuses one stressor with another.


In [ ]:
tier_corr, stressor_tier, tier_corr_png = render_tier_correlation_dashboard(OUT_FULL, PAPER_FULL, PAPER_FIG)
tier_summary, mechanism_summary, cm_summary, explainability_png = render_explainability_dashboard(
    OUT_FULL,
    PAPER_FULL,
    PAPER_FIG,
)

print('Tier attribution summary')
display(tier_summary.round(4))
print('Mechanism attribution summary')
display(mechanism_summary.round(4))
print('Diagnosis confusion summary')
display(cm_summary.round(4))
print('Tier-share correlation matrix')
display(tier_corr.round(4))
print('Mean tier evidence by stressor')
display(stressor_tier.round(4))

display(Image(filename=str(explainability_png)))
display(Image(filename=str(tier_corr_png)))


## 10. LLM-Assisted Grounded Triage Results

This section treats the LLM layer as a secondary results subsection built on top of structured DICE evidence. The detector itself is unchanged. DICE still produces the anomaly score, threshold, alert decision, dominant tier, dominant mechanism, and ranked residual cues. The LLM layer only converts those structured outputs into reviewer-facing summaries, triage notes, and follow-up recommendations.

The notebook therefore reports a grounded diagnostics pack rather than an LLM-based detector. It exports structured case cards, a local-model catalog, and prompt bundles for three downstream tasks: reviewer summary, triage note, and follow-up recommendation.

The recommended evaluation order is:
- `Qwen/Qwen2.5-7B-Instruct` as the primary DICE baseline,
- `microsoft/Phi-4-mini-instruct` as the lightweight comparison,
- `meta-llama/Meta-Llama-3.1-8B-Instruct` as an ecosystem baseline,
- `Qwen/Qwen2.5-14B-Instruct` for stronger offline review.

If no local inference runtime is installed, the notebook still exports the grounded artifacts needed for later model execution. That keeps the LLM layer aligned with the paper narrative while preserving the detector boundaries.


In [ ]:
llm_cards = export_llm_case_cards(OUT_FULL, APPENDIX_FULL)
llm_models = export_llm_diagnostic_model_catalog(APPENDIX_FULL)
llm_models = llm_models.sort_values('priority_rank').reset_index(drop=True)
llm_prompt_bundle = export_llm_diagnostic_prompt_bundle(llm_cards, llm_models, APPENDIX_FULL)

llm_summary = pd.DataFrame([
    {
        'n_case_cards': len(llm_cards),
        'n_model_profiles': len(llm_models),
        'n_prompt_rows': len(llm_prompt_bundle),
        'prompt_types': ', '.join(sorted(llm_prompt_bundle['prompt_type'].unique())),
    }
])
print('LLM-assisted grounded triage summary')
display(llm_summary)
print('Case-card preview')
display(llm_cards[['case_id', 'workload', 'stressor', 'dominant_tier', 'dominant_mechanism']].head(10))
print('Suggested local/offline model catalog (primary baseline first)')
display(llm_models)
print('Prompt bundle preview')
display(llm_prompt_bundle[['model_id', 'case_id', 'prompt_type']].head(12))

runtime_modules = {}
for mod in ['transformers', 'torch', 'vllm', 'llama_cpp']:
    runtime_modules[mod] = importlib.util.find_spec(mod) is not None
print('Local inference runtime available:', runtime_modules)
if not any(runtime_modules.values()):
    print('No LLM runtime is installed in this environment. The notebook exported grounded prompts and model metadata only; actual model outputs still require a local inference backend.')


### Run a Local Grounded LLM

This cell runs one local model over the exported DICE case cards. It is disabled by default because the released environment does not currently include an inference runtime. When you install a runtime, start with `Qwen/Qwen2.5-7B-Instruct` as the main baseline and `microsoft/Phi-4-mini-instruct` as the lightweight comparison.


In [ ]:
RUN_LOCAL_LLM = False
LOCAL_LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LOCAL_LLM_PROMPT_TYPE = "triage"
MAX_LLM_CASES = 8
MAX_NEW_TOKENS = 320
LLM_TEMPERATURE = 0.0

llm_outputs = None
if RUN_LOCAL_LLM:
    try:
        llm_outputs = run_transformers_llm_batch(
            llm_prompt_bundle,
            APPENDIX_FULL,
            model_id=LOCAL_LLM_MODEL_ID,
            prompt_type=LOCAL_LLM_PROMPT_TYPE,
            max_cases=MAX_LLM_CASES,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=LLM_TEMPERATURE,
        )
        print("Generated local LLM outputs")
        display(llm_outputs.head(5))
    except Exception as e:
        print(f"Local LLM run failed: {e}")
else:
    print("Set RUN_LOCAL_LLM = True after installing a local inference runtime.")


### Score Grounding, Cue Coverage, and Hallucination Rate

This cell scores the generated LLM outputs against structured DICE evidence. It reports three simple diagnostics: grounding of the dominant tier/mechanism, cue coverage of the top-ranked evidence, and a conservative hallucination heuristic based on unsupported tier or mechanism mentions.


In [ ]:
if llm_outputs is None:
    candidate_slug = _slugify_model_id(LOCAL_LLM_MODEL_ID)
    candidate_csv = APPENDIX_FULL / f"llm_outputs_{candidate_slug}_{LOCAL_LLM_PROMPT_TYPE}.csv"
    if candidate_csv.exists():
        llm_outputs = pd.read_csv(candidate_csv)
    else:
        llm_outputs = None

if llm_outputs is None:
    print("No local LLM outputs found yet. Run the previous cell first or place a saved output CSV in the appendix folder.")
else:
    eval_stem = _slugify_model_id(str(llm_outputs["model_id"].iloc[0])) + "_" + str(llm_outputs["prompt_type"].iloc[0])
    llm_grounding_summary, llm_grounding_detail = evaluate_llm_grounding_outputs(
        llm_outputs,
        llm_cards,
        APPENDIX_FULL,
        stem=eval_stem,
    )
    print("Grounded LLM evaluation summary")
    display(llm_grounding_summary.round(4))
    print("Grounded LLM evaluation detail")
    display(llm_grounding_detail.head(10))


## 11. Residual Evidence Concentration

This section shows how much anomaly evidence is captured by the top residual contributors.
That is a more DICE-specific interpretability view than simple feature-frequency counts.


In [ ]:
case_diag = pd.read_csv(OUT_FULL / 'case_diagnosis_summary.csv')
anom_diag = case_diag[case_diag['label'] == 1].copy()
mech_cols = [
    'compute_contrib',
    'memory_io_contrib',
    'thermal_power_contrib',
    'scheduler_runtime_contrib',
    'platform_pressure_contrib',
]
anom_diag['total_evidence'] = anom_diag[mech_cols].sum(axis=1).replace(0.0, np.nan)

for k in range(1, 6):
    cols = [f'top_feature_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'feature_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

for k in range(1, 4):
    cols = [f'top_mechanism_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'mechanism_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

coverage_cols = [
    'feature_top1_coverage',
    'feature_top2_coverage',
    'feature_top3_coverage',
    'feature_top4_coverage',
    'feature_top5_coverage',
    'mechanism_top1_coverage',
    'mechanism_top2_coverage',
    'mechanism_top3_coverage',
]
evidence_frontier = anom_diag.groupby('config', sort=False)[coverage_cols].mean().reset_index()
evidence_frontier['label'] = evidence_frontier['config'].map(CFG_LABEL).fillna(evidence_frontier['config'])
evidence_frontier.to_csv(PAPER_FULL / 'residual_evidence_concentration.csv', index=False)

stressor_evidence = (
    anom_diag[anom_diag['config'] == 'tier0_tier1_tier2']
    .groupby('stressor', sort=False)[
        [
            'feature_top1_coverage',
            'feature_top3_coverage',
            'feature_top5_coverage',
            'mechanism_top1_coverage',
            'mechanism_top2_coverage',
            'mechanism_top3_coverage',
        ]
    ]
    .mean()
    .reset_index()
)
stressor_evidence.to_csv(PAPER_FULL / 'stressor_residual_evidence_concentration.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
feature_k = [1, 2, 3, 4, 5]
mechanism_k = [1, 2, 3]
for _, row in evidence_frontier.iterrows():
    axes[0].plot(feature_k, [row[f'feature_top{k}_coverage'] for k in feature_k], marker='o', linewidth=2, label=row['label'])
    axes[1].plot(mechanism_k, [row[f'mechanism_top{k}_coverage'] for k in mechanism_k], marker='o', linewidth=2, label=row['label'])
axes[0].set_xlabel('Top-k residual features')
axes[0].set_ylabel('Mean anomaly evidence coverage')
axes[0].set_xticks(feature_k)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_title('Residual evidence concentration by feature rank')
axes[1].set_xlabel('Top-k mechanism groups')
axes[1].set_ylabel('Mean anomaly evidence coverage')
axes[1].set_xticks(mechanism_k)
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title('Residual evidence concentration by mechanism rank')
axes[1].legend(loc='lower right')
fig.tight_layout()
coverage_png = PAPER_FIG / 'fig_residual_evidence_concentration.png'
fig.savefig(coverage_png, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Residual evidence concentration across digital-twin heads')
display(evidence_frontier)

print('Final-head stressor evidence concentration')
display(stressor_evidence)

display(Image(filename=str(coverage_png)))


## 12. Uncertainty and Confidence Intervals

The deployed DICE head remains lightweight.
Extra offline runtime is spent here on bootstrap confidence intervals so the reported paper metrics are better defended.


In [ ]:
BOOTSTRAP_SAMPLES = 1000
bootstrap_ci, bootstrap_png = render_bootstrap_confidence(case_pred, PAPER_FULL, PAPER_FIG, samples=BOOTSTRAP_SAMPLES, seed=0)
print('Bootstrap confidence intervals')
display(bootstrap_ci)
display(Image(filename=str(bootstrap_png)))


## Quick Jump: Paper Figures

If you only want the reviewer-facing paper artifacts, start here once the result CSVs already exist:
- `## 13. Paper-Ready Figure Bundle`
- then review the claim-boundary and reproducibility sections below.


## 13. Paper-Ready Figure Bundle

This section assembles the reviewer-facing composite figures used in the main paper and appendix. It is the fastest place to start if the result CSVs already exist and you only need paper-ready visual artifacts.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')

if (PAPER_FULL / 'conformal_reliability_summary.csv').exists():
    reliability_bundle = pd.read_csv(PAPER_FULL / 'conformal_reliability_summary.csv')
else:
    reliability_bundle = reliability.copy() if 'reliability' in globals() else pd.DataFrame()

performance_stack, performance_stack_png = render_paper_performance_stack(
    case_pred,
    overall_full,
    sequential,
    reliability_bundle,
    PAPER_FULL,
    PAPER_FIG,
)

tier_summary, mechanism_summary, cm_summary, explainability_png = render_explainability_dashboard(
    OUT_FULL,
    PAPER_FULL,
    PAPER_FIG,
)

if 'frontier' not in globals():
    diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
    frontier = (
        overall_full[['config', 'roc_auc_wc', 'pr_auc_wc']]
        .merge(
            sequential[['config', 'benign_run_alert_rate', 'anomaly_detect_rate', 'median_time_to_detect_s']],
            on='config',
            how='left',
        )
        .merge(
            diagnosis[['config', 'top1_acc', 'top2_acc']],
            on='config',
            how='left',
        )
    )
    if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
        holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
        frontier = frontier.merge(
            holdout[['config', 'mean_pr_auc', 'worst_pr_auc']],
            on='config',
            how='left',
        )
        frontier['portable_pr_auc'] = frontier['mean_pr_auc'].fillna(frontier['pr_auc_wc'])
    else:
        holdout = pd.DataFrame()
        frontier['portable_pr_auc'] = frontier['pr_auc_wc']
else:
    if 'holdout' not in globals() or holdout is None:
        holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv') if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists() else pd.DataFrame()

if 'bootstrap_ci' not in globals():
    bootstrap_ci, bootstrap_png = render_bootstrap_confidence(case_pred, PAPER_FULL, PAPER_FIG, samples=1000, seed=0)

portability_summary, portability_png = render_portability_dashboard(
    frontier,
    holdout,
    bootstrap_ci,
    PAPER_FULL,
    PAPER_FIG,
)

print('Performance stack summary')
display(performance_stack)
print('Portability summary')
display(portability_summary)
display(Image(filename=str(performance_stack_png)))
display(Image(filename=str(explainability_png)))
display(Image(filename=str(portability_png)))


## 13B. Abstract-Ready Metric Summary

This section collects the smallest set of paper-safe headline numbers for the abstract.
It prioritizes anomaly detection and workload-holdout robustness.
It reports detector power overhead only if paired DICE-off and DICE-on traces are available.


In [ ]:
display(Markdown("### Abstract-ready metrics"))

ABSTRACT_FINAL_CONFIG = 'tier0_tier1_tier2'
ABSTRACT_DIR = PAPER_FULL
ABSTRACT_DIR.mkdir(parents=True, exist_ok=True)


def _profile_dirs(profile: str) -> tuple[Path, Path]:
    return (
        default_full_out_dir(DATASET_ROOT, 'global', profile),
        default_full_out_dir(DATASET_ROOT, 'workload_holdout', profile),
    )


def _load_abstract_row(profile: str) -> dict[str, object] | None:
    global_dir, holdout_dir = _profile_dirs(profile)
    needed = [
        global_dir / 'overall_metrics.csv',
        holdout_dir / 'holdout_robustness_summary.csv',
    ]
    if any(not p.exists() for p in needed):
        return None

    overall = pd.read_csv(global_dir / 'overall_metrics.csv')
    holdout = pd.read_csv(holdout_dir / 'holdout_robustness_summary.csv')

    overall_row = overall.loc[overall['config'] == ABSTRACT_FINAL_CONFIG].iloc[0]
    holdout_row = holdout.loc[holdout['config'] == ABSTRACT_FINAL_CONFIG].iloc[0]

    runtime_s = float('nan')
    invocation_path = global_dir / 'notebook_invocation.json'
    runtime_path = global_dir / 'config_runtime_summary.csv'
    if invocation_path.exists():
        runtime_s = float(json.loads(invocation_path.read_text()).get('elapsed_s', float('nan')))
    elif runtime_path.exists():
        runtime_df = pd.read_csv(runtime_path)
        runtime_row = runtime_df.loc[runtime_df['config'] == ABSTRACT_FINAL_CONFIG]
        if not runtime_row.empty:
            runtime_s = float(runtime_row['fit_eval_seconds'].iloc[0])

    return {
        'feature_profile': profile,
        'global_dir': str(global_dir),
        'holdout_dir': str(holdout_dir),
        'base_roc_auc': float(overall_row['roc_auc']),
        'base_pr_auc': float(overall_row['pr_auc']),
        'wc_roc_auc': float(overall_row['roc_auc_wc']),
        'wc_pr_auc': float(overall_row['pr_auc_wc']),
        'holdout_mean_pr_auc': float(holdout_row['mean_pr_auc']),
        'holdout_worst_pr_auc': float(holdout_row['worst_pr_auc']),
        'holdout_mean_roc_auc': float(holdout_row['mean_roc_auc']),
        'runtime_s': runtime_s,
    }


rows = []
for profile_name in ['mixed', 'full']:
    row = _load_abstract_row(profile_name)
    if row is not None:
        rows.append(row)

if not rows:
    display(Markdown(
        'No abstract-ready result bundles were found yet. Run the mixed global and mixed holdout evaluations first. '
        'Add the full profile later if you want the upper-bound comparison.'
    ))
else:
    abstract_df = pd.DataFrame(rows)
    abstract_csv = ABSTRACT_DIR / 'abstract_metrics_summary.csv'
    abstract_df.to_csv(abstract_csv, index=False)
    display(abstract_df[[
        'feature_profile',
        'base_roc_auc',
        'base_pr_auc',
        'holdout_mean_pr_auc',
        'holdout_worst_pr_auc',
        'runtime_s',
    ]].round(4))

    mixed_row = abstract_df.loc[abstract_df['feature_profile'] == 'mixed'].iloc[0]
    snippet_lines = []
    snippet_lines.append('Detection headline (recommended):')
    snippet_lines.append(
        f"On the released trace set, the mixed-profile final head achieves ROC-AUC/AUC-PR of "
        f"{mixed_row['base_roc_auc']:.4f}/{mixed_row['base_pr_auc']:.4f} on the base run score."
    )
    snippet_lines.append('')
    snippet_lines.append('Holdout robustness (recommended):')
    snippet_lines.append(
        f"Under workload holdout, the same head reaches mean AUC-PR {mixed_row['holdout_mean_pr_auc']:.4f}, "
        f"worst-case AUC-PR {mixed_row['holdout_worst_pr_auc']:.4f}, and mean ROC-AUC {mixed_row['holdout_mean_roc_auc']:.4f}."
    )

    full_row = abstract_df.loc[abstract_df['feature_profile'] == 'full']
    if not full_row.empty:
        full_row = full_row.iloc[0]
        snippet_lines.append('')
        snippet_lines.append('Full-profile upper-bound comparison (use only if it helps):')
        snippet_lines.append(
            f"Switching Tier-1 and Tier-2 from core to full changes base ROC-AUC/AUC-PR to "
            f"{full_row['base_roc_auc']:.4f}/{full_row['base_pr_auc']:.4f} and holdout mean/worst AUC-PR to "
            f"{full_row['holdout_mean_pr_auc']:.4f}/{full_row['holdout_worst_pr_auc']:.4f}."
        )

    paired_power_path = ABSTRACT_DIR / 'dice_power_overhead_summary.csv'
    if paired_power_path.exists():
        power_df = pd.read_csv(paired_power_path)
        if {'delta_power_w', 'delta_power_pct'}.issubset(power_df.columns):
            snippet_lines.append('')
            snippet_lines.append('Paired detector power overhead:')
            snippet_lines.append(
                f"Across paired runs, the median detector power delta is {power_df['delta_power_w'].median():.4f} W "
                f"({power_df['delta_power_pct'].median():.2f}\\%)."
            )

    snippet_text = '\n'.join(snippet_lines) + '\n'
    snippet_md = ABSTRACT_DIR / 'ABSTRACT_SNIPPETS.md'
    snippet_md.write_text(snippet_text)
    display(Markdown('```text\n' + snippet_text + '```'))
    print('Wrote:', abstract_csv)
    print('Wrote:', snippet_md)


## 14. Research Directions: Digital Twins + LLMs

The papers below are good next-step inspirations for DICE. They are not claims of what DICE already does; they point to concrete directions that would make the next version of the work stronger.

1. **Sequential conformal for time series**: Xu and Xie, *Sequential Predictive Conformal Inference for Time Series* (ICML 2023).
   Use this to move DICE from split-conformal under exchangeability toward adaptive sequential calibration under temporal dependence.
   Link: https://proceedings.mlr.press/v202/xu23r.html

2. **Fleet-scale / federated twins**: San, Pawar, and Rasheed, *Decentralized digital twins of complex dynamical systems* (Scientific Reports 2023).
   Use this to turn the current optional-fleet story into a real decentralized or federated DICE update path.
   Link: https://www.nature.com/articles/s41598-023-47078-9

3. **World-model direction for twins**: Zhou et al., *Digital Twin AI: Opportunities and Challenges from Large Language Models to World Models* (arXiv 2026).
   Use this as the framing for evolving DICE from a behavioral micro-twin into a richer world-model twin while staying honest about what the current paper implements.
   Link: https://arxiv.org/abs/2601.01321

4. **Generative trajectory twins**: Makarov et al., *Large language models forecast patient health trajectories enabling digital twins* (npj Digital Medicine 2025).
   The important idea is not the medical domain; it is the combination of trajectory forecasting, preserved cross-correlations, and explainability. That is relevant if DICE ever moves from anomaly scoring toward generative telemetry forecasting.
   Link: https://www.nature.com/articles/s41746-025-02004-3

5. **Local LLM-enabled twin reasoning**: Lammert et al., *Large language models-enabled digital twins for precision medicine in rare gynecological tumors* (npj Digital Medicine 2025).
   This is a good analogue for a local DICE copilot that reasons over structured case cards, recommends follow-up diagnostics, and keeps sensitive data local.
   Link: https://www.nature.com/articles/s41746-025-01810-z

6. **Edge-aware LLM plus twin systems**: Hong, Wu, and Morello, *LLM-Twin* (Scientific Reports 2024).
   The useful idea here is the mini-giant deployment split. For DICE, that suggests a tiny local model for case summarization plus an optional larger offline model for deeper review.
   Link: https://www.nature.com/articles/s41598-024-69474-5

7. **Self-checking diagnostic summaries**: Liu et al., *Large Language Models have Intrinsic Self-Correction Ability* (arXiv 2024).
   If you build an LLM-based DICE copilot, this supports adding a second-pass self-check to keep the report conservative and grounded.
   Link: https://arxiv.org/abs/2406.15673

The strongest next-step research directions for DICE are therefore:
- sequential conformal recalibration under temporal drift,
- federated or decentralized edge-package refresh,
- richer trajectory-level twins that preserve cross-feature structure,
- a local LLM copilot that only reads structured DICE evidence rather than raw telemetry.


## 15. Paper Claim Boundaries

This notebook is designed to support a clear and defensible paper narrative. The main claim boundaries are as follows.

- **Not a full virtual replica.** DICE is a behavioral micro-twin that maintains a virtual benign reference in telemetry space.
- **Not an LLM-based detector.** The detector is the digital twin plus residual scoring, conformal calibration, and persistence.
- **Not a synthesized hardware implementation.** The accelerator section reports a projected score-stage complexity estimate only.
- **Yes to calibrated detection and diagnosis.** The main supported claims are tier-aware behavioral digital twinning, false-alarm-controlled block decisions, diagnosis cues, observability-budget analysis, and a plausible co-design path.


## 16. Reproducibility Manifest

This final cell confirms the dataset hash, environment hash, and notebook runtime summary so the released analysis can be checked across machines.


In [ ]:
manifest = json.loads(MANIFEST.read_text())
print('Manifest path:', MANIFEST)
print('Dataset SHA256:', manifest['dataset_digest']['sha256'])
print('Environment file SHA256:', manifest['environment_files']['environment_yml']['sha256'])
print('Requirements SHA256:', manifest['environment_files']['requirements_txt']['sha256'])
if NOTEBOOK_RUNTIME.exists():
    runtime_summary = json.loads(NOTEBOOK_RUNTIME.read_text())
    print('Notebook runtime summary:', runtime_summary)
print('Main paper artifacts:', PAPER_FULL)
print('Appendix artifacts :', APPENDIX_FULL)
